# Preparação e análise inicial dos dados – TCC

Este notebook documenta o processo de carregamento, compreensão, organização e preparação inicial dos dados utilizados no Trabalho de Conclusão de Curso.

O objetivo do trabalho é utilizar dados reais de propriedades leiteiras para posteriormente treinar e comparar modelos de Aprendizado de Máquina voltados à predição da produção leiteira.

Antes da modelagem, é necessário compreender a estrutura dos arquivos recebidos, identificar problemas de qualidade, padronizar os registros e integrar as diferentes fontes de dados. Essas atividades fazem parte do processo de descoberta de conhecimento adotado na pesquisa.

Neste notebook são realizadas inicialmente as seguintes atividades:

1. identificação dos arquivos recebidos;
2. identificação das abas existentes em cada arquivo;
3. análise da estrutura das bases;
4. separação das informações por tipo;
5. consolidação dos controles leiteiros;
6. identificação de registros duplicados;
7. investigação de informações conflitantes;
8. preparação para a construção da base analítica final.

Os arquivos originais não são modificados durante esse processo. Todas as transformações são realizadas em memória por meio da biblioteca Pandas.

## 1. Configuração inicial e definição dos caminhos

Nesta etapa são importadas as bibliotecas utilizadas para manipulação dos dados e definidos os caminhos das bases originais.

Os arquivos foram organizados de acordo com sua propriedade de origem, mantendo os dados recebidos sem alterações:

- `cerqueira_cesar`: contém o cadastro dos animais e os controles leiteiros mensais da propriedade de Cerqueira César;
- `uenp`: contém a base de dados da bovinocultura da UENP.

A utilização de caminhos relativos permite que o projeto seja executado em diferentes computadores sem a necessidade de alterar manualmente os endereços dos arquivos.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# O notebook está dentro da pasta "notebooks"
PASTA_PROJETO = Path.cwd().parent

# Pastas principais
PASTA_DADOS = PASTA_PROJETO / "dados"
PASTA_ORIGINAIS = PASTA_DADOS / "originais"

# Bases separadas por propriedade
PASTA_CC = PASTA_ORIGINAIS / "cerqueira_cesar"
PASTA_UENP = PASTA_ORIGINAIS / "uenp"

print("Pasta do projeto:", PASTA_PROJETO)
print("Cerqueira César:", PASTA_CC)
print("UENP:", PASTA_UENP)

### 1.1 Identificação dos arquivos disponíveis

Após a definição dos diretórios do projeto, são identificados automaticamente os arquivos Excel disponíveis em cada propriedade.

Os arquivos são mantidos separados de acordo com sua origem para preservar a rastreabilidade dos dados durante as etapas de preparação e integração.

Nesta etapa não é realizada nenhuma modificação nos arquivos. O objetivo é apenas verificar se todas as bases necessárias estão disponíveis e acessíveis pelo notebook.

In [ ]:
# Identificação dos arquivos Excel disponíveis nas duas propriedades

arquivos_cc = sorted(PASTA_CC.glob("*.xlsx"))
arquivos_uenp = sorted(PASTA_UENP.glob("*.xlsx"))

# Lista geral para manter compatibilidade com as etapas seguintes do notebook
arquivos_excel = arquivos_cc + arquivos_uenp

print(f"Arquivos de Cerqueira César: {len(arquivos_cc)}")
for arquivo in arquivos_cc:
    print(" -", arquivo.name)

print(f"\nArquivos da UENP: {len(arquivos_uenp)}")
for arquivo in arquivos_uenp:
    print(" -", arquivo.name)

print(f"\nTotal de arquivos Excel: {len(arquivos_excel)}")

## 2. Levantamento da estrutura dos arquivos Excel

Os dados foram recebidos em diferentes arquivos Excel e nem todos apresentam a mesma organização.

Por esse motivo, antes de realizar qualquer tratamento foi feito um levantamento das abas existentes em cada arquivo.

Essa etapa permite compreender quais tipos de informação estão disponíveis e evita selecionar ou excluir dados antes de conhecer completamente a estrutura original das bases.

Foram identificados arquivos de controle leiteiro mensal e arquivos contendo informações cadastrais, produtivas, reprodutivas, sanitárias e de pesagem.

In [ ]:
for arquivo in arquivos_excel:
    print("\n" + "=" * 70)
    print("ARQUIVO:", arquivo)

    excel = pd.ExcelFile(arquivo)

    print("ABAS:")
    for aba in excel.sheet_names:
        print("  -", aba)

### Principais fontes identificadas

**Propriedade de Cerqueira César**

- arquivo cadastral dos animais;
- arquivos mensais de controle leiteiro.

**Fazenda Escola da UENP**

O arquivo apresenta diferentes abas, incluindo:

- Visão Geral;
- Produção;
- Pesagem;
- Doenças;
- Reprodução;
- Manejo sanitário;
- Alimentação;
- entre outros registros.

## 3. Análise estrutural inicial dos dados

Após identificar os arquivos e suas respectivas abas, foi realizada uma inspeção inicial das bases.

Nesta etapa foram avaliados:

- número de linhas e colunas;
- nomes das variáveis;
- tipos de dados identificados pelo Pandas;
- primeiras observações;
- presença de valores ausentes.

O objetivo desta análise não é realizar a limpeza imediatamente, mas compreender como os dados foram originalmente registrados.

Esse procedimento revelou que as duas propriedades possuem estruturas de armazenamento diferentes, tornando necessária uma etapa posterior de padronização e integração.

In [ ]:
# Arquivos e abas selecionados para inspeção detalhada

arquivos_abas = [
    (PASTA_CC / "_Dados animais - TCC Ananda.xlsx", "Sheet1"),
    (PASTA_UENP / "Bovinocultura UENP.xlsx", "Visão Geral"),
    (PASTA_UENP / "Bovinocultura UENP.xlsx", "Produção"),
    (PASTA_UENP / "Bovinocultura UENP.xlsx", "Pesagem 2026"),
    (PASTA_UENP / "Bovinocultura UENP.xlsx", "Doenças"),
    (PASTA_UENP / "Bovinocultura UENP.xlsx", "Reprodução"),
    (PASTA_CC / "2026_07_28 Controle Leiteiro v1 - Lotes.xlsx", "Lotes")
]

for arquivo, aba in arquivos_abas:
    print("\n" + "=" * 80)
    print("ARQUIVO:", arquivo.name)
    print("ABA:", aba)

    df = pd.read_excel(arquivo, sheet_name=aba)

    print("Formato:", df.shape)
    print("\nColunas:")
    print(df.columns.tolist())

    print("\nPrimeiras 5 linhas:")
    display(df.head())

In [ ]:
for arquivo, aba in arquivos_abas:
    print("\n" + "="*80)
    print("ARQUIVO:", arquivo)
    print("ABA:", aba)

    df = pd.read_excel(arquivo, sheet_name=aba)

    print("\nTipos de dados:")
    print(df.dtypes)

    print("\nValores ausentes:")
    print(df.isnull().sum())

## 4. Estruturação dos dados cadastrais de Cerqueira César

O arquivo `_Dados animais - TCC Ananda.xlsx` apresentou uma estrutura diferente de uma tabela convencional.

As variáveis estavam distribuídas em linhas e os animais em colunas. Além disso, diferentes grupos de informações foram armazenados verticalmente no mesmo arquivo.

Por esse motivo, antes da transformação dos dados foi necessário localizar em quais linhas cada grupo de informação estava armazenado.

In [ ]:
dados_animais_raw = pd.read_excel(
    PASTA_CC / "_Dados animais - TCC Ananda.xlsx",
    sheet_name="Sheet1",
    header=None
)

display(dados_animais_raw.iloc[:10, :15])

In [ ]:
dados_animais = dados_animais_raw.T

display(dados_animais.head(10))

In [ ]:
controle_jul = pd.read_excel(
    PASTA_CC / "2026_07_28 Controle Leiteiro v1 - Lotes.xlsx",
    sheet_name="Lotes",
    header=3
)

display(controle_jul.head(15))
print(controle_jul.columns.tolist())

In [ ]:
dados = pd.read_excel(
    PASTA_CC / "_Dados animais - TCC Ananda.xlsx",
    sheet_name="Sheet1",
    header=None
)

for i, valor in enumerate(dados.iloc[0]):
    if pd.notna(valor):
        print(i, "->", valor)

In [ ]:
dados = pd.read_excel(
    PASTA_CC / "_Dados animais - TCC Ananda.xlsx",
    sheet_name="Sheet1",
    header=None
)

for linha in range(0, 5):
    print(f"\n--- LINHA {linha} ---")
    
    for coluna in range(dados.shape[1]):
        valor = dados.iloc[linha, coluna]
        
        if pd.notna(valor):
            print(f"coluna {coluna}: {valor}")

In [ ]:
import pandas as pd

dados = pd.read_excel(
    PASTA_CC / "_Dados animais - TCC Ananda.xlsx",
    sheet_name="Sheet1",
    header=None
)

# Procura os nomes das tabelas em TODA a planilha
termos = [
    "ANIMAL",
    "PARTO",
    "LACTACAO",
    "LACTAÇÃO",
    "PESAGEM",
    "PRODUCAO_LEITE",
    "PRODUÇÃO_LEITE",
    "PRODUCAO",
    "PRODUÇÃO",
    "DOENCA",
    "DOENÇA",
    "DIAGNOSTICO",
    "DIAGNÓSTICO"
]

for linha in range(dados.shape[0]):
    for coluna in range(dados.shape[1]):
        valor = dados.iloc[linha, coluna]

        if pd.notna(valor):
            texto = str(valor).strip().upper()

            if texto in termos:
                print(
                    f"{texto} -> linha {linha}, coluna {coluna}"
                )

### 4.1 Identificação dos blocos

A inspeção permitiu identificar seis blocos principais no arquivo:

- **Animal:** cadastro individual dos animais;
- **Parto:** informações relativas aos partos;
- **Lactação:** informações relativas às lactações;
- **Pesagem:** estrutura destinada aos registros de peso;
- **Doença:** informações sanitárias;
- **Diagnóstico:** ocorrência de diagnósticos associados aos animais.

A identificação das posições foi necessária para permitir a extração de cada bloco sem misturar diferentes tipos de informação.

In [ ]:
def transformar_bloco(df, inicio, fim):
    # Recorta o bloco
    bloco = df.iloc[inicio:fim].copy()

    # Remove a primeira linha, que contém apenas o nome da tabela
    bloco = bloco.iloc[1:]

    # Remove colunas completamente vazias
    bloco = bloco.dropna(axis=1, how="all")

    # Transpõe: animais passam a ocupar as linhas
    bloco = bloco.T

    # Primeira linha passa a ser o cabeçalho
    bloco.columns = bloco.iloc[0]

    # Remove a linha utilizada como cabeçalho
    bloco = bloco.iloc[1:]

    # Remove linhas totalmente vazias
    bloco = bloco.dropna(how="all")

    # Reinicia o índice
    bloco = bloco.reset_index(drop=True)

    return bloco

In [ ]:
animal = transformar_bloco(dados, 0, 7)

parto = transformar_bloco(dados, 7, 15)

lactacao = transformar_bloco(dados, 15, 27)

pesagem = transformar_bloco(dados, 27, 38)

doenca = transformar_bloco(dados, 38, 45)

diagnostico = transformar_bloco(dados, 45, 52)

In [ ]:
tabelas = {
    "ANIMAL": animal,
    "PARTO": parto,
    "LACTAÇÃO": lactacao,
    "PESAGEM": pesagem,
    "DOENÇA": doenca,
    "DIAGNÓSTICO": diagnostico
}

for nome, tabela in tabelas.items():
    print("\n" + "=" * 70)
    print(nome)
    print("Formato:", tabela.shape)
    print("Colunas:", tabela.columns.tolist())
    display(tabela.head())

In [ ]:
for linha in range(dados.shape[0]):
    valores = []

    for coluna in range(min(10, dados.shape[1])):
        valor = dados.iloc[linha, coluna]

        if pd.notna(valor):
            valores.append(f"col {coluna}: {valor}")

    print(f"LINHA {linha}: {' | '.join(valores)}")

### 4.2 Transformação dos blocos em tabelas

Como os animais estavam organizados em colunas, foi criada uma função para transformar cada bloco em uma estrutura tabular convencional.

Após a transformação, cada **linha representa um registro** e cada **coluna representa uma variável**.

Essa estrutura é mais adequada tanto para armazenamento posterior em banco de dados quanto para análise utilizando Pandas e algoritmos de Aprendizado de Máquina.

In [ ]:
import pandas as pd

def extrair_bloco(dados, linha_inicio, linha_fim):
    """
    Extrai um bloco vertical da planilha.
    
    linha_inicio = primeira linha que contém os nomes dos campos
    linha_fim = última linha que contém os campos
    """

    # Nomes das variáveis estão na coluna 1
    campos = dados.iloc[linha_inicio:linha_fim + 1, 1].tolist()

    # Dados dos animais começam na coluna 3
    valores = dados.iloc[linha_inicio:linha_fim + 1, 3:].T.copy()

    # Define os campos como nomes das colunas
    valores.columns = campos

    # Remove linhas completamente vazias
    valores = valores.dropna(how="all")

    # Reinicia o índice
    valores = valores.reset_index(drop=True)

    return valores

In [ ]:
tabelas = {
    "ANIMAL": animal,
    "PARTO": parto,
    "LACTAÇÃO": lactacao,
    "PESAGEM": pesagem,
    "DOENÇA": doenca,
    "DIAGNÓSTICO": diagnostico
}

for nome, tabela in tabelas.items():
    print("\n" + "=" * 70)
    print(nome)
    print("Formato:", tabela.shape)
    print("Colunas:", tabela.columns.tolist())
    display(tabela.head())

### 4.3 Resultado da estruturação

Após a transformação foram obtidas tabelas independentes para cadastro animal, parto, lactação, pesagem, doença e diagnóstico.

A tabela de animais apresentou **92 animais cadastrados**.

Foi observado que a estrutura destinada à pesagem não possui registros preenchidos para essa propriedade. Portanto, a ausência dessa informação deverá ser considerada posteriormente durante a integração das bases.

Também foi observado que alguns campos necessitam de revisão semântica antes da inclusão no banco de dados. Um exemplo é o campo denominado `id_parto` presente nos registros de lactação, que contém datas em alguns registros e deverá ser avaliado antes da criação dos relacionamentos definitivos.

## 5. Análise dos controles leiteiros de Cerqueira César

Os registros de produção leiteira foram fornecidos separadamente em arquivos de controle leiteiro mensal.

Como esses arquivos serão uma das principais fontes para o desenvolvimento do modelo preditivo, foi necessário compreender sua estrutura antes da consolidação.

Foram disponibilizados controles correspondentes a sete datas diferentes de avaliação.

### 5.1 Identificação automática das tabelas internas

A inspeção visual mostrou que cada aba de controle leiteiro contém mais de uma tabela posicionada horizontalmente.

Para evitar a definição manual da posição de cada tabela, foi utilizado o campo **Brinco** como marcador para localizar automaticamente o início dos blocos.

A análise mostrou que todos os controles apresentam o mesmo padrão principal:

- primeiro bloco com `Brinco` na coluna 2;
- segundo bloco com `Brinco` na coluna 14;
- cabeçalhos localizados na mesma linha.

Essa consistência permite automatizar a leitura de todos os arquivos.

In [ ]:
import pandas as pd
import os

arquivos_controle = [
    str(PASTA_CC / "2025_01_27 Controle Leiteiro v1 - Lotes.xlsx"),
    str(PASTA_CC / "2025_02_23 Controle Leiteiro v1 - Lotes.xlsx"),
    str(PASTA_CC / "2025_03_26 Controle Leiteiro - Lotes.xlsx"),
    str(PASTA_CC / "2026_04_27 Controle Leiteiro v1 - Lotes.xlsx"),
    str(PASTA_CC / "2026_05_27 Controle Leiteiro v1 - Lotes.xlsx"),
    str(PASTA_CC / "2026_06_30 Controle Leiteiro v1 - Lotes.xlsx"),
    str(PASTA_CC / "2026_07_28 Controle Leiteiro v1 - Lotes.xlsx")
]

for arquivo in arquivos_controle:
    
    print("\n" + "=" * 80)
    print(arquivo)

    excel = pd.ExcelFile(arquivo)

    for aba in excel.sheet_names:
        
        df = pd.read_excel(
            arquivo,
            sheet_name=aba,
            header=None
        )

        print(f"\nABA: {aba}")
        print("Formato:", df.shape)

        encontrados = []

        for linha in range(df.shape[0]):
            for coluna in range(df.shape[1]):

                valor = df.iloc[linha, coluna]

                if pd.notna(valor):
                    texto = str(valor).strip().lower()

                    if texto == "brinco":
                        encontrados.append(
                            (linha, coluna)
                        )

        print("Posições de 'Brinco':", encontrados)

### 5.2 Variáveis disponíveis nos controles leiteiros

Após identificar os dois blocos de cada planilha, foram analisados seus cabeçalhos.

As informações disponíveis incluem:

- identificação do animal pelo brinco;
- lote;
- data do último parto;
- previsão de parto;
- dias em lactação (DEL);
- data do controle;
- produção de leite;
- produção registrada anteriormente;
- estado reprodutivo.

Essas variáveis são relevantes para o objetivo da pesquisa, pois permitem relacionar a produção leiteira com características produtivas e reprodutivas dos animais.

In [ ]:
for arquivo in arquivos_controle:

    excel = pd.ExcelFile(arquivo)

    for aba in excel.sheet_names:

        df = pd.read_excel(
            arquivo,
            sheet_name=aba,
            header=None
        )

        print("\n" + "=" * 80)
        print("ARQUIVO:", arquivo)
        print("ABA:", aba)

        print("\nBLOCO 1:")
        for coluna in range(1, 14):
            valor = df.iloc[4, coluna]

            if pd.notna(valor):
                print(f"Coluna {coluna}: {valor}")

        print("\nBLOCO 2:")
        for coluna in range(14, min(24, df.shape[1])):
            valor = df.iloc[4, coluna]

            if pd.notna(valor):
                print(f"Coluna {coluna}: {valor}")

### 5.3 Padronização dos controles leiteiros

Como todos os arquivos apresentam uma estrutura semelhante, foi criada uma função para extrair automaticamente os dois blocos de cada aba.

Os dois blocos são convertidos para o mesmo conjunto de variáveis e posteriormente concatenados verticalmente.

Também são armazenadas informações sobre o arquivo e a aba de origem. Esses campos são importantes para rastreabilidade, pois permitem identificar posteriormente de onde cada registro foi extraído.

O objetivo desta etapa é produzir uma tabela única contendo os registros de produção de todos os controles disponíveis.

In [ ]:
import pandas as pd

def extrair_controle(arquivo, aba):
    """
    Extrai os dois blocos de controle leiteiro existentes
    em uma aba e os transforma em uma única tabela.
    """

    df = pd.read_excel(
        arquivo,
        sheet_name=aba,
        header=None
    )

    # -------------------------
    # BLOCO 1
    # -------------------------

    bloco1 = df.iloc[5:, 1:11].copy()

    bloco1.columns = [
        "numero",
        "brinco",
        "lote",
        "data_ultimo_parto",
        "data_parto_prevista",
        "del",
        "data_prova",
        "kg_leite",
        "kg_anterior",
        "estado_reprodutivo"
    ]

    # -------------------------
    # BLOCO 2
    # -------------------------

    bloco2 = df.iloc[5:, 13:23].copy()

    bloco2.columns = [
        "numero",
        "brinco",
        "lote",
        "data_ultimo_parto",
        "data_parto_prevista",
        "del",
        "data_prova",
        "kg_leite",
        "kg_anterior",
        "estado_reprodutivo"
    ]

    # Junta os dois blocos
    resultado = pd.concat(
        [bloco1, bloco2],
        ignore_index=True
    )

    # Remove apenas linhas em que não existe brinco
    resultado = resultado[
        resultado["brinco"].notna()
    ].copy()

    # Guarda a origem do dado
    resultado["arquivo_origem"] = arquivo
    resultado["aba_origem"] = aba

    return resultado

### 5.4 Consolidação inicial dos registros

Após a extração dos diferentes arquivos e abas, os controles foram concatenados em uma única tabela denominada `producao_cc`.

Essa tabela ainda representa uma **base intermediária**, pois os registros ainda não passaram pelo processo completo de limpeza e validação.

A consolidação inicial resultou em **1.682 linhas**, com registros referentes a sete datas de controle leiteiro.

Neste momento, o número de linhas não pode ser interpretado diretamente como número de medições válidas, pois ainda podem existir registros repetidos.

In [ ]:
controles = []

for arquivo in arquivos_controle:

    excel = pd.ExcelFile(arquivo)

    for aba in excel.sheet_names:

        controle = extrair_controle(
            arquivo,
            aba
        )

        controles.append(controle)

producao_cc = pd.concat(
    controles,
    ignore_index=True
)

In [ ]:
print("Formato:", producao_cc.shape)

print("\nColunas:")
print(producao_cc.columns.tolist())

display(producao_cc.head(20))

In [ ]:
print("Total de registros:", len(producao_cc))

print("Brincos diferentes:", producao_cc["brinco"].nunique())

print("\nDatas de prova encontradas:")
print(
    producao_cc["data_prova"]
    .value_counts(dropna=False)
)

In [ ]:
display(
    producao_cc[
        [
            "brinco",
            "lote",
            "del",
            "data_prova",
            "kg_leite",
            "kg_anterior",
            "estado_reprodutivo"
        ]
    ].head(50)
)

In [ ]:
# Garantir que as datas estejam no formato datetime
producao_cc["data_prova"] = pd.to_datetime(
    producao_cc["data_prova"],
    errors="coerce"
)

print("TOTAL DE REGISTROS:")
print(len(producao_cc))

print("\nBRINCOS DIFERENTES:")
print(producao_cc["brinco"].nunique())

print("\nREGISTROS POR DATA:")
print(
    producao_cc["data_prova"]
    .value_counts()
    .sort_index()
)

print("\nREGISTROS POR ARQUIVO/ABA:")
print(
    producao_cc
    .groupby(["arquivo_origem", "aba_origem"])
    .size()
)

## 6. Investigação de registros duplicados

Após a consolidação, foi realizada uma análise para verificar se uma mesma medição havia sido registrada mais de uma vez.

Inicialmente foi utilizada como chave de identificação da medição a combinação:

**animal + data da prova + produção de leite**

Essa chave foi utilizada apenas para investigação das possíveis duplicidades e poderá ser refinada durante as etapas posteriores de preparação dos dados.

In [ ]:
duplicados = producao_cc[
    producao_cc.duplicated(
        subset=[
            "brinco",
            "data_prova",
            "kg_leite"
        ],
        keep=False
    )
].sort_values(
    ["data_prova", "brinco"]
)

print("Quantidade de registros potencialmente duplicados:")
print(len(duplicados))

display(
    duplicados[
        [
            "brinco",
            "data_prova",
            "kg_leite",
            "lote",
            "arquivo_origem",
            "aba_origem"
        ]
    ].head(50)
)

In [ ]:
# ============================================================
# INVESTIGAR DUPLICATAS DOS CONTROLES LEITEIROS
# ============================================================

# Consideramos uma medição igual quando:
# mesmo animal + mesma data + mesma produção de leite

chave = ["brinco", "data_prova", "kg_leite"]

# 1. Total atual
print("TOTAL ATUAL:", len(producao_cc))

# 2. Quantidade de registros únicos
unicos = producao_cc.drop_duplicates(subset=chave)

print("TOTAL SE REMOVER DUPLICATAS:", len(unicos))
print("REGISTROS REPETIDOS:", len(producao_cc) - len(unicos))


# 3. Ver duplicações por data
duplicados = producao_cc[
    producao_cc.duplicated(subset=chave, keep=False)
].copy()

print("\nDUPLICAÇÕES POR DATA:")
print(
    duplicados.groupby("data_prova")
    .size()
    .sort_index()
)


# 4. Quantas vezes cada observação aparece
contagem = (
    producao_cc
    .groupby(chave, dropna=False)
    .size()
    .reset_index(name="quantidade")
)

print("\nFREQUÊNCIA DAS REPETIÇÕES:")
print(contagem["quantidade"].value_counts().sort_index())


# 5. Investigar especificamente maio
maio = producao_cc[
    producao_cc["data_prova"].dt.strftime("%Y-%m-%d") == "2026-05-27"
]

print("\nMAIO POR ABA:")
print(
    maio.groupby("aba_origem")
    .size()
)

print("\nREGISTROS ÚNICOS DE MAIO POR ABA:")
for aba in maio["aba_origem"].dropna().unique():

    temp = maio[maio["aba_origem"] == aba]

    print(
        aba,
        "-> total:",
        len(temp),
        "| únicos:",
        len(temp.drop_duplicates(subset=chave))
    )

### 6.1 Resultado da análise de duplicidade

A análise mostrou que grande parte das 1.682 linhas correspondia a repetições de uma mesma medição.

Foram observadas medições aparecendo duas, quatro e, em alguns casos, até oito vezes.

A investigação mostrou que essas duplicações são decorrentes principalmente da própria estrutura dos arquivos Excel, nos quais determinados registros aparecem repetidos em diferentes regiões da planilha.

No controle de maio também existem duas abas, `Lotes` e `Lotes (2)`, contendo grande quantidade de registros coincidentes.

Após considerar somente registros com data de prova válida e eliminar as repetições com base na chave utilizada para análise, foram identificadas **736 medições únicas**.

Nenhum registro foi removido definitivamente nesta etapa. A análise foi realizada primeiro para compreender a origem das duplicações.

In [ ]:
def extrair_controle(arquivo, aba):

    df = pd.read_excel(
        arquivo,
        sheet_name=aba,
        header=None
    )

    colunas_padrao = [
        "numero",
        "brinco",
        "lote",
        "data_ultimo_parto",
        "data_parto_prevista",
        "del",
        "data_prova",
        "kg_leite",
        "kg_anterior",
        "estado_reprodutivo"
    ]

    # BLOCO 1
    bloco1 = df.iloc[5:, 1:11].copy()
    bloco1.columns = colunas_padrao

    bloco1["linha_excel"] = bloco1.index
    bloco1["bloco_origem"] = "bloco_1"

    # BLOCO 2
    bloco2 = df.iloc[5:, 13:23].copy()
    bloco2.columns = colunas_padrao

    bloco2["linha_excel"] = bloco2.index
    bloco2["bloco_origem"] = "bloco_2"

    resultado = pd.concat(
        [bloco1, bloco2],
        ignore_index=True
    )

    resultado = resultado[
        resultado["brinco"].notna()
    ].copy()

    resultado["arquivo_origem"] = arquivo
    resultado["aba_origem"] = aba

    return resultado

### 6.2 Rastreamento da origem das duplicações

Para compreender por que uma mesma observação aparecia várias vezes, a função de extração foi temporariamente ampliada para registrar:

- bloco de origem;
- linha original do Excel;
- aba de origem;
- arquivo de origem.

Essa estratégia possibilitou rastrear cada observação até sua posição original na planilha.

Por exemplo, uma mesma medição poderia aparecer em duas linhas diferentes da mesma região do arquivo, confirmando que a duplicação não foi causada pelo código de concatenação, mas pela estrutura dos arquivos recebidos.

In [ ]:
controles = []

for arquivo in arquivos_controle:

    excel = pd.ExcelFile(arquivo)

    for aba in excel.sheet_names:

        controle = extrair_controle(
            arquivo,
            aba
        )

        controles.append(controle)

producao_cc_debug = pd.concat(
    controles,
    ignore_index=True
)

producao_cc_debug["data_prova"] = pd.to_datetime(
    producao_cc_debug["data_prova"],
    errors="coerce"
)

In [ ]:
display(
    producao_cc_debug[
        (producao_cc_debug["brinco"].astype(str).str.upper() == "ATENA 66")
        &
        (producao_cc_debug["data_prova"] == "2026-01-27")
    ][
        [
            "brinco",
            "lote",
            "del",
            "data_prova",
            "kg_leite",
            "kg_anterior",
            "estado_reprodutivo",
            "bloco_origem",
            "linha_excel",
            "aba_origem"
        ]
    ]
)

In [ ]:
display(
    producao_cc_debug[
        (producao_cc_debug["brinco"].astype(str).str.upper() == "ATENA 66")
        &
        (producao_cc_debug["data_prova"] == "2026-05-27")
    ][
        [
            "brinco",
            "kg_leite",
            "kg_anterior",
            "del",
            "bloco_origem",
            "linha_excel",
            "aba_origem"
        ]
    ]
)

### 6.3 Investigação específica do controle de maio

O controle referente a maio apresentou aproximadamente o dobro de registros dos demais meses.

Esse arquivo possui duas abas, `Lotes` e `Lotes (2)`.

Por esse motivo, foi realizada uma análise específica para verificar se as abas continham novos registros ou cópias dos mesmos controles.

A análise mostrou que cada aba contém aproximadamente 200 registros extraídos e que grande parte das medições se repete entre as duas abas, além das repetições existentes internamente em cada uma delas.

In [ ]:
# ============================================================
# INVESTIGAR DUPLICAÇÕES DE MAIO
# ============================================================

maio = producao_cc_debug[
    producao_cc_debug["data_prova"] == pd.Timestamp("2026-05-27")
].copy()

print("TOTAL DE LINHAS EM MAIO:", len(maio))

print("\nPOR ABA:")
print(maio["aba_origem"].value_counts())

print("\nPOR ABA E BLOCO:")
print(
    maio.groupby(
        ["aba_origem", "bloco_origem"]
    ).size()
)

# --------------------------------------------
# Ver alguns registros duplicados
# --------------------------------------------

chave = [
    "brinco",
    "data_prova",
    "kg_leite"
]

duplicados_maio = maio[
    maio.duplicated(
        subset=chave,
        keep=False
    )
].sort_values(
    ["brinco", "aba_origem", "linha_excel"]
)

display(
    duplicados_maio[
        [
            "brinco",
            "lote",
            "del",
            "data_prova",
            "kg_leite",
            "kg_anterior",
            "estado_reprodutivo",
            "bloco_origem",
            "linha_excel",
            "aba_origem"
        ]
    ].head(30)
)

## 7. Análise de conflitos entre registros duplicados

Antes de remover registros repetidos, foi necessário verificar se todas as cópias eram completamente iguais.

Foram comparados os seguintes atributos:

- lote;
- data do último parto;
- previsão de parto;
- DEL;
- produção anterior;
- estado reprodutivo.

A finalidade dessa análise é evitar que a remoção automática de duplicatas provoque perda de informações.

In [ ]:
# ============================================================
# ANALISAR CONFLITOS ENTRE REGISTROS DUPLICADOS
# ============================================================

chave_medicao = [
    "brinco",
    "data_prova",
    "kg_leite"
]

campos_analisar = [
    "lote",
    "data_ultimo_parto",
    "data_parto_prevista",
    "del",
    "kg_anterior",
    "estado_reprodutivo"
]

conflitos = []

for chave, grupo in producao_cc_debug.groupby(
    chave_medicao,
    dropna=False
):

    if len(grupo) <= 1:
        continue

    conflito_grupo = {}

    for campo in campos_analisar:

        valores = (
            grupo[campo]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()
        )

        if len(valores) > 1:
            conflito_grupo[campo] = valores.tolist()

    if conflito_grupo:

        conflitos.append({
            "brinco": chave[0],
            "data_prova": chave[1],
            "kg_leite": chave[2],
            "quantidade_copias": len(grupo),
            "conflitos": conflito_grupo
        })

print("Grupos duplicados com algum conflito:")
print(len(conflitos))

display(pd.DataFrame(conflitos).head(30))

### 7.1 Resultado da análise de conflitos

Foram encontrados **20 grupos de medições duplicadas contendo alguma informação divergente**.

A maioria das divergências está relacionada ao nome do lote, como diferentes formas de registrar grupos semelhantes.

Também foram observados alguns conflitos em informações reprodutivas, por exemplo registros em que uma cópia apresenta `DG +` e outra apresenta `Abortou`.

Esses casos não devem ser resolvidos automaticamente, pois representam possíveis diferenças semânticas relevantes.

Por esse motivo, os registros conflitantes serão mantidos separadamente para revisão antes da construção da base analítica final.

In [ ]:
print("TOTAL DE MEDIÇÕES ÚNICAS:")
print(
    producao_cc_debug[
        producao_cc_debug["data_prova"].notna()
    ]
    .drop_duplicates(
        subset=["brinco", "data_prova", "kg_leite"]
    )
    .shape[0]
)

In [ ]:
# ============================================================
# CONSOLIDAR MEDIÇÕES DUPLICADAS
# ============================================================

chave_medicao = [
    "brinco",
    "data_prova",
    "kg_leite"
]

def primeiro_valor_nao_nulo(serie):
    valores = serie.dropna()

    if len(valores) == 0:
        return None

    return valores.iloc[0]


producao_cc_limpa = (
    producao_cc_debug
    .groupby(chave_medicao, as_index=False)
    .agg({
        "numero": primeiro_valor_nao_nulo,
        "lote": primeiro_valor_nao_nulo,
        "data_ultimo_parto": primeiro_valor_nao_nulo,
        "data_parto_prevista": primeiro_valor_nao_nulo,
        "del": primeiro_valor_nao_nulo,
        "kg_anterior": primeiro_valor_nao_nulo,
        "estado_reprodutivo": primeiro_valor_nao_nulo
    })
)

print("TOTAL APÓS CONSOLIDAÇÃO:")
print(len(producao_cc_limpa))

display(producao_cc_limpa.head(20))

In [ ]:
conflitos_df = pd.DataFrame(conflitos)

display(conflitos_df)

In [ ]:
conflitos_df.to_excel(
    "conflitos_controle_leiteiro.xlsx",
    index=False
)

In [ ]:
conflitos_reprod = []

for item in conflitos:
    if "estado_reprodutivo" in item["conflitos"]:
        conflitos_reprod.append(item)

conflitos_reprod_df = pd.DataFrame(conflitos_reprod)

display(conflitos_reprod_df)

## 8. Investigação detalhada dos registros conflitantes

A análise anterior identificou 20 grupos de medições duplicadas que apresentam divergências em pelo menos uma das informações associadas ao controle leiteiro.

Antes de definir uma regra para consolidação desses registros, é necessário identificar quais variáveis apresentam divergências e com que frequência isso ocorre.

Esta etapa tem como objetivos:

- identificar quais campos são responsáveis pelos conflitos;
- verificar se as divergências seguem algum padrão;
- diferenciar variações de nomenclatura de possíveis diferenças reais de informação;
- evitar a perda de informações relevantes durante a remoção das duplicidades.

Nenhuma informação será removida nesta etapa. Os registros serão apenas analisados para definição posterior das regras de consolidação.

In [ ]:
# Contagem dos campos que apresentam divergências entre registros duplicados

contagem_conflitos = {
    campo: 0
    for campo in campos_analisar
}

for conflito in conflitos:
    campos_com_conflito = conflito["conflitos"]

    for campo in campos_analisar:
        if campo in campos_com_conflito:
            contagem_conflitos[campo] += 1

contagem_conflitos = pd.Series(
    contagem_conflitos,
    name="Quantidade de conflitos"
).sort_values(ascending=False)

contagem_conflitos

### 8.1 Distribuição dos conflitos por variável

A análise mostrou que os conflitos identificados entre registros duplicados estão concentrados em apenas duas variáveis.

Dos 20 grupos conflitantes:

- 17 apresentam divergências no campo `lote`;
- 3 apresentam divergências no campo `estado_reprodutivo`;
- não foram identificados conflitos em `data_ultimo_parto`, `data_parto_prevista`, `DEL` ou `kg_anterior`.

Esse resultado indica que as principais informações produtivas associadas às medições são consistentes entre as cópias.

As divergências no campo `lote` podem estar relacionadas a diferentes formas de nomenclatura ou atualização da classificação dos animais. Já os conflitos em `estado_reprodutivo` podem representar diferenças semanticamente relevantes e, por isso, serão analisados individualmente antes da consolidação definitiva.

In [ ]:
# Selecionar somente os conflitos relacionados ao estado reprodutivo

conflitos_estado_reprodutivo = [
    conflito
    for conflito in conflitos
    if "estado_reprodutivo" in conflito["conflitos"]
]

conflitos_estado_reprodutivo_df = pd.DataFrame(
    conflitos_estado_reprodutivo
)

display(conflitos_estado_reprodutivo_df)

### 8.2 Conflitos no estado reprodutivo

Foram identificados três grupos de registros duplicados com divergência no campo `estado_reprodutivo`.

Os casos encontrados foram:

- FAISCA 1111: `DG +` e `Abortou`;
- FIONA 1108: `Negativo` e `Abortou`;
- FLECHA-1100: `Negativo` e `Abortou`.

Essas divergências não devem ser resolvidas automaticamente, pois representam informações com significados distintos e potencialmente relevantes para a análise do desempenho produtivo.

Por esse motivo, será realizada uma inspeção dos registros originais correspondentes a esses animais e datas, preservando a rastreabilidade da informação antes da definição da regra de consolidação.

In [ ]:
casos_reprodutivos = [
    ("FAISCA 1111", "2026-01-27"),
    ("FIONA 1108", "2026-02-23"),
    ("FLECHA-1100", "2026-03-26")
]

for brinco, data in casos_reprodutivos:

    print("\n" + "=" * 80)
    print("ANIMAL:", brinco)
    print("DATA:", data)

    caso = producao_cc_debug[
        (producao_cc_debug["brinco"].astype(str).str.strip().str.upper() == brinco.upper())
        &
        (producao_cc_debug["data_prova"] == pd.Timestamp(data))
    ]

    display(
        caso[
            [
                "brinco",
                "lote",
                "data_ultimo_parto",
                "data_parto_prevista",
                "del",
                "data_prova",
                "kg_leite",
                "kg_anterior",
                "estado_reprodutivo",
                "bloco_origem",
                "linha_excel",
                "arquivo_origem",
                "aba_origem"
            ]
        ]
    )

### 8.3 Rastreamento dos conflitos reprodutivos

A inspeção dos registros originais mostrou um padrão consistente nos três conflitos relacionados ao estado reprodutivo.

Nos casos de FAISCA 1111, FIONA 1108 e FLECHA-1100, as informações de lote, último parto, previsão de parto, DEL, produção de leite e produção anterior são idênticas entre as duas cópias.

A única divergência está no estado reprodutivo.

Além disso, nos três casos, o registro contendo `Abortou` foi extraído do segundo bloco da planilha, enquanto o outro estado reprodutivo foi extraído do primeiro bloco.

Esse padrão sugere que as divergências não são aleatórias e podem estar relacionadas à organização original das planilhas ou à forma como determinadas informações reprodutivas foram atualizadas.

Antes da consolidação definitiva, será investigada a estrutura dos registros próximos a essas posições para compreender o significado dos dois blocos e determinar se existe uma regra consistente para seleção da informação.

In [ ]:
# Investigar o contexto dos registros conflitantes nos dois blocos

for brinco, data in casos_reprodutivos:

    print("\n" + "=" * 100)
    print(f"ANIMAL: {brinco} | DATA: {data}")

    arquivo = producao_cc_debug[
        (producao_cc_debug["brinco"].astype(str).str.strip().str.upper() == brinco.upper())
        &
        (producao_cc_debug["data_prova"] == pd.Timestamp(data))
    ]["arquivo_origem"].iloc[0]

    dados_mes = producao_cc_debug[
        producao_cc_debug["arquivo_origem"] == arquivo
    ]

    # Registros próximos da linha 121 do bloco 1
    print("\n--- CONTEXTO DO BLOCO 1 ---")

    display(
        dados_mes[
            (dados_mes["bloco_origem"] == "bloco_1")
            &
            (dados_mes["linha_excel"].between(116, 126))
        ][
            [
                "brinco",
                "lote",
                "kg_leite",
                "estado_reprodutivo",
                "bloco_origem",
                "linha_excel"
            ]
        ].sort_values("linha_excel")
    )

    # Registros próximos da linha 14 do bloco 2
    print("\n--- CONTEXTO DO BLOCO 2 ---")

    display(
        dados_mes[
            (dados_mes["bloco_origem"] == "bloco_2")
            &
            (dados_mes["linha_excel"].between(9, 19))
        ][
            [
                "brinco",
                "lote",
                "kg_leite",
                "estado_reprodutivo",
                "bloco_origem",
                "linha_excel"
            ]
        ].sort_values("linha_excel")
    )

### 8.4 Comparação entre os dois blocos dos controles leiteiros

A análise do contexto dos três conflitos reprodutivos mostrou que os registros dos dois blocos apresentam comportamento sistemático.

Nos três casos avaliados, todas as informações produtivas são iguais entre as duas cópias, incluindo lote, data do último parto, DEL, produção atual e produção anterior.

Entretanto, foram observadas diferenças no preenchimento do estado reprodutivo:

- registros vazios no primeiro bloco aparecem como `0` no segundo bloco;
- nos três conflitos identificados, o segundo bloco registra `Abortou`, enquanto o primeiro apresenta `DG +` ou `Negativo`.

Esse comportamento sugere que o segundo bloco pode conter informações reprodutivas mais completas ou atualizadas. Entretanto, antes de estabelecer uma regra de prioridade entre os blocos, será verificado se esse padrão ocorre de forma consistente em todos os controles leiteiros.

In [ ]:
# ============================================================
# COMPARAR O ESTADO REPRODUTIVO ENTRE BLOCO 1 E BLOCO 2
# ============================================================

print("VALORES DE ESTADO REPRODUTIVO ENCONTRADOS NA BASE:\n")

print(
    producao_cc_debug["estado_reprodutivo"]
    .value_counts(dropna=False)
)

In [ ]:
# Distribuição do estado reprodutivo por bloco de origem

comparacao_estado_blocos = pd.crosstab(
    producao_cc_debug["estado_reprodutivo"].fillna("AUSENTE"),
    producao_cc_debug["bloco_origem"]
)

display(comparacao_estado_blocos)

In [ ]:
# ============================================================
# COMPARAÇÃO DIRETA BLOCO 1 x BLOCO 2
# ============================================================

comparacao_blocos = producao_cc_debug[
    producao_cc_debug["data_prova"].notna()
].copy()

# Uma linha por medição e por bloco
comparacao_blocos = comparacao_blocos.drop_duplicates(
    subset=[
        "brinco",
        "data_prova",
        "kg_leite",
        "bloco_origem"
    ]
)

# Colocar o estado reprodutivo de cada bloco lado a lado
estados_lado_a_lado = comparacao_blocos.pivot_table(
    index=[
        "brinco",
        "data_prova",
        "kg_leite"
    ],
    columns="bloco_origem",
    values="estado_reprodutivo",
    aggfunc="first"
).reset_index()

display(estados_lado_a_lado.head(30))

In [ ]:
def classificar_comparacao(row):

    b1 = row.get("bloco_1")
    b2 = row.get("bloco_2")

    # Ambos ausentes
    if pd.isna(b1) and pd.isna(b2):
        return "ambos_ausentes"

    # Somente bloco 1 possui informação
    if pd.notna(b1) and pd.isna(b2):
        return "somente_bloco_1"

    # Somente bloco 2 possui informação
    if pd.isna(b1) and pd.notna(b2):
        return "somente_bloco_2"

    # Ambos possuem a mesma informação
    if str(b1).strip() == str(b2).strip():
        return "iguais"

    # Ambos possuem informação, mas são diferentes
    return "divergentes"


estados_lado_a_lado["comparacao"] = estados_lado_a_lado.apply(
    classificar_comparacao,
    axis=1
)

print(
    estados_lado_a_lado["comparacao"]
    .value_counts()
)

## 9. Validação da chave de identificação das medições

Durante a investigação inicial das duplicidades, foi utilizada a combinação `brinco + data_prova + kg_leite` para identificar registros repetidos.

Entretanto, a inclusão de `kg_leite` na própria chave pode fazer com que duas produções diferentes registradas para o mesmo animal na mesma data sejam consideradas medições distintas, em vez de possíveis registros conflitantes.

Antes da consolidação definitiva da base, será verificado se existem casos em que um mesmo animal apresenta mais de um valor de produção de leite (`kg_leite`) na mesma data de controle (`data_prova`).

Essa verificação é necessária para definir de forma mais segura a unidade de observação da base e evitar que inconsistências sejam incorporadas posteriormente à etapa de modelagem.

In [ ]:
# Verificar quantos valores diferentes de produção existem
# para o mesmo animal na mesma data de controle

producao_por_animal_data = (
    producao_cc_debug[
        producao_cc_debug["data_prova"].notna()
    ]
    .groupby(["brinco", "data_prova"])["kg_leite"]
    .nunique(dropna=False)
    .reset_index(name="quantidade_valores_kg_leite")
)

conflitos_producao = producao_por_animal_data[
    producao_por_animal_data["quantidade_valores_kg_leite"] > 1
]

print(
    "Total de combinações animal + data:",
    len(producao_por_animal_data)
)

print(
    "Casos com mais de um valor de kg_leite:",
    len(conflitos_producao)
)

display(conflitos_producao)

### 9.1 Identificação de divergências na produção de leite

A análise identificou 733 combinações distintas entre animal e data de controle.

Dessas combinações, três apresentam mais de um valor de `kg_leite` registrado para o mesmo animal na mesma data:

- Isis-1217, em 28/07/2026;
- JACIARA-1259, em 23/02/2026;
- JULIETE 1250, em 27/01/2026.

Esse resultado demonstra que a inclusão de `kg_leite` na chave de duplicidade utilizada anteriormente fazia com que esses registros fossem classificados como medições distintas, não sendo identificados como possíveis conflitos.

Como a produção de leite é uma variável central para as etapas posteriores do trabalho, os três casos serão rastreados até os registros de origem antes da definição da chave definitiva e da regra de consolidação.

Nenhum dos valores será removido ou substituído nesta etapa.

In [ ]:
# Inspecionar todos os registros correspondentes aos casos
# com mais de um valor de produção para o mesmo animal e data

casos_kg_conflitante = producao_cc_debug.merge(
    conflitos_producao[["brinco", "data_prova"]],
    on=["brinco", "data_prova"],
    how="inner"
)

colunas_exibir = [
    "brinco",
    "lote",
    "data_ultimo_parto",
    "data_parto_prevista",
    "del",
    "data_prova",
    "kg_leite",
    "kg_anterior",
    "estado_reprodutivo",
    "bloco_origem",
    "linha_excel",
    "arquivo_origem",
    "aba_origem"
]

display(
    casos_kg_conflitante[colunas_exibir]
    .sort_values(["brinco", "data_prova", "kg_leite", "bloco_origem"])
)

### 9.2 Rastreamento dos conflitos de produção

A inspeção dos três casos com mais de um valor de produção para o mesmo animal e data mostrou que as divergências apresentam padrões diferentes.

Para Isis-1217, em 28/07/2026, foram encontrados valores de 17,6 kg e 38,6 kg provenientes de blocos diferentes da planilha. A diferença entre os valores é expressiva e requer consulta à estrutura original do controle leiteiro.

Para JACIARA-1259, em 23/02/2026, foram encontrados valores de 24,4 kg e 28,6 kg. Os registros foram extraídos de posições distintas do mesmo bloco.

Para JULIETE 1250, em 27/01/2026, foram encontrados valores de 28,6 kg e 28,8 kg, também provenientes de posições distintas do mesmo bloco.

Esses resultados demonstram que a combinação `brinco + data_prova` não pode ser utilizada para remoção automática de duplicidades sem uma investigação adicional.

Os três casos serão analisados diretamente em seus arquivos de origem para determinar se representam duplicações estruturais, correções de registros, diferentes medições ou inconsistências da base.

In [ ]:
# Inspeção da estrutura original dos três casos com conflito de produção

casos_inspecao = [
    (
        PASTA_CC / "2025_01_27 Controle Leiteiro v1 - Lotes.xlsx",
        "JULIETE 1250"
    ),
    (
        PASTA_CC / "2025_02_23 Controle Leiteiro v1 - Lotes.xlsx",
        "JACIARA-1259"
    ),
    (
        PASTA_CC / "2026_07_28 Controle Leiteiro v1 - Lotes.xlsx",
        "Isis-1217"
    )
]

for arquivo, nome_animal in casos_inspecao:

    print("\n" + "=" * 100)
    print("ARQUIVO:", arquivo.name)
    print("ANIMAL:", nome_animal)

    bruto = pd.read_excel(
        arquivo,
        sheet_name="Lotes",
        header=None
    )

    # Procurar o animal em qualquer coluna da planilha
    mascara = bruto.apply(
        lambda coluna: coluna.astype(str)
        .str.strip()
        .str.upper()
        .eq(nome_animal.upper())
    )

    posicoes = mascara.stack()
    posicoes = posicoes[posicoes].index.tolist()

    print("POSIÇÕES ENCONTRADAS (índice DataFrame, coluna):")
    print(posicoes)

    for linha, coluna in posicoes:
        print(
            f"\n--- Ocorrência próxima da linha {linha + 1} "
            f"do Excel / coluna {coluna} ---"
        )

        inicio = max(0, linha - 3)
        fim = min(len(bruto), linha + 4)

        display(
            bruto.iloc[inicio:fim, max(0, coluna - 3):coluna + 8]
        )

### 9.3 Verificação dos conflitos nos arquivos originais

A consulta direta aos arquivos originais confirmou que os três conflitos de produção não foram introduzidos pelo processo de extração realizado no notebook. Os diferentes valores já estão presentes nas planilhas fornecidas pela propriedade.

Foram observados os seguintes casos:

- JULIETE 1250, em 27/01/2026: 28,6 kg e 28,8 kg;
- JACIARA-1259, em 23/02/2026: 28,6 kg e 24,4 kg;
- Isis-1217, em 28/07/2026: 17,6 kg e 38,6 kg.

Nos três casos, as demais informações associadas ao registro permanecem iguais ou substancialmente equivalentes entre as ocorrências, indicando que a divergência está especificamente relacionada ao valor da produção de leite.

A inspeção também demonstrou que as planilhas originais apresentam regiões repetidas ou diferentes blocos contendo registros dos mesmos animais. Portanto, os conflitos não podem ser resolvidos apenas pela posição do registro na planilha.

Como não há, até esta etapa, informação suficiente para determinar de forma objetiva qual dos valores conflitantes representa a medição correta, nenhuma das alternativas será selecionada automaticamente.

Os casos serão mantidos identificados para revisão e a definição da base analítica considerará explicitamente esse tipo de inconsistência antes da etapa de Machine Learning.

### 9.4 Registro das pendências de validação

Como os arquivos originais não permitem determinar de forma objetiva qual valor de produção é correto nos três casos identificados, essas ocorrências serão registradas como pendências de validação.

Essa abordagem permite preservar a rastreabilidade das inconsistências sem realizar imputações ou exclusões arbitrárias, possibilitando a continuidade das demais etapas de preparação dos dados.

A decisão sobre a utilização ou exclusão dessas observações na base destinada ao treinamento dos modelos será tomada posteriormente, após validação da informação ou definição metodológica documentada.

In [ ]:
pendencias_producao = (
    casos_kg_conflitante[
        [
            "brinco",
            "data_prova",
            "kg_leite",
            "kg_anterior",
            "arquivo_origem",
            "aba_origem",
            "bloco_origem"
        ]
    ]
    .sort_values(["brinco", "data_prova", "kg_leite"])
    .reset_index(drop=True)
)

pendencias_producao["motivo"] = (
    "Mais de um valor de kg_leite para o mesmo animal e data"
)

pendencias_producao["status_validacao"] = "Pendente"

display(pendencias_producao)

### 9.5 Revisão da definição de medição única

A investigação demonstrou que a chave inicialmente utilizada (`brinco + data_prova + kg_leite`) não é adequada para definir, isoladamente, uma medição única.

Com a utilização de `brinco + data_prova` como unidade de investigação, foram identificadas 733 combinações distintas entre animal e data de controle.

Dessas:

- 730 apresentam um único valor de produção de leite;
- 3 apresentam dois valores diferentes de produção para o mesmo animal e data;
- os 3 casos conflitantes permanecem pendentes de validação.

Portanto, o número anteriormente obtido de 736 registros únicos não deve ser interpretado como 736 medições independentes. Essa quantidade resulta da inclusão de `kg_leite` na chave, fazendo com que os três conflitos de produção fossem contabilizados duas vezes.

Para as próximas etapas, a combinação `brinco + data_prova` será utilizada como referência para identificar a unidade de observação do controle leiteiro, mantendo separadamente os casos cuja produção apresenta conflito.

In [ ]:
# Resumo da validação da unidade de observação

total_animal_data = len(producao_por_animal_data)

total_com_conflito = (
    producao_por_animal_data["quantidade_valores_kg_leite"] > 1
).sum()

total_sem_conflito = (
    producao_por_animal_data["quantidade_valores_kg_leite"] == 1
).sum()

print("RESUMO DA VALIDAÇÃO")
print("-" * 40)
print(f"Combinações animal + data: {total_animal_data}")
print(f"Sem conflito de produção: {total_sem_conflito}")
print(f"Com conflito de produção: {total_com_conflito}")

print("\nVerificação:")
print(
    "Total confere:",
    total_sem_conflito + total_com_conflito == total_animal_data
)

## 10. Definição das regras de consolidação dos registros duplicados

Após a validação da unidade de observação, a próxima etapa consiste em definir como as diferentes cópias de uma mesma medição serão consolidadas.

A análise anterior demonstrou que os blocos presentes nos controles leiteiros podem conter informações complementares. Dessa forma, selecionar apenas um dos blocos como fonte principal poderia provocar perda de informação.

O campo `estado_reprodutivo` requer atenção especial, pois foram observados registros preenchidos em apenas um dos blocos, registros concordantes e três casos com informações divergentes.

Também foi identificado o uso do valor `0` em situações nas quais outro bloco apresenta valor ausente. Antes da consolidação, será investigado se esse padrão ocorre de maneira sistemática, evitando que códigos utilizados para representar ausência de informação sejam interpretados como categorias reprodutivas reais.

In [ ]:
# Frequência dos valores de estado reprodutivo em cada bloco

estado_por_bloco = (
    producao_cc_debug
    .assign(
        estado_analise=producao_cc_debug["estado_reprodutivo"]
        .fillna("AUSENTE")
        .astype(str)
        .str.strip()
    )
    .groupby(["bloco_origem", "estado_analise"])
    .size()
    .reset_index(name="quantidade")
    .sort_values(["bloco_origem", "quantidade"], ascending=[True, False])
)

display(estado_por_bloco)

In [ ]:
# Verificar especificamente a relação entre valores ausentes e "0"
# nas cópias de uma mesma medição

comparacao_zero = estados_lado_a_lado.copy()

comparacao_zero["bloco_1"] = (
    comparacao_zero["bloco_1"]
    .fillna("AUSENTE")
    .astype(str)
    .str.strip()
)

comparacao_zero["bloco_2"] = (
    comparacao_zero["bloco_2"]
    .fillna("AUSENTE")
    .astype(str)
    .str.strip()
)

pares_estado = (
    comparacao_zero
    .groupby(["bloco_1", "bloco_2"])
    .size()
    .reset_index(name="quantidade")
    .sort_values("quantidade", ascending=False)
)

display(pares_estado.head(30))

### 10.1 Padronização dos valores ausentes no estado reprodutivo

A comparação entre os blocos mostrou que o campo `estado_reprodutivo` utiliza tanto valores ausentes quanto o valor `0` em registros sem uma categoria reprodutiva explícita.

Foram observados 123 casos com `0` no primeiro bloco e ausência no segundo, além de 121 casos com ausência no primeiro bloco e `0` no segundo. Esse padrão indica que o valor `0` está sendo utilizado nos controles como forma alternativa de representar ausência de informação, e não como uma categoria reprodutiva propriamente dita.

Dessa forma, para fins de padronização da base analítica, o valor `0` será tratado como ausente nesse campo.

A comparação também confirmou a existência de informações complementares entre os blocos. Foram encontrados, por exemplo, registros `DG +` ou `Negativo` em um bloco enquanto o outro estava ausente. Portanto, a consolidação deverá preservar a informação disponível em qualquer um dos blocos, em vez de estabelecer prioridade fixa para apenas um deles.

Permanecem três casos com divergências reprodutivas reais: dois registros `Negativo` versus `Abortou` e um registro `DG +` versus `Abortou`. Esses casos serão mantidos identificados separadamente e não serão resolvidos automaticamente.

In [ ]:
# Criar uma versão padronizada do estado reprodutivo
# sem modificar a informação original

producao_cc_debug["estado_reprodutivo_padronizado"] = (
    producao_cc_debug["estado_reprodutivo"]
    .replace(0, pd.NA)
    .replace("0", pd.NA)
)

print("Valores após a padronização:\n")

print(
    producao_cc_debug["estado_reprodutivo_padronizado"]
    .value_counts(dropna=False)
)

### 10.2 Verificação de informações complementares entre registros duplicados

Antes da consolidação dos registros, será avaliado se outras variáveis apresentam valores ausentes em uma cópia e preenchidos em outra.

Essa verificação é necessária porque a análise do estado reprodutivo demonstrou que os diferentes blocos podem conter informações complementares.

O objetivo é identificar quais campos podem ser consolidados de forma segura, preservando valores disponíveis sem substituir informações conflitantes.


In [ ]:
# Avaliar valores ausentes nas principais variáveis que serão
# consideradas durante a consolidação

campos_consolidacao = [
    "lote",
    "data_ultimo_parto",
    "data_parto_prevista",
    "del",
    "kg_anterior",
    "estado_reprodutivo_padronizado"
]

resumo_ausentes = pd.DataFrame({
    "campo": campos_consolidacao,
    "valores_ausentes": [
        producao_cc_debug[campo].isna().sum()
        for campo in campos_consolidacao
    ],
    "percentual_ausente": [
        round(
            producao_cc_debug[campo].isna().mean() * 100,
            2
        )
        for campo in campos_consolidacao
    ]
})

display(resumo_ausentes)

### 10.3 Análise de valores ausentes antes da consolidação

A avaliação das principais variáveis mostrou baixa proporção de valores ausentes em `lote`, `data_ultimo_parto`, `data_parto_prevista` e `DEL`, todas próximas de 1%.

A variável `kg_anterior` apresentou aproximadamente 6,4% de ausência, enquanto `estado_reprodutivo_padronizado` apresentou aproximadamente 34,7%.

A ausência de estado reprodutivo não será preenchida artificialmente nesta etapa, pois a inexistência de informação registrada não permite inferir com segurança a situação reprodutiva do animal.

Também foram identificadas diferentes representações computacionais de valores ausentes (`NaN` e `<NA>`). Essas representações serão uniformizadas antes da consolidação para garantir consistência nas operações posteriores.

In [ ]:
# Uniformizar as representações de ausência no estado reprodutivo

producao_cc_debug["estado_reprodutivo_padronizado"] = (
    producao_cc_debug["estado_reprodutivo"]
    .replace([0, "0"], np.nan)
)

print(
    producao_cc_debug["estado_reprodutivo_padronizado"]
    .value_counts(dropna=False)
)

### 10.4 Reavaliação dos conflitos utilizando a unidade de observação validada

As análises iniciais de conflitos utilizaram a combinação `brinco + data_prova + kg_leite`.

Após a identificação de três casos com diferentes valores de produção para o mesmo animal e data, a unidade de observação passou a ser investigada pela combinação `brinco + data_prova`.

Antes da consolidação definitiva, os conflitos entre as demais variáveis serão recalculados utilizando essa nova referência.

Essa etapa permite identificar todas as divergências existentes dentro de uma mesma combinação animal-data, incluindo os conflitos de produção anteriormente ocultados pela inclusão de `kg_leite` na chave.

In [ ]:
# Variáveis que devem ser verificadas dentro de cada combinação animal + data

campos_validacao = [
    "kg_leite",
    "lote",
    "data_ultimo_parto",
    "data_parto_prevista",
    "del",
    "kg_anterior",
    "estado_reprodutivo_padronizado"
]

# Quantidade de valores distintos de cada variável
# para o mesmo animal e data

conflitos_por_campo = {}

for campo in campos_validacao:

    verificacao = (
        producao_cc_debug[
            producao_cc_debug["data_prova"].notna()
        ]
        .groupby(["brinco", "data_prova"])[campo]
        .nunique(dropna=True)
    )

    conflitos_por_campo[campo] = (verificacao > 1).sum()

resumo_conflitos_final = (
    pd.Series(
        conflitos_por_campo,
        name="grupos_com_conflito"
    )
    .sort_values(ascending=False)
)

display(resumo_conflitos_final)

### 10.5 Resultado da reavaliação dos conflitos

A reavaliação utilizando `brinco + data_prova` como referência identificou divergências em apenas três variáveis.

Foram encontrados:

- 17 grupos com diferentes valores de `lote`;
- 3 grupos com diferentes valores de `kg_leite`;
- 3 grupos com diferentes valores de `estado_reprodutivo_padronizado`.

Não foram identificadas divergências em `data_ultimo_parto`, `data_parto_prevista`, `DEL` ou `kg_anterior`.

Os conflitos de produção e de estado reprodutivo já foram identificados e mantidos como pendências de validação. Antes da consolidação definitiva, serão analisadas as divergências existentes no campo `lote` para verificar se representam diferenças de nomenclatura ou mudanças semanticamente relevantes na classificação dos animais.

In [ ]:
# Identificar os grupos animal + data com mais de um lote registrado

grupos_conflito_lote = (
    producao_cc_debug[
        producao_cc_debug["data_prova"].notna()
    ]
    .groupby(["brinco", "data_prova"])["lote"]
    .nunique(dropna=True)
)

grupos_conflito_lote = grupos_conflito_lote[
    grupos_conflito_lote > 1
].reset_index()[["brinco", "data_prova"]]

# Recuperar os registros correspondentes
conflitos_lote_detalhados = producao_cc_debug.merge(
    grupos_conflito_lote,
    on=["brinco", "data_prova"],
    how="inner"
)

# Resumir os diferentes nomes de lote encontrados em cada grupo
resumo_conflitos_lote = (
    conflitos_lote_detalhados
    .groupby(["brinco", "data_prova"])["lote"]
    .agg(
        lambda x: sorted(
            set(
                str(v).strip()
                for v in x
                if pd.notna(v)
            )
        )
    )
    .reset_index(name="lotes_encontrados")
)

display(resumo_conflitos_lote)

### 10.6 Análise das divergências de lote

Os 17 conflitos identificados no campo `lote` estão concentrados exclusivamente no controle leiteiro de 27/05/2026.

Foram observados apenas dois padrões de divergência:

- `COMPOST` e `COMPOST PP`;
- `COMPOST PRIMÍPARAS` e `COMPOST AA PRIMÍPARAS`.

O padrão indica que as divergências estão relacionadas à nomenclatura ou subdivisão dos lotes utilizada nas diferentes regiões ou versões do controle leiteiro, e não a alterações aleatórias entre datas de controle.

Entretanto, as siglas adicionais podem representar informações específicas de manejo. Por esse motivo, elas não serão eliminadas ou substituídas automaticamente nesta etapa.

Os valores originais serão preservados para garantir rastreabilidade. Caso o campo `lote` seja posteriormente selecionado como variável para a modelagem, será criada uma versão padronizada de acordo com o nível de agrupamento adequado à análise.

In [ ]:
# Comparar as pendências de produção com as pendências de estado reprodutivo

chaves_conflito_producao = set(
    conflitos_producao[
        ["brinco", "data_prova"]
    ].itertuples(index=False, name=None)
)

# Identificar animal + data com mais de um estado reprodutivo válido
conflitos_estado_final = (
    producao_cc_debug[
        producao_cc_debug["data_prova"].notna()
    ]
    .groupby(["brinco", "data_prova"])[
        "estado_reprodutivo_padronizado"
    ]
    .nunique(dropna=True)
)

conflitos_estado_final = (
    conflitos_estado_final[
        conflitos_estado_final > 1
    ]
    .reset_index()[["brinco", "data_prova"]]
)

chaves_conflito_estado = set(
    conflitos_estado_final.itertuples(index=False, name=None)
)

print("Conflitos de produção:")
for item in sorted(chaves_conflito_producao):
    print(" -", item)

print("\nConflitos de estado reprodutivo:")
for item in sorted(chaves_conflito_estado):
    print(" -", item)

print("\nPresentes nos dois tipos de conflito:")
for item in sorted(
    chaves_conflito_producao & chaves_conflito_estado
):
    print(" -", item)

print(
    "\nTotal de combinações animal + data com algum desses conflitos:",
    len(chaves_conflito_producao | chaves_conflito_estado)
)

### 10.7 Classificação das observações com conflitos relevantes

A comparação entre os conflitos de produção e de estado reprodutivo mostrou que eles ocorrem em observações diferentes.

Foram identificadas:

- 3 combinações animal-data com mais de um valor de produção de leite;
- 3 combinações animal-data com estados reprodutivos divergentes;
- nenhuma combinação apresenta simultaneamente os dois tipos de conflito.

Dessa forma, entre as 733 combinações animal-data identificadas, 6 apresentam algum desses conflitos relevantes e 727 não apresentam divergência em produção ou estado reprodutivo.

As divergências de lote foram analisadas separadamente, pois estão concentradas no controle de maio de 2026 e correspondem a diferenças de nomenclatura ou subdivisão de manejo. Essas divergências não serão utilizadas, isoladamente, como critério para exclusão de uma observação.

As seis observações conflitantes permanecerão identificadas durante a construção da base consolidada, evitando a seleção arbitrária de valores.

In [ ]:
# ============================================================
# CONSTRUÇÃO DA BASE CONSOLIDADA DE CERQUEIRA CÉSAR
# ============================================================

def valor_unico_ou_ausente(serie):
    """
    Retorna o único valor não nulo existente no grupo.
    Se houver mais de um valor diferente, retorna NaN.
    """
    valores = pd.Series(serie).dropna().unique()

    if len(valores) == 1:
        return valores[0]

    return np.nan


# Uma linha por animal + data
producao_cc_consolidada = (
    producao_cc_debug[
        producao_cc_debug["data_prova"].notna()
    ]
    .groupby(
        ["brinco", "data_prova"],
        as_index=False
    )
    .agg({
        "kg_leite": valor_unico_ou_ausente,
        "data_ultimo_parto": valor_unico_ou_ausente,
        "data_parto_prevista": valor_unico_ou_ausente,
        "del": valor_unico_ou_ausente,
        "kg_anterior": valor_unico_ou_ausente,
        "estado_reprodutivo_padronizado": valor_unico_ou_ausente
    })
)

# Flags de conflito
producao_cc_consolidada["conflito_kg_leite"] = (
    producao_cc_consolidada.apply(
        lambda row: (
            row["brinco"],
            row["data_prova"]
        ) in chaves_conflito_producao,
        axis=1
    )
)

producao_cc_consolidada["conflito_estado_reprodutivo"] = (
    producao_cc_consolidada.apply(
        lambda row: (
            row["brinco"],
            row["data_prova"]
        ) in chaves_conflito_estado,
        axis=1
    )
)

producao_cc_consolidada["possui_conflito"] = (
    producao_cc_consolidada["conflito_kg_leite"]
    |
    producao_cc_consolidada["conflito_estado_reprodutivo"]
)

print("Total de observações consolidadas:", len(producao_cc_consolidada))
print(
    "Sem conflitos relevantes:",
    (~producao_cc_consolidada["possui_conflito"]).sum()
)
print(
    "Com conflitos relevantes:",
    producao_cc_consolidada["possui_conflito"].sum()
)

display(producao_cc_consolidada.head())

### 10.8 Validação da base consolidada de Cerqueira César

A consolidação resultou em 733 observações, correspondentes às combinações únicas entre animal e data de controle identificadas anteriormente.

Dessas observações:

- 727 não apresentam conflitos relevantes de produção ou estado reprodutivo;
- 3 apresentam conflito no valor de produção de leite;
- 3 apresentam conflito no estado reprodutivo;
- nenhuma observação apresenta simultaneamente os dois tipos de conflito.

Nos campos conflitantes, nenhum valor foi selecionado arbitrariamente. Quando mais de um valor distinto foi encontrado para a mesma observação, o campo correspondente foi mantido como ausente e o conflito foi registrado por meio de uma variável indicadora.

Antes de utilizar essa base nas etapas seguintes, serão realizados testes de integridade para confirmar a unicidade da combinação animal-data, a correta sinalização dos conflitos e a ausência de perdas inesperadas durante a consolidação.

In [ ]:
# ============================================================
# TESTES DE INTEGRIDADE DA BASE CONSOLIDADA
# ============================================================

# 1. Verificar duplicidade de animal + data
duplicados_consolidada = (
    producao_cc_consolidada
    .duplicated(subset=["brinco", "data_prova"])
    .sum()
)

# 2. Quantidade de conflitos por tipo
n_conflito_kg = (
    producao_cc_consolidada["conflito_kg_leite"].sum()
)

n_conflito_estado = (
    producao_cc_consolidada[
        "conflito_estado_reprodutivo"
    ].sum()
)

n_conflito_total = (
    producao_cc_consolidada["possui_conflito"].sum()
)

# 3. Conferir se produção ficou ausente exatamente
# nos três conflitos de kg_leite
kg_ausente_em_conflito = (
    producao_cc_consolidada.loc[
        producao_cc_consolidada["conflito_kg_leite"],
        "kg_leite"
    ]
    .isna()
    .sum()
)

# 4. Conferir estado reprodutivo nos conflitos
estado_ausente_em_conflito = (
    producao_cc_consolidada.loc[
        producao_cc_consolidada[
            "conflito_estado_reprodutivo"
        ],
        "estado_reprodutivo_padronizado"
    ]
    .isna()
    .sum()
)

print("VALIDAÇÃO DA BASE CONSOLIDADA")
print("-" * 45)

print(f"Total de observações: {len(producao_cc_consolidada)}")
print(f"Duplicidades animal + data: {duplicados_consolidada}")

print(f"\nConflitos de kg_leite: {n_conflito_kg}")
print(f"Conflitos de estado reprodutivo: {n_conflito_estado}")
print(f"Observações com algum conflito: {n_conflito_total}")

print(
    f"\nConflitos de kg_leite mantidos como ausentes: "
    f"{kg_ausente_em_conflito}"
)

print(
    f"Conflitos de estado mantidos como ausentes: "
    f"{estado_ausente_em_conflito}"
)

# Resultado geral
validacao_ok = (
    len(producao_cc_consolidada) == 733
    and duplicados_consolidada == 0
    and n_conflito_kg == 3
    and n_conflito_estado == 3
    and n_conflito_total == 6
    and kg_ausente_em_conflito == 3
    and estado_ausente_em_conflito == 3
)

print("\nBase passou nos testes:", validacao_ok)

## 11. Integração dos controles leiteiros com o cadastro dos animais de Cerqueira César

Após a consolidação e validação dos controles leiteiros, a próxima etapa consiste em relacionar cada observação de produção às características cadastrais do respectivo animal.

A tabela consolidada possui uma observação por combinação entre animal e data de controle, enquanto a base cadastral contém informações individuais como identificação, raça e data de nascimento.

Essa integração permitirá acrescentar características dos animais às observações produtivas, possibilitando posteriormente a construção de variáveis potencialmente relevantes para os modelos de Machine Learning.

Antes da junção das tabelas, será avaliada a compatibilidade entre os identificadores presentes nas duas fontes. Essa verificação é necessária porque os controles leiteiros utilizam o campo `brinco`, que pode combinar nome e número do animal, enquanto a base cadastral possui campos separados para identificação e código do animal.

Nenhuma associação será realizada automaticamente antes da validação dos identificadores.

### 11.1 Correção da estrutura da tabela cadastral

Durante a preparação para integração entre os controles leiteiros e o cadastro dos animais, foi identificada uma inconsistência na estrutura da tabela cadastral extraída.

A tabela apresentava 93 linhas e colunas sem identificação adequada, enquanto a primeira linha continha os nomes reais das variáveis (`id_animal`, `identificação`, `raça`, `data_nascimento` e `ativo`).

Isso indica que, durante a transformação do bloco original, a linha de cabeçalho foi mantida como registro de dados.

Antes da integração, a tabela será corrigida utilizando essa primeira linha como cabeçalho e removendo colunas completamente vazias. Essa correção não modifica os valores cadastrais dos animais, apenas reorganiza a estrutura tabular.

In [ ]:
print("Tipo da variável animal:", type(animal))
print("Conteúdo:", animal)

In [ ]:
# Criar uma cópia para preservar a tabela extraída originalmente
animal_corrigido = animal.copy()

# Utilizar a primeira linha como cabeçalho
animal_corrigido.columns = animal_corrigido.iloc[0]

# Remover a linha que passou a ser o cabeçalho
animal_corrigido = animal_corrigido.iloc[1:].copy()

# Remover colunas completamente vazias
animal_corrigido = animal_corrigido.dropna(axis=1, how="all")

# Remover o nome residual do índice das colunas
animal_corrigido.columns.name = None

# Reiniciar o índice
animal_corrigido = animal_corrigido.reset_index(drop=True)

print("Formato após a correção:", animal_corrigido.shape)

print("\nColunas:")
print(animal_corrigido.columns.tolist())

print("\nPrimeiras linhas:")
display(animal_corrigido.head(10))

In [ ]:
# ============================================================
# ANÁLISE DOS IDENTIFICADORES DOS ANIMAIS
# ============================================================

# Animais distintos presentes nos controles leiteiros
animais_producao = (
    producao_cc_consolidada["brinco"]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .sort_values()
)

print("Animais distintos nos controles leiteiros:")
print(len(animais_producao))

print("\nExemplos:")
display(animais_producao.head(20))


# Cadastro estruturado dos animais
print("\n" + "=" * 60)
print("CADASTRO DE ANIMAIS")
print("=" * 60)

print("Animais cadastrados:", len(animal_corrigido))

display(
    animal_corrigido[
        [
            "id_animal",
            "identificação",
            "raça",
            "data_nascimento",
            "ativo"
        ]
    ].head(20)
)


In [ ]:
print("Formato da tabela animal:", animal.shape)

print("\nColunas encontradas:")
print(animal.columns.tolist())

print("\nPrimeiras linhas:")
display(animal.head(10))

### 11.2 Verificação das demais tabelas cadastrais

Como a tabela de animais apresentou deslocamento do cabeçalho após a extração, as demais tabelas obtidas pela mesma função serão verificadas antes de qualquer integração.

Serão avaliadas as estruturas de parto, lactação, pesagem, doença e diagnóstico. O objetivo é identificar se o mesmo comportamento ocorreu nesses conjuntos de dados e garantir que as tabelas utilizadas posteriormente possuam cabeçalhos e registros corretamente estruturados.

Nesta etapa nenhuma informação será alterada.

In [ ]:
# Verificar a estrutura das demais tabelas extraídas

tabelas_verificacao = {
    "parto": parto,
    "lactacao": lactacao,
    "pesagem": pesagem,
    "doenca": doenca,
    "diagnostico": diagnostico
}

for nome, tabela in tabelas_verificacao.items():

    print("\n" + "=" * 70)
    print(nome.upper())
    print("=" * 70)

    print("Formato:", tabela.shape)
    print("Colunas:", tabela.columns.tolist())

    print("\nPrimeiras 3 linhas:")
    display(tabela.head(3))

### 11.3 Padronização dos identificadores dos animais

Os controles leiteiros apresentam 151 animais distintos, enquanto a tabela cadastral contém 92 animais.

A inspeção dos identificadores mostrou que o campo `brinco` dos controles geralmente combina o nome do animal com seu número de identificação, enquanto a tabela cadastral armazena esse número separadamente no campo `id_animal`.

Também foram observadas diferenças de formatação, como o uso de pontos e hífens nos identificadores. Por exemplo, `EUNICE-1.047` corresponde ao identificador numérico `1047` no cadastro.

Para realizar uma associação mais confiável entre as fontes, será extraído e padronizado o componente numérico presente no campo `brinco`. Antes de qualquer junção entre as tabelas, será avaliada a taxa de correspondência entre esse identificador e o `id_animal` cadastral.

Os dados originais serão preservados durante esse processo.

In [ ]:
import re

def extrair_id_brinco(valor):
    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip()

    # Procura o último grupo numérico do identificador,
    # permitindo ponto como separador visual: 1.047 -> 1047
    encontrados = re.findall(r"\d+(?:\.\d+)*", texto)

    if not encontrados:
        return np.nan

    numero = encontrados[-1].replace(".", "")

    try:
        return int(numero)
    except ValueError:
        return np.nan


# Tabela apenas com os animais distintos dos controles
identificadores_controle = pd.DataFrame({
    "brinco": animais_producao
}).reset_index(drop=True)

identificadores_controle["id_extraido"] = (
    identificadores_controle["brinco"]
    .apply(extrair_id_brinco)
)

display(identificadores_controle.head(30))

### 11.4 Validação da correspondência entre os identificadores

A extração do componente numérico do campo `brinco` mostrou-se adequada nos registros inspecionados, inclusive nos identificadores que utilizavam pontos como separadores visuais.

A próxima etapa consiste em comparar os identificadores extraídos dos controles leiteiros com o campo `id_animal` da tabela cadastral.

Essa comparação será realizada antes da integração das tabelas para quantificar:

- quantos animais dos controles possuem correspondência no cadastro;
- quantos não possuem correspondência;
- quantas observações de produção pertencem a animais identificados no cadastro;
- quais identificadores exigem investigação adicional.

A ausência de correspondência não será tratada automaticamente como erro nem resultará em exclusão nesta etapa.

In [ ]:
# ============================================================
# VALIDAÇÃO DOS IDENTIFICADORES COM O CADASTRO
# ============================================================

# Garantir que o identificador cadastral seja numérico
animal_corrigido["id_animal"] = pd.to_numeric(
    animal_corrigido["id_animal"],
    errors="coerce"
).astype("Int64")

identificadores_controle["id_extraido"] = pd.to_numeric(
    identificadores_controle["id_extraido"],
    errors="coerce"
).astype("Int64")


# Conjunto dos IDs existentes no cadastro
ids_cadastrados = set(
    animal_corrigido["id_animal"].dropna()
)


# Verificar correspondência
identificadores_controle["encontrado_cadastro"] = (
    identificadores_controle["id_extraido"]
    .isin(ids_cadastrados)
)


# Resumo por animal
total_controle = len(identificadores_controle)

encontrados = (
    identificadores_controle["encontrado_cadastro"].sum()
)

nao_encontrados = total_controle - encontrados


print("VALIDAÇÃO DOS IDENTIFICADORES")
print("-" * 45)

print("Animais distintos nos controles:", total_controle)
print("Encontrados no cadastro:", encontrados)
print("Não encontrados no cadastro:", nao_encontrados)

print(
    "Percentual encontrado:",
    f"{(encontrados / total_controle) * 100:.2f}%"
)


print("\nANIMAIS NÃO ENCONTRADOS NO CADASTRO:")

display(
    identificadores_controle.loc[
        ~identificadores_controle["encontrado_cadastro"],
        ["brinco", "id_extraido"]
    ].reset_index(drop=True)
)

### 11.5 Avaliação da cobertura do cadastro nas observações produtivas

A comparação dos identificadores mostrou que 90 dos 151 animais distintos presentes nos controles leiteiros possuem correspondência direta com a tabela cadastral, representando 59,60% dos animais observados.

Os 61 animais restantes não foram automaticamente descartados, pois a ausência no cadastro fornecido não implica necessariamente erro nos controles leiteiros.

Como cada animal pode possuir diferentes quantidades de controles ao longo do período analisado, a proporção de animais identificados não representa diretamente a cobertura das observações produtivas.

Por esse motivo, será avaliada também a quantidade das 733 observações consolidadas que pertence aos animais encontrados no cadastro. Essa análise permitirá determinar a cobertura efetiva das informações cadastrais sobre os registros de produção.

In [ ]:
# ============================================================
# COBERTURA DO CADASTRO NAS OBSERVAÇÕES DE PRODUÇÃO
# ============================================================

# Extrair o ID do animal em cada observação consolidada
producao_cc_consolidada["id_animal_extraido"] = (
    producao_cc_consolidada["brinco"]
    .apply(extrair_id_brinco)
)

producao_cc_consolidada["id_animal_extraido"] = pd.to_numeric(
    producao_cc_consolidada["id_animal_extraido"],
    errors="coerce"
).astype("Int64")


# Verificar se cada observação possui animal cadastrado
producao_cc_consolidada["animal_encontrado_cadastro"] = (
    producao_cc_consolidada["id_animal_extraido"]
    .isin(ids_cadastrados)
)


# Contagens
total_observacoes = len(producao_cc_consolidada)

observacoes_com_cadastro = (
    producao_cc_consolidada["animal_encontrado_cadastro"].sum()
)

observacoes_sem_cadastro = (
    total_observacoes - observacoes_com_cadastro
)

percentual_observacoes_cobertas = (
    observacoes_com_cadastro / total_observacoes * 100
)


print("COBERTURA DO CADASTRO NAS OBSERVAÇÕES")
print("-" * 50)

print("Total de observações:", total_observacoes)
print("Observações com animal cadastrado:", observacoes_com_cadastro)
print("Observações sem animal cadastrado:", observacoes_sem_cadastro)

print(
    "Percentual das observações cobertas:",
    f"{percentual_observacoes_cobertas:.2f}%"
)

### 11.6 Verificação dos animais cadastrados sem registros nos controles

Das 733 observações produtivas consolidadas, 472 (64,39%) pertencem a animais encontrados na tabela cadastral e 261 (35,61%) pertencem a animais sem correspondência cadastral.

As observações sem correspondência serão preservadas, uma vez que continuam contendo informações produtivas válidas. Nesses casos, as características provenientes do cadastro permanecerão ausentes após a integração.

Antes da junção das tabelas, será realizada também a verificação inversa: identificar quais animais presentes no cadastro não aparecem nos controles leiteiros analisados.

Essa etapa permite caracterizar completamente a relação entre as duas fontes antes da integração.

In [ ]:
# ============================================================
# ANIMAIS CADASTRADOS QUE NÃO APARECEM NOS CONTROLES
# ============================================================

ids_presentes_controles = set(
    identificadores_controle["id_extraido"].dropna()
)

animais_cadastro_sem_controle = (
    animal_corrigido.loc[
        ~animal_corrigido["id_animal"].isin(ids_presentes_controles)
    ]
    .copy()
)

print(
    "Animais cadastrados sem registro nos controles:",
    len(animais_cadastro_sem_controle)
)

display(animais_cadastro_sem_controle)

### 11.7 Integração entre produção e cadastro dos animais

Após a validação dos identificadores, foi possível estabelecer a correspondência entre os controles leiteiros e a tabela cadastral por meio do identificador numérico dos animais.

Dos 92 animais presentes no cadastro, 90 possuem registros nos controles leiteiros analisados. Os animais Gameta (ID 243) e Lontra (ID 1301) não apresentam observações produtivas nesse período.

Considerando as 733 observações consolidadas de produção, 472 (64,39%) pertencem a animais encontrados no cadastro, enquanto 261 (35,61%) pertencem a animais sem correspondência cadastral.

A integração será realizada utilizando uma junção à esquerda (`left join`), mantendo a base de produção como referência. Dessa forma, todas as observações produtivas serão preservadas, inclusive aquelas pertencentes a animais sem informações cadastrais.

Para os animais com correspondência serão incorporadas as variáveis de identificação, raça, data de nascimento e situação cadastral. Nos demais registros, essas informações permanecerão ausentes.

In [ ]:
# ============================================================
# INTEGRAÇÃO DA PRODUÇÃO COM O CADASTRO DOS ANIMAIS
# ============================================================

# Selecionar as informações cadastrais necessárias
cadastro_para_merge = animal_corrigido[
    [
        "id_animal",
        "identificação",
        "raça",
        "data_nascimento",
        "ativo"
    ]
].copy()


# Integrar utilizando o identificador numérico
producao_cc_integrada = producao_cc_consolidada.merge(
    cadastro_para_merge,
    how="left",
    left_on="id_animal_extraido",
    right_on="id_animal",
    validate="many_to_one"
)


# ============================================================
# VALIDAÇÃO DA INTEGRAÇÃO
# ============================================================

print("VALIDAÇÃO DA INTEGRAÇÃO")
print("-" * 50)

print(
    "Observações antes do merge:",
    len(producao_cc_consolidada)
)

print(
    "Observações depois do merge:",
    len(producao_cc_integrada)
)

print(
    "Animais com identificação cadastral:",
    producao_cc_integrada["identificação"].notna().sum()
)

print(
    "Animais sem identificação cadastral:",
    producao_cc_integrada["identificação"].isna().sum()
)

print(
    "Duplicidades animal + data:",
    producao_cc_integrada
    .duplicated(subset=["brinco", "data_prova"])
    .sum()
)


# Ver algumas observações integradas
display(
    producao_cc_integrada[
        [
            "brinco",
            "id_animal_extraido",
            "identificação",
            "raça",
            "data_nascimento",
            "data_prova",
            "kg_leite",
            "del",
            "estado_reprodutivo_padronizado"
        ]
    ].head(15)
)

### 11.8 Avaliação das demais informações disponíveis em Cerqueira César

A integração entre os controles leiteiros e o cadastro foi concluída preservando as 733 observações produtivas originais. Destas, 472 puderam ser associadas às informações cadastrais dos animais e 261 permaneceram sem correspondência cadastral.

Antes de incorporar outras fontes de informação à base analítica, é necessário avaliar sua disponibilidade, qualidade e relação temporal com cada controle leiteiro.

As tabelas de parto, lactação, pesagem, doença e diagnóstico serão analisadas individualmente. A inclusão de uma variável na base destinada ao Machine Learning dependerá de sua utilidade para o problema, disponibilidade de dados e possibilidade de associação ao animal e à data da observação.

Essa avaliação também é necessária para evitar o uso de informações futuras em relação ao momento do controle leiteiro, o que poderia introduzir vazamento de dados durante o treinamento dos modelos.

In [ ]:
# ============================================================
# RESUMO COMPACTO DAS FONTES AUXILIARES
# ============================================================

tabelas_auxiliares = {
    "PARTO": parto,
    "LACTACAO": lactacao,
    "PESAGEM": pesagem,
    "DOENCA": doenca,
    "DIAGNOSTICO": diagnostico
}

for nome, tabela in tabelas_auxiliares.items():

    print("\n" + "=" * 60)
    print(f"{nome} — {tabela.shape[0]} linhas × {tabela.shape[1]} colunas")
    print("=" * 60)

    resumo = pd.DataFrame({
        "preenchidos": tabela.notna().sum(),
        "ausentes": tabela.isna().sum(),
        "percentual_preenchido": (
            tabela.notna().mean() * 100
        ).round(2)
    })

    # Não exibir colunas cujo próprio nome é NaN
    resumo = resumo[resumo.index.notna()]

    display(resumo)

### 11.9 Seleção preliminar das fontes auxiliares de Cerqueira César

A análise de disponibilidade mostrou diferenças importantes entre as fontes auxiliares.

As tabelas de parto e lactação apresentam elevada completude nas principais variáveis e serão avaliadas para incorporação à base analítica. Informações como número da lactação, data de início da lactação e características do parto podem contribuir para caracterizar o estágio produtivo do animal.

A tabela de pesagem não apresentou registros de peso utilizáveis na fonte disponibilizada e, portanto, não será incorporada à base de Cerqueira César neste momento.

As informações de doença e diagnóstico possuem menor cobertura, com registros preenchidos para aproximadamente 25% dos animais. Entretanto, essas informações serão preservadas para avaliação posterior, pois eventos sanitários podem estar relacionados à produção leiteira.

Variáveis que resumem a produção acumulada da lactação, como `prod_305d ou menos` e `prod_365d ou menos`, não serão utilizadas como variáveis preditoras nesta etapa, pois podem incorporar informações posteriores à data de um determinado controle leiteiro e introduzir vazamento de informação no processo de modelagem.

Antes da integração das informações de lactação, será verificada a consistência das chaves e datas utilizadas nessa tabela.

In [ ]:
# ============================================================
# VERIFICAÇÃO DA ESTRUTURA DA TABELA DE LACTAÇÃO
# ============================================================

colunas_verificacao = [
    "id_lactação",
    "id_animal",
    "id_parto",
    "numero_lactação",
    "data_inicio",
    "data_fim",
    "status",
    "dias_lactação"
]

display(
    lactacao[colunas_verificacao].head(20)
)

print("\nTipos dos valores encontrados em id_parto:")
print(
    lactacao["id_parto"]
    .apply(lambda x: type(x).__name__)
    .value_counts(dropna=False)
)

print("\nExemplos de id_parto:")
print(
    lactacao["id_parto"]
    .head(20)
    .tolist()
)

### 11.10 Verificação da variável `id_parto` na tabela de lactação

A inspeção da tabela de lactação mostrou que a variável denominada `id_parto` não contém identificadores numéricos. Todos os 92 registros armazenados nesse campo são valores do tipo data.

Como a tabela também possui a variável `data_inicio`, será verificado se os dois campos representam a mesma informação.

Essa análise é necessária antes da integração, pois uma variável com nomenclatura incompatível com seu conteúdo não deve ser utilizada como chave de relacionamento sem validação.

In [ ]:
# ============================================================
# COMPARAÇÃO ENTRE id_parto E data_inicio
# ============================================================

id_parto_data = pd.to_datetime(
    lactacao["id_parto"],
    errors="coerce"
)

data_inicio = pd.to_datetime(
    lactacao["data_inicio"],
    errors="coerce"
)

comparacao_datas = pd.DataFrame({
    "id_animal": lactacao["id_animal"],
    "id_parto": id_parto_data,
    "data_inicio": data_inicio
})

comparacao_datas["datas_iguais"] = (
    comparacao_datas["id_parto"]
    == comparacao_datas["data_inicio"]
)

print("COMPARAÇÃO id_parto × data_inicio")
print("-" * 45)

print("Total de registros:", len(comparacao_datas))
print(
    "Datas iguais:",
    comparacao_datas["datas_iguais"].sum()
)
print(
    "Datas diferentes:",
    (~comparacao_datas["datas_iguais"]).sum()
)

print("\nRegistros com datas diferentes:")

display(
    comparacao_datas.loc[
        ~comparacao_datas["datas_iguais"]
    ]
)

### 11.11 Associação temporal entre controles leiteiros e lactações

A comparação entre `id_parto` e `data_inicio` demonstrou igualdade em todos os 92 registros da tabela de lactação. Dessa forma, o campo denominado `id_parto` não será utilizado como identificador ou chave de relacionamento, pois nesta fonte ele representa uma duplicação da data de início da lactação.

Para relacionar as informações de lactação aos controles leiteiros, será utilizada uma abordagem temporal baseada no identificador do animal e no período da lactação.

Uma observação produtiva será considerada pertencente a uma lactação quando:

- o identificador do animal for correspondente;
- a data do controle for igual ou posterior à data de início da lactação;
- e a data do controle for igual ou anterior à data de término, quando esta estiver disponível.

Para lactações ainda em andamento, representadas por ausência de `data_fim`, será considerada válida a associação a partir da respectiva `data_inicio`.

Antes de incorporar essas informações à base analítica, será avaliada a quantidade de observações que pode ser associada de forma válida a uma lactação.

In [ ]:
# ============================================================
# TESTE DE ASSOCIAÇÃO TEMPORAL COM AS LACTAÇÕES
# ============================================================

# Preparar apenas as variáveis necessárias
lactacao_temporal = lactacao[
    [
        "id_animal",
        "numero_lactação",
        "data_inicio",
        "data_fim",
        "status"
    ]
].copy()

# Padronizar tipos
lactacao_temporal["id_animal"] = pd.to_numeric(
    lactacao_temporal["id_animal"],
    errors="coerce"
).astype("Int64")

lactacao_temporal["data_inicio"] = pd.to_datetime(
    lactacao_temporal["data_inicio"],
    errors="coerce"
)

lactacao_temporal["data_fim"] = pd.to_datetime(
    lactacao_temporal["data_fim"],
    errors="coerce"
)

# Base temporária para testar a associação
teste_lactacao = producao_cc_integrada.merge(
    lactacao_temporal,
    how="left",
    left_on="id_animal_extraido",
    right_on="id_animal",
    suffixes=("", "_lactacao")
)

# Verificar se a data do controle está dentro do período da lactação
teste_lactacao["controle_dentro_lactacao"] = (
    (teste_lactacao["data_prova"] >= teste_lactacao["data_inicio"])
    &
    (
        teste_lactacao["data_fim"].isna()
        |
        (teste_lactacao["data_prova"] <= teste_lactacao["data_fim"])
    )
)

# Resumo
total = len(producao_cc_integrada)

associadas = (
    teste_lactacao
    .groupby(teste_lactacao.index)["controle_dentro_lactacao"]
    .any()
    .sum()
)

print("ASSOCIAÇÃO TEMPORAL COM LACTAÇÕES")
print("-" * 50)

print("Total de observações produtivas:", total)
print("Observações associadas a uma lactação:", associadas)
print("Observações ainda não associadas:", total - associadas)

print(
    "Percentual associado:",
    f"{(associadas / total) * 100:.2f}%"
)

### 11.12 Investigação das observações sem associação temporal à lactação

A associação temporal identificou uma lactação correspondente para 260 das 733 observações produtivas (35,47%).

Como 472 observações pertencem a animais presentes no cadastro, existem observações de animais identificados que não puderam ser associadas ao período de lactação registrado na tabela auxiliar.

Essa ausência de associação não será interpretada automaticamente como inconsistência. A tabela de lactação pode representar apenas parte do histórico produtivo de cada animal, enquanto os controles leiteiros podem abranger diferentes lactações ao longo do período analisado.

Antes de decidir pela utilização das informações dessa tabela, serão investigadas as observações não associadas, comparando a data do controle, o DEL registrado no controle e as datas de início e término disponíveis na tabela de lactação.

In [ ]:
# ============================================================
# INVESTIGAÇÃO DAS OBSERVAÇÕES CADASTRADAS SEM LACTAÇÃO ASSOCIADA
# ============================================================

# Adicionar as informações de lactação apenas para inspeção
investigacao_lactacao = producao_cc_integrada.merge(
    lactacao_temporal,
    how="left",
    left_on="id_animal_extraido",
    right_on="id_animal",
    suffixes=("", "_lactacao")
)

# Verificar se o controle está dentro do período informado
investigacao_lactacao["controle_dentro_lactacao"] = (
    (investigacao_lactacao["data_prova"] >= investigacao_lactacao["data_inicio"])
    &
    (
        investigacao_lactacao["data_fim"].isna()
        |
        (investigacao_lactacao["data_prova"] <= investigacao_lactacao["data_fim"])
    )
)

# Selecionar somente animais que existem no cadastro,
# mas cujo controle não caiu dentro da lactação registrada
sem_lactacao_associada = investigacao_lactacao[
    investigacao_lactacao["animal_encontrado_cadastro"]
    &
    ~investigacao_lactacao["controle_dentro_lactacao"]
].copy()

print("OBSERVAÇÕES CADASTRADAS SEM LACTAÇÃO ASSOCIADA")
print("-" * 55)

print("Total:", len(sem_lactacao_associada))

print(
    "Animais distintos:",
    sem_lactacao_associada["brinco"].nunique()
)

display(
    sem_lactacao_associada[
        [
            "brinco",
            "data_prova",
            "del",
            "numero_lactação",
            "data_inicio",
            "data_fim",
            "status"
        ]
    ]
    .sort_values(["brinco", "data_prova"])
    .head(40)
)

### 11.13 Decisão sobre a utilização da tabela de lactação

A investigação das 212 observações de animais cadastrados que não apresentaram associação temporal com a tabela de lactação demonstrou que a fonte não contém o histórico completo das lactações correspondentes aos controles leiteiros analisados.

Foram identificados diversos casos em que a lactação registrada na tabela auxiliar já havia sido encerrada, enquanto o animal apresentava novos controles leiteiros em 2026. Em alguns desses registros, valores baixos de DEL indicam claramente o início de uma nova lactação não representada na tabela auxiliar.

Dessa forma, as variáveis `numero_lactação`, `data_inicio`, `data_fim` e `status` da tabela de lactação não serão incorporadas diretamente às observações produtivas, evitando a associação de informações pertencentes a lactações anteriores.

Para caracterizar o estágio produtivo de cada observação será priorizada a variável `DEL` (dias em lactação), presente diretamente nos controles leiteiros e vinculada à data de cada medição.

A tabela de lactação original será preservada como fonte auxiliar e histórica, mas não será utilizada neste momento para caracterizar diretamente as observações da base analítica.


### 11.14 Validação temporal dos dias em lactação (DEL)

Como a tabela auxiliar de lactação não representa de forma completa os períodos produtivos observados nos controles leiteiros, a variável `DEL` será utilizada como principal indicador do estágio da lactação.

Antes de sua utilização na base analítica, será avaliada sua consistência temporal.

Para cada animal, os registros serão ordenados cronologicamente e será comparada a evolução do DEL entre controles consecutivos. Em condições normais dentro da mesma lactação, espera-se que o aumento do DEL seja aproximadamente equivalente ao número de dias transcorridos entre os controles.

Reduções acentuadas do DEL podem representar o início de uma nova lactação após um novo parto e serão preservadas para investigação, em vez de serem automaticamente classificadas como erro.

In [ ]:
# ============================================================
# VALIDAÇÃO TEMPORAL DO DEL
# ============================================================

del_temporal = (
    producao_cc_integrada[
        [
            "brinco",
            "data_prova",
            "del",
            "kg_leite"
        ]
    ]
    .copy()
    .sort_values(["brinco", "data_prova"])
)

# Garantir tipo numérico
del_temporal["del"] = pd.to_numeric(
    del_temporal["del"],
    errors="coerce"
)

# Valor anterior do mesmo animal
del_temporal["del_anterior"] = (
    del_temporal
    .groupby("brinco")["del"]
    .shift(1)
)

del_temporal["data_anterior"] = (
    del_temporal
    .groupby("brinco")["data_prova"]
    .shift(1)
)

# Dias transcorridos entre controles
del_temporal["dias_entre_controles"] = (
    del_temporal["data_prova"]
    - del_temporal["data_anterior"]
).dt.days

# Variação observada no DEL
del_temporal["variacao_del"] = (
    del_temporal["del"]
    - del_temporal["del_anterior"]
)

# Diferença entre o avanço esperado e observado
del_temporal["diferenca_del_tempo"] = (
    del_temporal["variacao_del"]
    - del_temporal["dias_entre_controles"]
)

# Possível reinício de lactação
del_temporal["possivel_nova_lactacao"] = (
    (del_temporal["del_anterior"].notna())
    &
    (del_temporal["del"] < del_temporal["del_anterior"])
)

print("VALIDAÇÃO TEMPORAL DO DEL")
print("-" * 50)

print(
    "Registros com DEL preenchido:",
    del_temporal["del"].notna().sum()
)

print(
    "Registros sem DEL:",
    del_temporal["del"].isna().sum()
)

print(
    "Possíveis reinícios de lactação:",
    del_temporal["possivel_nova_lactacao"].sum()
)

print("\nPossíveis reinícios identificados:")

display(
    del_temporal[
        del_temporal["possivel_nova_lactacao"]
    ][
        [
            "brinco",
            "data_anterior",
            "data_prova",
            "del_anterior",
            "del",
            "dias_entre_controles",
            "variacao_del",
            "kg_leite"
        ]
    ]
)

#### Resultado da validação dos reinícios de DEL

Todos os 733 registros produtivos apresentaram informação de DEL preenchida.

Foram identificadas 33 situações em que o DEL de um animal diminuiu em relação ao controle anterior. As reduções observadas foram acentuadas e resultaram, em geral, em valores baixos de DEL, padrão compatível com o início de uma nova lactação.

Essas ocorrências não serão classificadas automaticamente como inconsistências. Elas serão interpretadas como possíveis transições entre lactações, especialmente porque diversos animais apresentam intervalos entre controles antes do reinício do DEL.

A seguir, será avaliada a consistência dos registros pertencentes à continuidade de uma mesma lactação, comparando a variação do DEL com o intervalo de dias entre controles consecutivos.

In [ ]:
# ============================================================
# CONSISTÊNCIA DO DEL DENTRO DA MESMA LACTAÇÃO
# ============================================================

# Considerar somente pares consecutivos com DEL anterior
# e sem redução do DEL
continuidade_del = del_temporal[
    del_temporal["del_anterior"].notna()
    & ~del_temporal["possivel_nova_lactacao"]
].copy()

# Diferença absoluta entre:
# - quanto o DEL avançou
# - quantos dias realmente passaram entre os controles
continuidade_del["erro_absoluto_del"] = (
    continuidade_del["diferenca_del_tempo"].abs()
)

print("CONSISTÊNCIA DO DEL ENTRE CONTROLES")
print("-" * 50)

print("Comparações disponíveis:", len(continuidade_del))

print(
    "Diferença exatamente igual a 0 dias:",
    (continuidade_del["erro_absoluto_del"] == 0).sum()
)

print(
    "Diferença de até 1 dia:",
    (continuidade_del["erro_absoluto_del"] <= 1).sum()
)

print(
    "Diferença de até 3 dias:",
    (continuidade_del["erro_absoluto_del"] <= 3).sum()
)

print(
    "Diferença maior que 3 dias:",
    (continuidade_del["erro_absoluto_del"] > 3).sum()
)

print("\nResumo da diferença em dias:")
display(
    continuidade_del["diferenca_del_tempo"]
    .describe()
)

print("\nCasos com diferença maior que 3 dias:")

casos_inconsistentes_del = continuidade_del[
    continuidade_del["erro_absoluto_del"] > 3
].copy()

if casos_inconsistentes_del.empty:
    print("Nenhum caso identificado.")
else:
    display(
        casos_inconsistentes_del[
            [
                "brinco",
                "data_anterior",
                "data_prova",
                "del_anterior",
                "del",
                "dias_entre_controles",
                "variacao_del",
                "diferenca_del_tempo",
                "erro_absoluto_del",
                "kg_leite"
            ]
        ].sort_values(
            "erro_absoluto_del",
            ascending=False
        )
    )

#### Resultado da validação temporal do DEL

A análise confirmou elevada consistência temporal da variável DEL nos controles leiteiros.

Entre 549 comparações consecutivas classificadas como continuidade da mesma lactação, 456 apresentaram correspondência exata entre o avanço do DEL e o número de dias transcorridos entre os controles. Todas as 549 comparações apresentaram diferença máxima de apenas um dia, não sendo identificado nenhum caso com diferença superior a três dias.

Também foram identificadas anteriormente 33 reduções acentuadas do DEL, compatíveis com possíveis reinícios de lactação.

Com base nesses resultados, o DEL presente diretamente nos controles leiteiros será mantido como variável representativa do estágio da lactação. A tabela auxiliar de lactação não será utilizada para substituir essa informação, uma vez que não representa de forma completa todas as lactações observadas no período analisado.

### 11.15 Validação das informações de parto

A tabela de parto apresenta elevada completude, com 92 registros contendo identificador do animal, data do parto, tipo de parto e número de crias.

Antes de incorporar essas informações à base analítica, será avaliada sua relação com os controles leiteiros.

Como o DEL representa o número de dias transcorridos desde o início da lactação, a combinação entre a data do controle e o DEL permite estimar aproximadamente a data de início da lactação. Essa data será comparada aos registros disponíveis na tabela de parto.

O objetivo desta análise é verificar se os partos registrados são temporalmente compatíveis com as lactações observadas nos controles e determinar de que forma essas informações podem ser utilizadas sem introduzir informações futuras no processo de modelagem.

In [ ]:
# ============================================================
# VALIDAÇÃO DOS REGISTROS DE PARTO COM O DEL
# ============================================================

parto_validacao = parto[
    [
        "id_animal",
        "data_hora_parto",
        "tipo_parto",
        "numero_crias"
    ]
].copy()

# Padronização dos tipos
parto_validacao["id_animal"] = pd.to_numeric(
    parto_validacao["id_animal"],
    errors="coerce"
).astype("Int64")

parto_validacao["data_hora_parto"] = pd.to_datetime(
    parto_validacao["data_hora_parto"],
    errors="coerce"
)

# Preparar observações produtivas
controle_parto = producao_cc_integrada[
    [
        "brinco",
        "id_animal_extraido",
        "data_prova",
        "del",
        "kg_leite"
    ]
].copy()

controle_parto["del"] = pd.to_numeric(
    controle_parto["del"],
    errors="coerce"
)

# Estimativa da data de início da lactação a partir do DEL
controle_parto["data_parto_estimada_del"] = (
    controle_parto["data_prova"]
    - pd.to_timedelta(controle_parto["del"], unit="D")
)

# Associar com os registros de parto disponíveis
comparacao_parto = controle_parto.merge(
    parto_validacao,
    how="left",
    left_on="id_animal_extraido",
    right_on="id_animal"
)

# Diferença entre a data estimada pelo DEL e a data registrada
comparacao_parto["diferenca_parto_dias"] = (
    comparacao_parto["data_parto_estimada_del"]
    - comparacao_parto["data_hora_parto"]
).dt.days

print("VALIDAÇÃO DOS REGISTROS DE PARTO")
print("-" * 50)

print(
    "Observações produtivas:",
    len(comparacao_parto)
)

print(
    "Observações com registro de parto disponível:",
    comparacao_parto["data_hora_parto"].notna().sum()
)

print(
    "Animais com registro de parto disponível:",
    comparacao_parto.loc[
        comparacao_parto["data_hora_parto"].notna(),
        "brinco"
    ].nunique()
)

print("\nResumo da diferença entre parto estimado pelo DEL e parto registrado:")

display(
    comparacao_parto.loc[
        comparacao_parto["data_hora_parto"].notna(),
        "diferenca_parto_dias"
    ].describe()
)

print("\nExemplos:")

display(
    comparacao_parto.loc[
        comparacao_parto["data_hora_parto"].notna(),
        [
            "brinco",
            "data_prova",
            "del",
            "data_parto_estimada_del",
            "data_hora_parto",
            "diferenca_parto_dias",
            "tipo_parto",
            "numero_crias",
            "kg_leite"
        ]
    ]
    .sort_values(["brinco", "data_prova"])
    .head(30)
)

### 11.16 Correspondência entre o parto registrado e a lactação observada

A comparação inicial mostrou que nem todos os controles de um mesmo animal pertencem ao parto disponível na tabela auxiliar.

Em alguns animais, os primeiros controles de 2026 pertencem a uma lactação anterior, enquanto a tabela de parto contém um evento ocorrido posteriormente no mesmo ano. Nesses casos, associar diretamente o parto ao animal atribuiria informações de uma lactação diferente à observação produtiva.

Por outro lado, quando o parto registrado corresponde à lactação observada, a data estimada a partir do DEL apresenta elevada proximidade com a data registrada.

Considerando a consistência previamente observada no DEL, será adotada uma tolerância de até um dia entre a data de parto estimada pelo DEL e a data registrada para identificar controles pertencentes ao parto disponível na tabela auxiliar.

Essa classificação será utilizada apenas para avaliar a cobertura e a possibilidade de utilização das características do parto, sem modificar ainda a base analítica definitiva.

In [ ]:
# ============================================================
# IDENTIFICAÇÃO DOS CONTROLES COMPATÍVEIS COM O PARTO REGISTRADO
# ============================================================

comparacao_parto["parto_compativel"] = (
    comparacao_parto["diferenca_parto_dias"]
    .abs()
    .le(1)
)

total_com_parto = comparacao_parto["data_hora_parto"].notna().sum()

compativeis = comparacao_parto["parto_compativel"].sum()

nao_compativeis = (
    comparacao_parto["data_hora_parto"].notna()
    & ~comparacao_parto["parto_compativel"]
).sum()

print("CORRESPONDÊNCIA ENTRE PARTO E LACTAÇÃO")
print("-" * 50)

print("Observações com algum parto disponível:", total_com_parto)
print("Observações compatíveis com o parto registrado:", compativeis)
print("Observações referentes a outra lactação:", nao_compativeis)

print(
    "Percentual compatível entre observações com parto disponível:",
    f"{(compativeis / total_com_parto) * 100:.2f}%"
)

print(
    "Animais com pelo menos uma correspondência:",
    comparacao_parto.loc[
        comparacao_parto["parto_compativel"],
        "brinco"
    ].nunique()
)

print("\nDistribuição do tipo de parto entre as correspondências válidas:")

display(
    comparacao_parto.loc[
        comparacao_parto["parto_compativel"],
        "tipo_parto"
    ].value_counts(dropna=False)
)

print("\nDistribuição do número de crias entre as correspondências válidas:")

display(
    comparacao_parto.loc[
        comparacao_parto["parto_compativel"],
        "numero_crias"
    ].value_counts(dropna=False)
)

### 11.17 Decisão preliminar sobre as informações de parto

A associação temporal entre os controles leiteiros e os registros de parto identificou 403 observações compatíveis entre as 472 observações para as quais havia algum parto disponível, correspondendo a 85,38%.

Foram identificadas 69 observações relacionadas a uma lactação diferente daquela representada pelo parto disponível na tabela auxiliar. Esses registros não foram classificados como erros, pois o histórico disponibilizado não contém necessariamente todos os partos de cada animal.

Entre as 403 observações corretamente associadas ao parto registrado, 386 estavam relacionadas a partos classificados como normais, 11 a partos com auxílio e 6 a abortos registrados aos 150 dias.

A variável `numero_crias` apresentou variabilidade muito reduzida: 400 observações estavam associadas a partos com uma cria e apenas 3 a partos com duas crias.

Dessa forma, `numero_crias` não será priorizada como variável preditora nesta etapa devido à baixa variabilidade observada. A variável `tipo_parto` será preservada como variável candidata para avaliação posterior, sem que sua inclusão definitiva no modelo seja determinada neste momento.

As características do parto somente poderão ser associadas às observações produtivas quando houver correspondência temporal entre o parto registrado e a lactação representada pelo controle, evitando a atribuição de informações pertencentes a outra lactação.

### 11.18 Incorporação temporal das características do parto

Após a validação da correspondência entre os registros de parto e as lactações observadas, será incorporada à base uma versão temporalmente validada da variável `tipo_parto`.

A informação será preenchida somente quando a data de parto registrada apresentar diferença máxima de um dia em relação à data estimada a partir do DEL. Nas demais observações, o valor permanecerá ausente, indicando que não existe um registro de parto validado para aquela lactação na fonte disponibilizada.

Essa estratégia permite preservar informações potencialmente úteis sem atribuir características de um parto a uma lactação diferente.

In [ ]:
# ============================================================
# INCORPORAÇÃO SEGURA DO TIPO DE PARTO
# ============================================================

# Criar tabela auxiliar mantendo a mesma identificação
parto_validado = comparacao_parto[
    [
        "brinco",
        "data_prova",
        "parto_compativel",
        "tipo_parto",
        "numero_crias"
    ]
].copy()

# Somente utilizar características quando o parto foi validado
parto_validado["tipo_parto_validado"] = (
    parto_validado["tipo_parto"]
    .where(parto_validado["parto_compativel"])
)

parto_validado["numero_crias_validado"] = (
    parto_validado["numero_crias"]
    .where(parto_validado["parto_compativel"])
)

# Incorporar à base integrada
producao_cc_integrada = producao_cc_integrada.merge(
    parto_validado[
        [
            "brinco",
            "data_prova",
            "tipo_parto_validado",
            "numero_crias_validado"
        ]
    ],
    how="left",
    on=["brinco", "data_prova"]
)

print("BASE APÓS INCORPORAÇÃO DAS INFORMAÇÕES DE PARTO")
print("-" * 55)

print("Observações:", len(producao_cc_integrada))

print(
    "Tipo de parto validado preenchido:",
    producao_cc_integrada["tipo_parto_validado"].notna().sum()
)

print(
    "Tipo de parto sem informação validada:",
    producao_cc_integrada["tipo_parto_validado"].isna().sum()
)

print("\nDistribuição:")
display(
    producao_cc_integrada["tipo_parto_validado"]
    .value_counts(dropna=False)
)

### 11.19 Análise dos registros sanitários

Após a incorporação temporalmente validada das informações de parto, serão avaliados os registros de doenças e diagnósticos disponíveis para os animais de Cerqueira César.

A tabela de doenças apresenta informações preenchidas para 23 registros, enquanto a tabela de diagnósticos permite relacionar esses eventos aos respectivos animais e contém informações sobre data do diagnóstico, data de cura e situação do caso.

Como uma ocorrência sanitária pode afetar a produção leiteira apenas em determinados períodos, essas informações não serão associadas ao animal de forma permanente. A relação temporal entre o evento sanitário e cada controle leiteiro será considerada antes da construção de qualquer variável destinada à modelagem.

Inicialmente, serão analisados os tipos de doenças registrados, suas categorias, a ocorrência de mastite e a disponibilidade das datas necessárias para estabelecer essa relação temporal.

In [ ]:
# ============================================================
# ANÁLISE INICIAL DOS REGISTROS SANITÁRIOS
# ============================================================

print("REGISTROS DE DOENÇAS")
print("-" * 55)

doencas_validas = doenca[
    doenca["data_doença"].notna()
].copy()

print("Eventos de doença preenchidos:", len(doencas_validas))

print("\nTipos de doença:")
display(
    doencas_validas["nome"]
    .value_counts(dropna=False)
)

print("\nCategorias:")
display(
    doencas_validas["categoria"]
    .value_counts(dropna=False)
)

print("\nClassificação de mastite:")
display(
    doencas_validas["eh_mastite"]
    .value_counts(dropna=False)
)


print("\n" + "=" * 55)
print("REGISTROS DE DIAGNÓSTICO")
print("-" * 55)

diagnosticos_validos = diagnostico[
    diagnostico["data_diagnostico"].notna()
].copy()

print("Diagnósticos preenchidos:", len(diagnosticos_validos))

print("\nStatus:")
display(
    diagnosticos_validos["status"]
    .value_counts(dropna=False)
)

print("\nPeríodo dos diagnósticos:")
print(
    "Primeiro:",
    pd.to_datetime(
        diagnosticos_validos["data_diagnostico"],
        errors="coerce"
    ).min()
)

print(
    "Último:",
    pd.to_datetime(
        diagnosticos_validos["data_diagnostico"],
        errors="coerce"
    ).max()
)

print("\nRegistros disponíveis:")

display(
    diagnosticos_validos[
        [
            "id_animal",
            "id_doença",
            "data_diagnostico",
            "data_cura",
            "status"
        ]
    ].sort_values("data_diagnostico")
)

### 11.20 Caracterização dos registros sanitários

Foram identificados 23 eventos sanitários com data registrada, distribuídos entre mastite, metrite, aborto, distúrbios metabólicos e alterações locomotoras.

A mastite foi a ocorrência mais frequente, representando 13 dos 23 eventos. Os registros sanitários abrangem o período entre junho de 2024 e julho de 2026.

Como os eventos apresentam duração e momento de ocorrência específicos, não é adequado caracterizar permanentemente um animal como doente apenas por possuir algum diagnóstico em seu histórico.

Para evitar vazamento temporal, somente eventos ocorridos até a data de cada controle leiteiro poderão ser utilizados na construção de variáveis preditoras. Inicialmente será avaliada a quantidade de controles realizados durante um episódio sanitário ou após uma ocorrência recente.

Devido ao pequeno número de eventos disponíveis, a inclusão definitiva dessas informações no modelo dependerá da cobertura obtida sobre as observações produtivas.

### 11.21 Investigação da relação entre doenças e diagnósticos

As tabelas de doenças e diagnósticos contêm informações complementares. A primeira descreve as características dos eventos sanitários, enquanto a segunda identifica os animais afetados e apresenta informações sobre diagnóstico, cura e situação clínica.

Inicialmente, foi investigada a possibilidade de utilizar a variável `id_doença` para relacionar as duas tabelas. Entretanto, verificou-se que essa variável apresenta valores repetidos, como mastite, metrite e aborto, não constituindo um identificador único de cada ocorrência.

Também foi avaliada a combinação entre tipo de doença e data, que igualmente não apresentou unicidade em todos os casos.

Diante disso, a estrutura das duas tabelas foi investigada antes da realização da integração, evitando a multiplicação ou associação incorreta dos registros sanitários.

#### Investigação da chave de associação entre doenças e diagnósticos

A tentativa inicial de integrar as tabelas utilizando `id_doença` como chave única não foi realizada, pois foram identificados valores repetidos nessa variável na tabela de doenças.

Isso demonstra que, apesar de sua denominação, `id_doença` não funciona necessariamente como identificador único de cada ocorrência sanitária. Utilizar diretamente essa variável em uma junção poderia multiplicar registros e gerar associações incorretas.

Antes da integração, será investigada a estrutura das duas tabelas para identificar quais informações permitem relacionar corretamente cada diagnóstico ao respectivo evento sanitário.

In [ ]:
# ============================================================
# INVESTIGAÇÃO DA CHAVE ENTRE DOENÇA E DIAGNÓSTICO
# ============================================================

print("ESTRUTURA DA TABELA DOENÇA")
print("-" * 55)

display(
    doencas_validas[
        [
            "id_doença",
            "data_doença",
            "nome",
            "categoria",
            "eh_mastite"
        ]
    ].sort_values(["data_doença", "id_doença"])
)

print("\nFrequência de id_doença:")
display(
    doencas_validas["id_doença"]
    .value_counts(dropna=False)
)


print("\n" + "=" * 55)
print("COMPARAÇÃO COM A TABELA DIAGNÓSTICO")
print("-" * 55)

display(
    diagnosticos_validos[
        [
            "id_animal",
            "id_doença",
            "data_diagnostico",
            "data_cura",
            "status"
        ]
    ].sort_values(["data_diagnostico", "id_doença"])
)


# ------------------------------------------------------------
# Comparar id_doença e datas entre as duas tabelas
# ------------------------------------------------------------

doenca_comparacao = doencas_validas[
    ["id_doença", "data_doença"]
].copy()

diagnostico_comparacao = diagnosticos_validos[
    ["id_doença", "data_diagnostico"]
].copy()

doenca_comparacao["data_doença"] = pd.to_datetime(
    doenca_comparacao["data_doença"],
    errors="coerce"
)

diagnostico_comparacao["data_diagnostico"] = pd.to_datetime(
    diagnostico_comparacao["data_diagnostico"],
    errors="coerce"
)

# Criar pares doença + data para verificar unicidade
pares_doenca = (
    doenca_comparacao
    .groupby(["id_doença", "data_doença"], dropna=False)
    .size()
    .reset_index(name="quantidade")
)

print("\nUnicidade da combinação id_doença + data_doença:")
print("Pares existentes:", len(pares_doenca))
print(
    "Pares repetidos:",
    (pares_doenca["quantidade"] > 1).sum()
)

print("\nPares repetidos:")
display(
    pares_doenca[
        pares_doenca["quantidade"] > 1
    ]
)

### 11.22 Identificação da relação estrutural entre doenças e diagnósticos

A investigação demonstrou que a variável denominada `id_doença` não constitui um identificador único dos eventos sanitários, pois seus valores representam predominantemente o tipo de ocorrência, como mastite, metrite e aborto.

Também foi constatado que a combinação entre tipo de doença e data não é suficiente para identificar todos os eventos de forma única. Foram encontrados dois abortos registrados na mesma data e duas ocorrências de metrite registradas na mesma data, pertencentes a animais diferentes.

Entretanto, a inspeção das tabelas indicou que os registros de doença e diagnóstico preservam os mesmos índices provenientes da estrutura original da planilha. Esses índices aparentemente estabelecem correspondência entre a descrição do evento sanitário e o respectivo animal.

Antes de utilizar essa relação, será verificado se os índices dos registros preenchidos nas duas tabelas são equivalentes e se as respectivas datas também apresentam correspondência.

In [ ]:
# ============================================================
# VALIDAÇÃO DA CORRESPONDÊNCIA PELO ÍNDICE ORIGINAL
# ============================================================

indices_doenca = set(doencas_validas.index)
indices_diagnostico = set(diagnosticos_validos.index)

indices_comuns = indices_doenca.intersection(indices_diagnostico)
somente_doenca = indices_doenca - indices_diagnostico
somente_diagnostico = indices_diagnostico - indices_doenca

print("VALIDAÇÃO DOS ÍNDICES DOS REGISTROS SANITÁRIOS")
print("-" * 55)

print("Registros de doença:", len(indices_doenca))
print("Registros de diagnóstico:", len(indices_diagnostico))
print("Índices presentes nas duas tabelas:", len(indices_comuns))
print("Índices somente em doença:", len(somente_doenca))
print("Índices somente em diagnóstico:", len(somente_diagnostico))

print("\nÍndices somente em doença:", sorted(somente_doenca))
print("Índices somente em diagnóstico:", sorted(somente_diagnostico))


# ------------------------------------------------------------
# Comparar as datas nos mesmos índices
# ------------------------------------------------------------

validacao_indices = pd.DataFrame({
    "data_doenca": pd.to_datetime(
        doencas_validas.loc[sorted(indices_comuns), "data_doença"],
        errors="coerce"
    ),
    "data_diagnostico": pd.to_datetime(
        diagnosticos_validos.loc[sorted(indices_comuns), "data_diagnostico"],
        errors="coerce"
    )
})

validacao_indices["datas_iguais"] = (
    validacao_indices["data_doenca"]
    == validacao_indices["data_diagnostico"]
)

print("\nCorrespondência das datas:")
print(
    "Datas iguais:",
    validacao_indices["datas_iguais"].sum()
)

print(
    "Datas diferentes:",
    (~validacao_indices["datas_iguais"]).sum()
)

print("\nCasos com datas diferentes:")

display(
    validacao_indices[
        ~validacao_indices["datas_iguais"]
    ]
)

### 11.23 Construção da tabela sanitária integrada

A validação demonstrou que os 23 registros preenchidos da tabela de doenças possuem correspondência exata com os 23 registros da tabela de diagnósticos por meio do índice original preservado durante a estruturação dos dados.

Todos os índices estavam presentes nas duas tabelas e, adicionalmente, as datas de doença e diagnóstico coincidiram nos 23 casos analisados.

Dessa forma, o índice original será utilizado como chave estrutural para reconstruir os registros sanitários completos. Esse índice não representa um identificador clínico do evento, mas uma referência à posição correspondente das informações no arquivo original.

A tabela sanitária integrada reunirá o animal afetado, o tipo e a categoria da doença, a classificação de mastite, as datas de diagnóstico e cura e a situação registrada.

In [ ]:
# ============================================================
# CONSTRUÇÃO DA TABELA SANITÁRIA INTEGRADA
# ============================================================

# Parte descritiva do evento
sanitario_doenca = doencas_validas[
    [
        "data_doença",
        "nome",
        "categoria",
        "eh_mastite"
    ]
].copy()

# Parte relacionada ao animal e evolução clínica
sanitario_diagnostico = diagnosticos_validos[
    [
        "id_animal",
        "id_doença",
        "data_diagnostico",
        "data_cura",
        "status"
    ]
].copy()

# Integrar utilizando o índice estrutural previamente validado
sanitario = sanitario_diagnostico.join(
    sanitario_doenca,
    how="inner"
)

# Preservar explicitamente a origem estrutural
sanitario = sanitario.reset_index(
    names="indice_origem"
)

# Padronizar tipos
sanitario["id_animal"] = pd.to_numeric(
    sanitario["id_animal"],
    errors="coerce"
).astype("Int64")

for coluna in [
    "data_doença",
    "data_diagnostico",
    "data_cura"
]:
    sanitario[coluna] = pd.to_datetime(
        sanitario[coluna],
        errors="coerce"
    )

print("TABELA SANITÁRIA INTEGRADA")
print("-" * 55)

print("Registros:", len(sanitario))
print(
    "Animais distintos:",
    sanitario["id_animal"].nunique()
)
print(
    "Eventos de mastite:",
    sanitario["eh_mastite"]
    .astype(str)
    .str.lower()
    .eq("sim")
    .sum()
)

print("\nTabela resultante:")

display(
    sanitario[
        [
            "indice_origem",
            "id_animal",
            "nome",
            "categoria",
            "eh_mastite",
            "data_diagnostico",
            "data_cura",
            "status"
        ]
    ].sort_values("data_diagnostico")
)

### 11.24 Avaliação da cobertura dos registros sanitários nos controles leiteiros

Após a construção da tabela sanitária integrada, será avaliada a relação entre os eventos sanitários registrados e as observações de produção de leite.

O objetivo desta etapa é verificar quantos controles leiteiros pertencem a animais que possuem histórico sanitário disponível e quantos desses controles apresentam proximidade temporal com algum diagnóstico.

Essa análise será realizada antes da criação de variáveis sanitárias para o modelo, pois a quantidade de eventos disponíveis é reduzida. Dessa forma, será possível avaliar se essas informações apresentam cobertura suficiente para contribuir com a modelagem.

Nesta etapa, nenhuma informação será incorporada definitivamente à base de produção.

In [ ]:
# ============================================================
# COBERTURA DOS REGISTROS SANITÁRIOS NA BASE DE PRODUÇÃO
# ============================================================

ids_sanitarios = set(
    sanitario["id_animal"]
    .dropna()
    .astype(int)
)

controles_com_historico_sanitario = (
    producao_cc_integrada["id_animal_extraido"]
    .isin(ids_sanitarios)
)

print("COBERTURA DOS REGISTROS SANITÁRIOS")
print("-" * 55)

print(
    "Total de controles leiteiros:",
    len(producao_cc_integrada)
)

print(
    "Animais com eventos sanitários registrados:",
    sanitario["id_animal"].nunique()
)

print(
    "Controles pertencentes a animais com histórico sanitário:",
    controles_com_historico_sanitario.sum()
)

print(
    "Controles sem histórico sanitário registrado:",
    (~controles_com_historico_sanitario).sum()
)

print(
    "Animais distintos da produção com histórico sanitário:",
    producao_cc_integrada.loc[
        controles_com_historico_sanitario,
        "id_animal_extraido"
    ].nunique()
)

### 11.25 Análise temporal dos eventos sanitários em relação aos controles leiteiros

A presença de histórico sanitário para um animal não significa necessariamente que o evento de doença ocorreu próximo ao controle leiteiro analisado. Por isso, é necessário avaliar a relação temporal entre a data do diagnóstico e a data de cada controle de produção.

Nesta etapa, cada controle leiteiro será comparado ao evento sanitário registrado para o mesmo animal. Será calculado o número de dias entre o diagnóstico e o controle, permitindo identificar eventos ocorridos antes da medição produtiva.

Inicialmente serão avaliadas diferentes janelas temporais após o diagnóstico (7, 15 e 30 dias). Essa comparação é exploratória e tem como objetivo verificar a cobertura dos dados antes da definição de qualquer variável sanitária para o modelo.

Somente diagnósticos ocorridos na data do controle ou anteriormente serão considerados, evitando o uso de informações futuras.

In [ ]:
# ============================================================
# RELAÇÃO TEMPORAL ENTRE DIAGNÓSTICOS E CONTROLES LEITEIROS
# ============================================================

# Selecionar somente as informações necessárias
eventos_sanitarios = sanitario[
    [
        "id_animal",
        "nome",
        "eh_mastite",
        "data_diagnostico",
        "data_cura",
        "status"
    ]
].copy()

# Criar base temporária de análise
sanitario_controles = producao_cc_integrada[
    [
        "brinco",
        "id_animal_extraido",
        "data_prova",
        "kg_leite"
    ]
].merge(
    eventos_sanitarios,
    how="inner",
    left_on="id_animal_extraido",
    right_on="id_animal",
    validate="many_to_one"
)

# Calcular distância temporal entre diagnóstico e controle
sanitario_controles["dias_desde_diagnostico"] = (
    sanitario_controles["data_prova"]
    - sanitario_controles["data_diagnostico"]
).dt.days

print("RELAÇÃO ENTRE EVENTOS SANITÁRIOS E CONTROLES")
print("-" * 60)

print("Controles de animais com histórico sanitário:", len(sanitario_controles))

print(
    "Controles anteriores ao diagnóstico:",
    (sanitario_controles["dias_desde_diagnostico"] < 0).sum()
)

print(
    "Controles no mesmo dia ou após o diagnóstico:",
    (sanitario_controles["dias_desde_diagnostico"] >= 0).sum()
)

print(
    "Controles entre 0 e 7 dias após diagnóstico:",
    sanitario_controles["dias_desde_diagnostico"].between(0, 7).sum()
)

print(
    "Controles entre 0 e 15 dias após diagnóstico:",
    sanitario_controles["dias_desde_diagnostico"].between(0, 15).sum()
)

print(
    "Controles entre 0 e 30 dias após diagnóstico:",
    sanitario_controles["dias_desde_diagnostico"].between(0, 30).sum()
)

print("\nDistribuição dos dias desde o diagnóstico:")
display(
    sanitario_controles["dias_desde_diagnostico"].describe()
)

### 11.26 Verificação de eventos sanitários ativos na data do controle

A análise anterior demonstrou baixa ocorrência de controles leiteiros realizados nos primeiros 30 dias após um diagnóstico. Entretanto, alguns eventos sanitários apresentam intervalos prolongados entre diagnóstico e cura.

Por esse motivo, a proximidade com a data do diagnóstico, isoladamente, pode não representar adequadamente a situação sanitária do animal no momento do controle.

Nesta etapa será verificado se a data de cada controle leiteiro está compreendida entre a data de diagnóstico e a data de cura registrada para o respectivo evento. Essa análise permitirá identificar controles realizados enquanto havia um evento sanitário documentadamente ativo.

Os registros sem data de cura não serão classificados automaticamente como doença ativa indefinidamente, pois não há informação suficiente para determinar até quando o evento permaneceu ativo.

In [ ]:
# ============================================================
# EVENTOS SANITÁRIOS ATIVOS NA DATA DO CONTROLE
# ============================================================

sanitario_controles["evento_ativo_no_controle"] = (
    sanitario_controles["data_diagnostico"].notna()
    & sanitario_controles["data_cura"].notna()
    & (sanitario_controles["data_prova"] >= sanitario_controles["data_diagnostico"])
    & (sanitario_controles["data_prova"] <= sanitario_controles["data_cura"])
)

casos_ativos = sanitario_controles[
    sanitario_controles["evento_ativo_no_controle"]
].copy()

print("EVENTOS SANITÁRIOS ATIVOS NA DATA DO CONTROLE")
print("-" * 60)

print(
    "Controles realizados durante evento sanitário ativo:",
    len(casos_ativos)
)

print(
    "Animais distintos:",
    casos_ativos["id_animal"].nunique()
)

print("\nTipos de evento:")
print(casos_ativos["nome"].value_counts())

print("\nCasos identificados:")

display(
    casos_ativos[
        [
            "brinco",
            "data_prova",
            "kg_leite",
            "nome",
            "eh_mastite",
            "data_diagnostico",
            "data_cura",
            "status"
        ]
    ].sort_values(["data_prova", "brinco"])
)

### 11.27 Decisão sobre a utilização das informações sanitárias

A análise dos registros sanitários demonstrou que, embora existam 23 animais com eventos de saúde registrados e 130 controles leiteiros pertencentes a esses animais, a correspondência temporal entre doença e produção é limitada.

Considerando a proximidade entre diagnóstico e controle leiteiro, apenas 8 observações ocorreram nos primeiros 30 dias após um diagnóstico. A análise dos períodos compreendidos entre diagnóstico e cura identificou somente 7 controles realizados durante um evento sanitário documentadamente ativo, correspondentes a apenas 2 animais.

Todos esses controles estavam relacionados a casos de metrite. Não foram identificados controles leiteiros realizados durante períodos documentadamente ativos dos casos de mastite registrados.

Diante da baixa cobertura temporal e da concentração das observações em apenas dois animais, as informações sanitárias não serão utilizadas, neste momento, como variáveis preditoras na construção da base destinada ao treinamento dos modelos de Machine Learning. A inclusão de indicadores de doença ativa produziria variáveis extremamente esparsas e pouco representativas do conjunto analisado.

Os registros sanitários serão preservados como informação auxiliar e como parte da caracterização e avaliação da qualidade e disponibilidade dos dados da propriedade. Essa decisão poderá ser reavaliada caso sejam disponibilizados novos registros sanitários com maior cobertura temporal.

### 11.28 Inventário das variáveis disponíveis na base integrada de Cerqueira César

Após a avaliação individual das fontes auxiliares da propriedade, será realizado um inventário da base integrada de produção.

O objetivo desta etapa é identificar quais informações estão efetivamente disponíveis para cada uma das 733 observações, bem como avaliar sua cobertura antes da definição das variáveis que poderão compor a base destinada ao Machine Learning.

A presença de uma variável nesta etapa não implica sua utilização no modelo. Aspectos como disponibilidade, qualidade, significado temporal, redundância e possibilidade de vazamento de informação ainda serão considerados antes da seleção final dos atributos.

In [ ]:
# ============================================================
# INVENTÁRIO DA BASE INTEGRADA DE CERQUEIRA CÉSAR
# ============================================================

print("DIMENSÕES DA BASE")
print("-" * 60)
print("Observações:", producao_cc_integrada.shape[0])
print("Colunas:", producao_cc_integrada.shape[1])

print("\nCOLUNAS DISPONÍVEIS")
print("-" * 60)

for i, coluna in enumerate(producao_cc_integrada.columns, start=1):
    print(f"{i:02d}. {coluna}")

# ------------------------------------------------------------
# Cobertura das variáveis
# ------------------------------------------------------------

inventario_cc = (
    producao_cc_integrada
    .notna()
    .sum()
    .rename("preenchidos")
    .to_frame()
)

inventario_cc["ausentes"] = (
    len(producao_cc_integrada)
    - inventario_cc["preenchidos"]
)

inventario_cc["cobertura_%"] = (
    inventario_cc["preenchidos"]
    / len(producao_cc_integrada)
    * 100
).round(2)

inventario_cc["tipo"] = (
    producao_cc_integrada.dtypes.astype(str)
)

inventario_cc = inventario_cc[
    ["tipo", "preenchidos", "ausentes", "cobertura_%"]
]

display(inventario_cc)

### 11.29 Validação temporal da variável `kg_anterior`

A variável `kg_anterior` apresenta cobertura completa nas 733 observações consolidadas e pode representar informação relevante sobre o histórico produtivo do animal.

Entretanto, antes de considerá-la como variável preditora, é necessário verificar empiricamente seu significado. Para isso, o valor informado como produção anterior será comparado à produção registrada para o mesmo animal no controle cronologicamente anterior disponível na base.

Essa validação é necessária para determinar se `kg_anterior` representa efetivamente uma medição produtiva anterior e, consequentemente, se sua utilização pode ser realizada sem introduzir informação futura no processo de modelagem.

In [ ]:
# ============================================================
# VALIDAÇÃO TEMPORAL DE kg_anterior
# ============================================================

validacao_kg_anterior = (
    producao_cc_integrada[
        [
            "brinco",
            "data_prova",
            "kg_leite",
            "kg_anterior"
        ]
    ]
    .sort_values(["brinco", "data_prova"])
    .copy()
)

# Produção observada no controle anterior disponível
validacao_kg_anterior["kg_leite_controle_anterior"] = (
    validacao_kg_anterior
    .groupby("brinco")["kg_leite"]
    .shift(1)
)

# Diferença entre o campo informado na planilha
# e a produção realmente observada no controle anterior
validacao_kg_anterior["diferenca"] = (
    validacao_kg_anterior["kg_anterior"]
    - validacao_kg_anterior["kg_leite_controle_anterior"]
)

# Somente casos em que temos um controle anterior disponível
comparacoes_kg_anterior = validacao_kg_anterior[
    validacao_kg_anterior["kg_leite_controle_anterior"].notna()
].copy()

print("VALIDAÇÃO DE kg_anterior")
print("-" * 55)

print(
    "Comparações possíveis:",
    len(comparacoes_kg_anterior)
)

print(
    "Correspondência exata:",
    comparacoes_kg_anterior["diferenca"].eq(0).sum()
)

print(
    "Diferença absoluta <= 0,1 kg:",
    comparacoes_kg_anterior["diferenca"].abs().le(0.1).sum()
)

print(
    "Diferença absoluta <= 1 kg:",
    comparacoes_kg_anterior["diferenca"].abs().le(1).sum()
)

print("\nDistribuição das diferenças:")

display(
    comparacoes_kg_anterior["diferenca"].describe()
)

### 11.30 Investigação das divergências de `kg_anterior`

A comparação temporal demonstrou que, em 549 das 580 observações comparáveis (94,66%), o valor de `kg_anterior` corresponde exatamente à produção registrada no controle anterior disponível para o mesmo animal.

Entretanto, foram identificadas 31 observações divergentes. Em todos esses casos, o valor informado em `kg_anterior` é inferior à produção registrada no controle anterior presente na base, com diferenças que chegam a 40 kg.

Antes de decidir pela utilização dessa variável no modelo, essas divergências serão analisadas individualmente para verificar se estão relacionadas a mudanças de lactação, intervalos entre controles, ausência de controles intermediários ou outras características dos registros originais.

In [ ]:
# ============================================================
# INVESTIGAÇÃO DAS DIVERGÊNCIAS DE kg_anterior
# ============================================================

divergencias_kg_anterior = (
    comparacoes_kg_anterior[
        comparacoes_kg_anterior["diferenca"].abs() > 0.1
    ]
    .copy()
)

# Intervalo em dias em relação ao controle anterior disponível
divergencias_kg_anterior["data_controle_anterior"] = (
    validacao_kg_anterior
    .groupby("brinco")["data_prova"]
    .shift(1)
    .loc[divergencias_kg_anterior.index]
)

divergencias_kg_anterior["intervalo_dias"] = (
    divergencias_kg_anterior["data_prova"]
    - divergencias_kg_anterior["data_controle_anterior"]
).dt.days

print("DIVERGÊNCIAS DE kg_anterior")
print("-" * 60)

print("Total de divergências:", len(divergencias_kg_anterior))

print(
    "Animais distintos:",
    divergencias_kg_anterior["brinco"].nunique()
)

print("\nIntervalo entre controles:")
display(
    divergencias_kg_anterior["intervalo_dias"].describe()
)

print("\nCasos divergentes:")

display(
    divergencias_kg_anterior[
        [
            "brinco",
            "data_controle_anterior",
            "data_prova",
            "intervalo_dias",
            "kg_leite_controle_anterior",
            "kg_anterior",
            "kg_leite",
            "diferenca"
        ]
    ].sort_values(["brinco", "data_prova"])
)

### 11.31 Investigação de `kg_anterior = 0` em relação ao estágio de lactação

As 31 divergências identificadas na validação de `kg_anterior` apresentaram um padrão comum: em todos os casos, o campo `kg_anterior` possui valor igual a zero.

Essas observações também apresentam intervalos relativamente longos em relação ao último controle disponível na base, variando de 63 a 155 dias. Dessa forma, o último controle disponível nos arquivos não pode ser assumido automaticamente como a medição imediatamente anterior utilizada pela propriedade.

Considerando que alguns desses intervalos podem compreender o encerramento de uma lactação e o início de outra, será avaliado o DEL das observações divergentes. O objetivo é verificar se os valores iguais a zero estão associados ao início de uma nova lactação ou se também ocorrem em animais com lactação já estabelecida.

In [ ]:
# ============================================================
# kg_anterior = 0 E ESTÁGIO DE LACTAÇÃO
# ============================================================

analise_zero_kg_anterior = divergencias_kg_anterior.merge(
    producao_cc_integrada[
        ["brinco", "data_prova", "del"]
    ],
    on=["brinco", "data_prova"],
    how="left",
    validate="one_to_one"
)

print("DEL NAS 31 DIVERGÊNCIAS")
print("-" * 60)

display(
    analise_zero_kg_anterior["del"].describe()
)

print("\nDistribuição por faixa de DEL:")

faixas_del = pd.cut(
    analise_zero_kg_anterior["del"],
    bins=[-1, 30, 60, 100, 200, float("inf")],
    labels=[
        "0–30 dias",
        "31–60 dias",
        "61–100 dias",
        "101–200 dias",
        "> 200 dias"
    ]
)

display(
    faixas_del.value_counts().sort_index()
)

print("\nCasos individuais:")

display(
    analise_zero_kg_anterior[
        [
            "brinco",
            "data_controle_anterior",
            "data_prova",
            "intervalo_dias",
            "del",
            "kg_leite_controle_anterior",
            "kg_anterior",
            "kg_leite"
        ]
    ].sort_values(["del", "brinco"])
)

### 11.32 Decisão sobre a variável `kg_anterior`

A validação temporal demonstrou que `kg_anterior` apresenta elevada consistência com o histórico produtivo disponível. Das 580 observações em que foi possível comparar o campo com a produção registrada no controle anterior disponível para o mesmo animal, 549 (94,66%) apresentaram correspondência exata.

As 31 divergências restantes não apresentaram diferenças aleatórias. Em todos os casos, `kg_anterior` era igual a zero e o DEL da observação estava entre 2 e 30 dias. Além disso, o intervalo em relação ao último controle disponível variou de 63 a 155 dias. Esse padrão é compatível com observações realizadas no início de uma nova lactação, enquanto o último controle anteriormente disponível pertence à lactação precedente.

Dessa forma, os valores iguais a zero não serão substituídos pela produção do último controle disponível, pois essa substituição associaria produções pertencentes a lactações distintas.

A variável `kg_anterior` será preservada como candidata à modelagem, por representar informação produtiva anterior disponível ao momento do controle e apresentar elevada consistência temporal nos registros analisados. Sua utilização definitiva será avaliada em conjunto com as demais variáveis candidatas durante a construção da base destinada ao Machine Learning.

### 11.33 Avaliação da variável `data_parto_prevista`

Embora o inventário inicial tenha indicado cobertura de 100% para `data_parto_prevista`, a inspeção dos dados mostrou que a coluna contém representações textuais utilizadas para indicar ausência de informação.

Dessa forma, a contagem de valores não nulos realizada pelo Pandas não representa necessariamente a disponibilidade real dessa variável.

Antes de qualquer tratamento ou decisão sobre sua utilização, serão avaliados os tipos de valores armazenados, a frequência das diferentes representações e a quantidade de datas efetivamente válidas. Essa análise permitirá determinar a cobertura semântica da variável e identificar possíveis inconsistências de preenchimento.

In [ ]:
# ============================================================
# AVALIAÇÃO DE data_parto_prevista
# ============================================================

print("AVALIAÇÃO DE data_parto_prevista")
print("-" * 60)

print("Total de observações:", len(producao_cc_integrada))

print("\nTipos de objetos encontrados:")
display(
    producao_cc_integrada["data_parto_prevista"]
    .map(lambda x: type(x).__name__)
    .value_counts(dropna=False)
)

print("\nValores mais frequentes:")
display(
    producao_cc_integrada["data_parto_prevista"]
    .astype(str)
    .value_counts(dropna=False)
    .head(15)
)

# Conversão apenas para análise.
# A coluna original não será modificada nesta etapa.
data_parto_prevista_convertida = pd.to_datetime(
    producao_cc_integrada["data_parto_prevista"],
    errors="coerce"
)

print("\nDisponibilidade após interpretação como data:")
print(
    "Datas válidas:",
    data_parto_prevista_convertida.notna().sum()
)

print(
    "Valores não interpretáveis/ausentes:",
    data_parto_prevista_convertida.isna().sum()
)

print(
    "Cobertura real de datas:",
    round(
        data_parto_prevista_convertida.notna().mean() * 100,
        2
    ),
    "%"
)

print("\nValores originais que não puderam ser interpretados como data:")

display(
    producao_cc_integrada.loc[
        data_parto_prevista_convertida.isna(),
        "data_parto_prevista"
    ]
    .astype(str)
    .value_counts(dropna=False)
)

### 11.34 Validação temporal de `data_parto_prevista`

A avaliação inicial demonstrou que 503 das 733 observações (68,62%) possuem uma data de parto previsto interpretável. Os demais registros correspondem principalmente ao marcador textual `-`, utilizado para representar ausência de informação, além de um registro armazenado apenas como horário.

Como a data de parto prevista representa um evento futuro em relação ao controle leiteiro, sua consistência será avaliada por meio da diferença entre `data_parto_prevista` e `data_prova`.

Essa análise permitirá verificar se as datas disponíveis são temporalmente coerentes e caracterizar o intervalo entre o controle leiteiro e o parto previsto antes de qualquer decisão sobre sua utilização como variável candidata.

In [ ]:
# ============================================================
# VALIDAÇÃO TEMPORAL DE data_parto_prevista
# ============================================================

validacao_parto_previsto = producao_cc_integrada[
    [
        "brinco",
        "data_prova",
        "data_parto_prevista",
        "estado_reprodutivo_padronizado",
        "del"
    ]
].copy()

# Conversão apenas na tabela de análise
validacao_parto_previsto["data_parto_prevista_convertida"] = pd.to_datetime(
    validacao_parto_previsto["data_parto_prevista"],
    errors="coerce"
)

# Diferença entre o parto previsto e a data do controle
validacao_parto_previsto["dias_ate_parto_previsto"] = (
    validacao_parto_previsto["data_parto_prevista_convertida"]
    - validacao_parto_previsto["data_prova"]
).dt.days

casos_com_parto_previsto = validacao_parto_previsto[
    validacao_parto_previsto["data_parto_prevista_convertida"].notna()
].copy()

print("VALIDAÇÃO TEMPORAL DE data_parto_prevista")
print("-" * 60)

print(
    "Observações com data válida:",
    len(casos_com_parto_previsto)
)

print(
    "Partos previstos antes da data do controle:",
    (casos_com_parto_previsto["dias_ate_parto_previsto"] < 0).sum()
)

print(
    "Partos previstos na própria data do controle:",
    (casos_com_parto_previsto["dias_ate_parto_previsto"] == 0).sum()
)

print(
    "Partos previstos após a data do controle:",
    (casos_com_parto_previsto["dias_ate_parto_previsto"] > 0).sum()
)

print("\nDistribuição dos dias até o parto previsto:")

display(
    casos_com_parto_previsto["dias_ate_parto_previsto"].describe()
)

print("\nCasos com parto previsto anterior ou igual à data do controle:")

display(
    casos_com_parto_previsto.loc[
        casos_com_parto_previsto["dias_ate_parto_previsto"] <= 0,
        [
            "brinco",
            "data_prova",
            "data_parto_prevista_convertida",
            "dias_ate_parto_previsto",
            "estado_reprodutivo_padronizado",
            "del"
        ]
    ].sort_values("dias_ate_parto_previsto")
)

### 11.35 Relação entre `data_parto_prevista` e estado reprodutivo

As 503 datas válidas de parto previsto apresentaram consistência temporal, estando todas posicionadas após a respectiva data do controle leiteiro.

Entretanto, antes de derivar uma nova variável a partir dessas datas, é necessário avaliar sua relação com o estado reprodutivo registrado no mesmo controle. A data de parto prevista pode estar diretamente associada à confirmação de gestação e, portanto, representar informação parcialmente redundante em relação ao estado reprodutivo.

Nesta etapa será avaliada a disponibilidade conjunta dessas informações e a distribuição dos estados reprodutivos entre observações com e sem data de parto prevista.

In [ ]:
# ============================================================
# PARTO PREVISTO x ESTADO REPRODUTIVO
# ============================================================

analise_reprodutiva = validacao_parto_previsto.copy()

analise_reprodutiva["possui_parto_previsto"] = (
    analise_reprodutiva["data_parto_prevista_convertida"].notna()
)

print("PARTO PREVISTO x ESTADO REPRODUTIVO")
print("-" * 60)

print("\nEstado reprodutivo nas observações COM parto previsto:")

display(
    analise_reprodutiva.loc[
        analise_reprodutiva["possui_parto_previsto"],
        "estado_reprodutivo_padronizado"
    ]
    .value_counts(dropna=False)
)

print("\nEstado reprodutivo nas observações SEM parto previsto:")

display(
    analise_reprodutiva.loc[
        ~analise_reprodutiva["possui_parto_previsto"],
        "estado_reprodutivo_padronizado"
    ]
    .value_counts(dropna=False)
)

print("\nTabela cruzada:")

display(
    pd.crosstab(
        analise_reprodutiva["estado_reprodutivo_padronizado"],
        analise_reprodutiva["possui_parto_previsto"],
        margins=True,
        dropna=False
    )
)

### 11.36 Investigação de registros `Negativo` com data de parto prevista

O cruzamento entre as informações reprodutivas mostrou que todos os 298 registros classificados como `DG +` possuem uma data de parto prevista. Entretanto, também foram identificadas 129 observações classificadas como `Negativo` que apresentam uma data futura de parto prevista.

Essa combinação não será interpretada automaticamente como erro, pois os campos podem representar informações registradas ou atualizadas em momentos distintos no sistema de controle da propriedade.

Antes da definição das variáveis reprodutivas candidatas à modelagem, esses registros serão analisados longitudinalmente por animal, observando a evolução do estado reprodutivo e da data de parto prevista entre controles sucessivos.

In [ ]:
# ============================================================
# INVESTIGAÇÃO: NEGATIVO + PARTO PREVISTO
# ============================================================

casos_negativo_parto = analise_reprodutiva[
    (analise_reprodutiva["estado_reprodutivo_padronizado"] == "Negativo")
    & (analise_reprodutiva["possui_parto_previsto"])
].copy()

print("NEGATIVO + PARTO PREVISTO")
print("-" * 60)

print("Observações:", len(casos_negativo_parto))
print("Animais distintos:", casos_negativo_parto["brinco"].nunique())

print("\nDias até o parto previsto nesses casos:")
display(
    casos_negativo_parto["dias_ate_parto_previsto"].describe()
)

print("\nDistribuição das datas de controle:")
display(
    casos_negativo_parto["data_prova"]
    .value_counts()
    .sort_index()
)

print("\nCasos individuais:")
display(
    casos_negativo_parto[
        [
            "brinco",
            "data_prova",
            "estado_reprodutivo_padronizado",
            "data_parto_prevista_convertida",
            "dias_ate_parto_previsto",
            "del"
        ]
    ].sort_values(["brinco", "data_prova"])
)

### 11.37 Decisão sobre as informações reprodutivas

A análise demonstrou que `data_parto_prevista` possui 503 datas válidas (68,62% das observações), todas posteriores à respectiva data do controle leiteiro. Os intervalos variaram entre 62 e 273 dias.

O cruzamento com `estado_reprodutivo_padronizado` mostrou que as duas informações não são equivalentes. Todos os 298 registros classificados como `DG +` apresentaram data de parto prevista, porém também foram encontradas datas previstas em registros classificados como `Negativo`, sem estado reprodutivo informado e, pontualmente, nas categorias `Descarte` e `Abortou`.

Nos 129 registros classificados como `Negativo` e que possuíam parto previsto, o intervalo até a data prevista variou entre 240 e 273 dias. Além disso, observou-se que a previsão pode ser atualizada entre controles sucessivos do mesmo animal. Dessa forma, não há evidência suficiente para tratar essas combinações automaticamente como inconsistências.

A data absoluta de parto previsto não será utilizada diretamente como atributo do modelo. A possibilidade de derivar uma variável temporal, como dias até o parto previsto, será mantida para avaliação posterior, juntamente com o estado reprodutivo. A seleção definitiva deverá considerar a disponibilidade dessas informações, sua interpretação temporal e o possível ganho de informação para a modelagem.

Os marcadores `-` e o registro contendo apenas horário serão tratados como ausência de data na etapa de preparação da base analítica.

### 11.38 Avaliação das variáveis cadastrais

Após a integração com o cadastro dos animais, 472 das 733 observações de controle leiteiro (64,39%) possuem informações cadastrais associadas.

Nesta etapa serão avaliadas as variáveis `raça`, `data_nascimento` e `ativo`. O objetivo é verificar a consistência das categorias raciais, a validade das datas de nascimento e a variabilidade do status cadastral.

A data de nascimento não será utilizada diretamente como atributo preditor. Caso apresente consistência temporal, será utilizada posteriormente para derivar a idade do animal na data de cada controle leiteiro.

A análise também permitirá identificar padronizações necessárias antes da construção da base candidata para Machine Learning.

In [ ]:
# ============================================================
# AVALIAÇÃO DAS VARIÁVEIS CADASTRAIS
# ============================================================

print("VARIÁVEIS CADASTRAIS")
print("-" * 60)

# ------------------------------------------------------------
# RAÇA
# ------------------------------------------------------------

print("\n1. RAÇA")
print("-" * 30)

print(
    "Observações com raça:",
    producao_cc_integrada["raça"].notna().sum()
)

print(
    "Categorias distintas:",
    producao_cc_integrada["raça"].nunique(dropna=True)
)

print("\nDistribuição das categorias:")

display(
    producao_cc_integrada["raça"]
    .value_counts(dropna=False)
)

# ------------------------------------------------------------
# DATA DE NASCIMENTO
# ------------------------------------------------------------

print("\n2. DATA DE NASCIMENTO")
print("-" * 30)

data_nascimento_convertida = pd.to_datetime(
    producao_cc_integrada["data_nascimento"],
    errors="coerce"
)

print(
    "Valores preenchidos originalmente:",
    producao_cc_integrada["data_nascimento"].notna().sum()
)

print(
    "Datas válidas após conversão:",
    data_nascimento_convertida.notna().sum()
)

print(
    "Valores não interpretáveis:",
    (
        producao_cc_integrada["data_nascimento"].notna()
        & data_nascimento_convertida.isna()
    ).sum()
)

# Idade aproximada na data do controle
idade_anos_teste = (
    (
        producao_cc_integrada["data_prova"]
        - data_nascimento_convertida
    ).dt.days / 365.25
)

print("\nIdade na data do controle (anos):")
display(
    idade_anos_teste.describe()
)

print(
    "\nDatas de nascimento posteriores ao controle:",
    (idade_anos_teste < 0).sum()
)

# ------------------------------------------------------------
# STATUS ATIVO
# ------------------------------------------------------------

print("\n3. STATUS CADASTRAL")
print("-" * 30)

display(
    producao_cc_integrada["ativo"]
    .value_counts(dropna=False)
)

### 11.39 Investigação de inconsistências na idade dos animais

A conversão das datas de nascimento foi possível para todas as 472 observações que possuem cadastro associado. Entretanto, foram identificadas duas observações em que a data de nascimento registrada é posterior à data do respectivo controle leiteiro, resultando em idade negativa.

Como essa situação é biologicamente impossível, os registros serão rastreados antes da criação da variável derivada de idade. Nenhuma data será corrigida ou excluída automaticamente nesta etapa.

In [ ]:
# ============================================================
# INVESTIGAÇÃO DAS IDADES NEGATIVAS
# ============================================================

casos_idade_negativa = producao_cc_integrada.loc[
    idade_anos_teste < 0,
    [
        "brinco",
        "data_prova",
        "kg_leite",
        "id_animal_extraido",
        "id_animal",
        "identificação",
        "raça",
        "data_nascimento",
        "del"
    ]
].copy()

casos_idade_negativa["idade_anos_calculada"] = (
    (
        casos_idade_negativa["data_prova"]
        - pd.to_datetime(
            casos_idade_negativa["data_nascimento"],
            errors="coerce"
        )
    ).dt.days / 365.25
)

print("CASOS COM IDADE NEGATIVA")
print("-" * 60)

print("Observações:", len(casos_idade_negativa))
print(
    "Animais distintos:",
    casos_idade_negativa["brinco"].nunique()
)

display(casos_idade_negativa)

### 11.40 Decisão sobre as variáveis cadastrais

A avaliação cadastral mostrou que 472 das 733 observações possuem informações associadas ao cadastro dos animais.

A variável `raça` apresentou oito categorias distintas. Foram identificadas apenas inconsistências de padronização textual, como `HVB` e `Hvb`, além de diferença de capitalização em `girolando 1/2`. Essas representações serão padronizadas sem agrupar composições raciais distintas.

Todas as 472 datas de nascimento preenchidas puderam ser convertidas para formato de data. Entretanto, duas observações, pertencentes ao mesmo animal (`JULIETE 1250`), apresentaram data de nascimento posterior à data do controle leiteiro. Como não há informação suficiente para determinar a data correta, o valor original será preservado para rastreabilidade, mas não será utilizado para o cálculo da idade desse animal.

Para os demais registros válidos, a data de nascimento será transformada em idade do animal na data de cada controle leiteiro, pois essa representação possui maior significado para a modelagem do que a utilização da data absoluta de nascimento.

A variável `ativo` não será utilizada como atributo preditor, uma vez que todos os 472 registros cadastrados apresentam a mesma categoria (`ativo`), não havendo variabilidade informativa para o modelo.

### 11.41 Criação das variáveis derivadas e padronizadas

Após a validação individual das principais variáveis da base de Cerqueira César, será criada uma cópia destinada à preparação das informações candidatas à modelagem.

Nesta etapa, os dados originais serão preservados e novas variáveis serão derivadas apenas a partir de informações previamente avaliadas.

Serão realizadas as seguintes transformações:

- padronização textual da raça, sem agrupar composições raciais distintas;
- cálculo da idade do animal na data de cada controle leiteiro;
- invalidação da idade quando a data de nascimento for posterior ao controle;
- interpretação válida da data de parto prevista;
- cálculo do número de dias entre o controle e o parto previsto;
- extração do mês do controle leiteiro como informação temporal.

Nenhum valor ausente será imputado nesta etapa e nenhuma observação será excluída.

In [ ]:
# ============================================================
# CRIAÇÃO DAS VARIÁVEIS DERIVADAS E PADRONIZADAS
# ============================================================

base_cc_preparada = producao_cc_integrada.copy()

# ------------------------------------------------------------
# 1. PADRONIZAÇÃO DA RAÇA
# ------------------------------------------------------------

base_cc_preparada["raca_padronizada"] = (
    base_cc_preparada["raça"]
    .astype("string")
    .str.strip()
    .replace({
        "Hvb": "HVB",
        "girolando 1/2": "Girolando 1/2"
    })
)

# ------------------------------------------------------------
# 2. DATA DE NASCIMENTO E IDADE
# ------------------------------------------------------------

base_cc_preparada["data_nascimento_convertida"] = pd.to_datetime(
    base_cc_preparada["data_nascimento"],
    errors="coerce"
)

base_cc_preparada["idade_anos"] = (
    (
        base_cc_preparada["data_prova"]
        - base_cc_preparada["data_nascimento_convertida"]
    ).dt.days / 365.25
)

# Idades negativas são biologicamente impossíveis.
# O valor original permanece preservado em data_nascimento.
base_cc_preparada.loc[
    base_cc_preparada["idade_anos"] < 0,
    "idade_anos"
] = np.nan

# ------------------------------------------------------------
# 3. DATA DE PARTO PREVISTA
# ------------------------------------------------------------

base_cc_preparada["data_parto_prevista_validada"] = pd.to_datetime(
    base_cc_preparada["data_parto_prevista"],
    errors="coerce"
)

base_cc_preparada["dias_ate_parto_previsto"] = (
    base_cc_preparada["data_parto_prevista_validada"]
    - base_cc_preparada["data_prova"]
).dt.days

# ------------------------------------------------------------
# 4. INFORMAÇÃO TEMPORAL DO CONTROLE
# ------------------------------------------------------------

base_cc_preparada["mes_controle"] = (
    base_cc_preparada["data_prova"].dt.month
)

# ------------------------------------------------------------
# VERIFICAÇÃO DAS NOVAS VARIÁVEIS
# ------------------------------------------------------------

print("BASE PREPARADA DE CERQUEIRA CÉSAR")
print("-" * 60)

print("Observações:", len(base_cc_preparada))
print("Colunas:", base_cc_preparada.shape[1])

print("\nCobertura das variáveis criadas:")

variaveis_criadas = [
    "raca_padronizada",
    "idade_anos",
    "data_parto_prevista_validada",
    "dias_ate_parto_previsto",
    "mes_controle"
]

resumo_variaveis_criadas = pd.DataFrame({
    "preenchidos": base_cc_preparada[variaveis_criadas].notna().sum(),
    "ausentes": base_cc_preparada[variaveis_criadas].isna().sum()
})

resumo_variaveis_criadas["cobertura_%"] = (
    resumo_variaveis_criadas["preenchidos"]
    / len(base_cc_preparada)
    * 100
).round(2)

display(resumo_variaveis_criadas)

print("\nDistribuição da raça padronizada:")

display(
    base_cc_preparada["raca_padronizada"]
    .value_counts(dropna=False)
)

print("\nResumo da idade calculada:")

display(
    base_cc_preparada["idade_anos"].describe()
)

print("\nMeses disponíveis:")

display(
    base_cc_preparada["mes_controle"]
    .value_counts()
    .sort_index()
)

### 11.42 Investigação de idades biologicamente incompatíveis

Após a criação da variável `idade_anos`, foram identificados registros com idade extremamente baixa na data do controle leiteiro.

Embora essas idades não sejam negativas, valores muito baixos são incompatíveis com a presença do animal em uma base de produção leiteira. Dessa forma, será realizada uma verificação específica dos registros com idade inferior a um ano.

Nenhuma data de nascimento será corrigida automaticamente. Caso sejam identificadas inconsistências, os valores originais serão preservados para rastreabilidade e a idade derivada correspondente será considerada ausente na base analítica.

In [ ]:
# ============================================================
# INVESTIGAÇÃO DE IDADES MUITO BAIXAS
# ============================================================

casos_idade_muito_baixa = base_cc_preparada.loc[
    base_cc_preparada["idade_anos"] < 1,
    [
        "brinco",
        "data_prova",
        "kg_leite",
        "del",
        "id_animal_extraido",
        "identificação",
        "raca_padronizada",
        "data_nascimento",
        "data_nascimento_convertida",
        "idade_anos"
    ]
].copy()

print("IDADES INFERIORES A 1 ANO")
print("-" * 60)

print("Observações:", len(casos_idade_muito_baixa))

print(
    "Animais distintos:",
    casos_idade_muito_baixa["brinco"].nunique()
)

display(
    casos_idade_muito_baixa.sort_values(
        ["brinco", "data_prova"]
    )
)

### 11.43 Tratamento da inconsistência na data de nascimento

A investigação das idades inferiores a um ano mostrou que os registros pertencem ao mesmo animal (`JULIETE 1250`), cuja data de nascimento cadastrada é 26/05/2026.

Entretanto, esse animal possui controles leiteiros anteriores a essa data e já se encontrava em lactação, demonstrando que a data de nascimento cadastrada é inconsistente.

Dessa forma, a data original será preservada para rastreabilidade, mas a variável derivada `idade_anos` será considerada ausente em todas as observações desse animal. Nenhuma data de nascimento será estimada ou corrigida sem uma fonte que permita determinar seu valor verdadeiro.

In [ ]:
# ============================================================
# TRATAMENTO DA DATA DE NASCIMENTO INCONSISTENTE
# ============================================================

# Identifica animais cuja data de nascimento cadastrada
# ocorre depois de pelo menos um controle leiteiro
ids_nascimento_inconsistente = (
    base_cc_preparada.loc[
        base_cc_preparada["data_nascimento_convertida"]
        > base_cc_preparada["data_prova"],
        "id_animal_extraido"
    ]
    .dropna()
    .unique()
)

print("Animais com data de nascimento inconsistente:")
print(ids_nascimento_inconsistente)

# A data original é mantida.
# Apenas a idade derivada deixa de ser considerada confiável
# para todas as observações desses animais.
base_cc_preparada.loc[
    base_cc_preparada["id_animal_extraido"].isin(
        ids_nascimento_inconsistente
    ),
    "idade_anos"
] = np.nan

print("\nVALIDAÇÃO APÓS O TRATAMENTO")
print("-" * 60)

print(
    "Idades válidas:",
    base_cc_preparada["idade_anos"].notna().sum()
)

print(
    "Idades ausentes:",
    base_cc_preparada["idade_anos"].isna().sum()
)

print(
    "Idades negativas:",
    (base_cc_preparada["idade_anos"] < 0).sum()
)

print(
    "Idades inferiores a 1 ano:",
    (base_cc_preparada["idade_anos"] < 1).sum()
)

display(
    base_cc_preparada["idade_anos"].describe()
)

### 11.44 Classificação das variáveis para a futura modelagem

Após as etapas de consolidação, integração, validação e transformação dos dados de Cerqueira César, as variáveis disponíveis serão classificadas conforme sua função na análise.

A classificação considera:

- a variável-alvo da tarefa de regressão;
- atributos potencialmente disponíveis no momento da previsão;
- variáveis que necessitam de transformação;
- informações utilizadas apenas para identificação ou rastreabilidade;
- atributos inadequados devido à redundância, ausência de variabilidade, baixa disponibilidade ou risco de utilização indevida de informação.

Esta classificação ainda não representa a seleção definitiva de atributos do modelo. As variáveis classificadas como candidatas poderão ser avaliadas posteriormente durante a construção e comparação dos modelos de Machine Learning.

In [ ]:
# ============================================================
# CLASSIFICAÇÃO DAS VARIÁVEIS DE CERQUEIRA CÉSAR
# ============================================================

decisao_variaveis_cc = pd.DataFrame([
    {
        "variavel": "kg_leite",
        "papel": "Alvo",
        "decisao": "Utilizar",
        "justificativa": "Produção leiteira observada que será estimada pelo modelo."
    },
    {
        "variavel": "del",
        "papel": "Preditora candidata",
        "decisao": "Utilizar",
        "justificativa": "Variável validada temporalmente e representativa do estágio de lactação."
    },
    {
        "variavel": "kg_anterior",
        "papel": "Preditora candidata",
        "decisao": "Utilizar",
        "justificativa": "Apresentou elevada consistência com o histórico produtivo e representa informação anterior ao controle atual."
    },
    {
        "variavel": "raca_padronizada",
        "papel": "Preditora candidata",
        "decisao": "Avaliar no ML",
        "justificativa": "Característica biológica relevante, disponível apenas para parte das observações."
    },
    {
        "variavel": "idade_anos",
        "papel": "Preditora candidata",
        "decisao": "Avaliar no ML",
        "justificativa": "Derivada da data de nascimento após validação das inconsistências cadastrais."
    },
    {
        "variavel": "estado_reprodutivo_padronizado",
        "papel": "Preditora candidata",
        "decisao": "Avaliar no ML",
        "justificativa": "Informação reprodutiva potencialmente relevante, porém com valores ausentes."
    },
    {
        "variavel": "dias_ate_parto_previsto",
        "papel": "Preditora candidata",
        "decisao": "Avaliar no ML",
        "justificativa": "Representação temporal derivada da previsão de parto; possui cobertura parcial."
    },
    {
        "variavel": "tipo_parto_validado",
        "papel": "Preditora candidata",
        "decisao": "Avaliar no ML",
        "justificativa": "Informação temporalmente validada, porém disponível somente em parte das observações."
    },
    {
        "variavel": "mes_controle",
        "papel": "Preditora candidata",
        "decisao": "Avaliar no ML",
        "justificativa": "Representa componente temporal do controle e poderá capturar variações ao longo do período observado."
    },
    {
        "variavel": "data_prova",
        "papel": "Temporal / rastreabilidade",
        "decisao": "Não usar diretamente",
        "justificativa": "Será mantida para ordenação temporal, validação e futuras divisões dos dados."
    },
    {
        "variavel": "data_ultimo_parto",
        "papel": "Auxiliar",
        "decisao": "Não usar diretamente",
        "justificativa": "Sua informação temporal já é representada de forma mais adequada pelo DEL."
    },
    {
        "variavel": "data_parto_prevista",
        "papel": "Auxiliar",
        "decisao": "Não usar diretamente",
        "justificativa": "A data absoluta foi substituída pela variável derivada dias_ate_parto_previsto."
    },
    {
        "variavel": "data_nascimento",
        "papel": "Auxiliar cadastral",
        "decisao": "Não usar diretamente",
        "justificativa": "Foi utilizada para calcular idade_anos."
    },
    {
        "variavel": "ativo",
        "papel": "Cadastro",
        "decisao": "Não utilizar",
        "justificativa": "Não apresenta variabilidade entre os animais cadastrados."
    },
    {
        "variavel": "numero_crias_validado",
        "papel": "Auxiliar",
        "decisao": "Não priorizar",
        "justificativa": "Apresenta baixa variabilidade e cobertura parcial."
    },
    {
        "variavel": "brinco",
        "papel": "Identificação",
        "decisao": "Não usar como preditora",
        "justificativa": "Identificador do animal utilizado para rastreabilidade."
    },
    {
        "variavel": "id_animal_extraido",
        "papel": "Identificação",
        "decisao": "Não usar como preditora",
        "justificativa": "Identificador utilizado na integração das bases."
    },
    {
        "variavel": "id_animal",
        "papel": "Identificação",
        "decisao": "Não usar como preditora",
        "justificativa": "Identificador cadastral, sem significado biológico direto."
    },
    {
        "variavel": "identificação",
        "papel": "Identificação",
        "decisao": "Não usar como preditora",
        "justificativa": "Nome do animal utilizado apenas para rastreabilidade."
    },
    {
        "variavel": "animal_encontrado_cadastro",
        "papel": "Controle de qualidade",
        "decisao": "Não usar como preditora",
        "justificativa": "Indica apenas se houve correspondência com o cadastro."
    },
    {
        "variavel": "conflito_kg_leite",
        "papel": "Controle de qualidade",
        "decisao": "Não usar como preditora",
        "justificativa": "Flag utilizada para documentar conflitos encontrados na fonte."
    },
    {
        "variavel": "conflito_estado_reprodutivo",
        "papel": "Controle de qualidade",
        "decisao": "Não usar como preditora",
        "justificativa": "Flag utilizada para controle da qualidade dos registros."
    },
    {
        "variavel": "possui_conflito",
        "papel": "Controle de qualidade",
        "decisao": "Não usar como preditora",
        "justificativa": "Indicador de rastreabilidade dos conflitos existentes."
    }
])

# Adiciona a cobertura observada na base preparada
decisao_variaveis_cc["cobertura_%"] = decisao_variaveis_cc[
    "variavel"
].map(
    lambda coluna:
        round(
            base_cc_preparada[coluna].notna().mean() * 100,
            2
        )
        if coluna in base_cc_preparada.columns
        else np.nan
)
# Ajuste da cobertura semântica de data_parto_prevista:
# valores "-" e registros não interpretáveis não representam datas válidas
decisao_variaveis_cc.loc[
    decisao_variaveis_cc["variavel"] == "data_parto_prevista",
    "cobertura_%"
] = round(
    base_cc_preparada["data_parto_prevista_validada"]
    .notna()
    .mean() * 100,
    2
)

display(decisao_variaveis_cc)

### 11.45 Construção da base candidata de Cerqueira César

Após a validação e classificação das variáveis, será construída uma base processada específica de Cerqueira César.

Essa base preservará a variável-alvo, os atributos candidatos à futura modelagem e campos necessários para identificação, rastreabilidade e controle temporal.

A presença de uma variável nesta base ainda não implica sua utilização definitiva no modelo. A seleção final será realizada posteriormente, após a estruturação da base da UENP e a harmonização das informações disponíveis nas duas propriedades.

Nenhuma observação será excluída nesta etapa em função de valores ausentes. Os três registros cujo valor de produção permanece indefinido por conflito na fonte original também serão preservados na base processada, mantendo a rastreabilidade do conjunto consolidado.

In [ ]:
# ============================================================
# BASE CANDIDATA DE CERQUEIRA CÉSAR
# ============================================================

colunas_base_cc = [
    # Origem / rastreabilidade
    "brinco",
    "id_animal_extraido",
    "data_prova",

    # Alvo
    "kg_leite",

    # Preditoras candidatas
    "del",
    "kg_anterior",
    "raca_padronizada",
    "idade_anos",
    "estado_reprodutivo_padronizado",
    "dias_ate_parto_previsto",
    "tipo_parto_validado",
    "mes_controle",

    # Controle de qualidade
    "animal_encontrado_cadastro",
    "conflito_kg_leite",
    "conflito_estado_reprodutivo",
    "possui_conflito"
]

base_cc_ml = base_cc_preparada[colunas_base_cc].copy()

# Identificação da propriedade para futura harmonização
base_cc_ml.insert(
    0,
    "propriedade",
    "Cerqueira César"
)

print("BASE CANDIDATA DE CERQUEIRA CÉSAR")
print("-" * 60)

print("Observações:", base_cc_ml.shape[0])
print("Colunas:", base_cc_ml.shape[1])

print("\nColunas da base:")

for i, coluna in enumerate(base_cc_ml.columns, start=1):
    print(f"{i:02d}. {coluna}")

print("\nCobertura das variáveis:")

resumo_base_cc_ml = pd.DataFrame({
    "preenchidos": base_cc_ml.notna().sum(),
    "ausentes": base_cc_ml.isna().sum()
})

resumo_base_cc_ml["cobertura_%"] = (
    resumo_base_cc_ml["preenchidos"]
    / len(base_cc_ml)
    * 100
).round(2)

display(resumo_base_cc_ml)

### 11.46 Validação final e armazenamento da base processada de Cerqueira César

Antes do encerramento da preparação dos dados de Cerqueira César, a base candidata será submetida a uma validação final de integridade.

Serão verificadas a quantidade de observações, a unicidade da combinação animal-data, a presença dos conflitos previamente identificados, a disponibilidade da variável-alvo e a consistência das principais variáveis derivadas.

A base processada será então armazenada separadamente dos arquivos originais. Nenhuma observação será removida nesta etapa, preservando-se os 733 controles consolidados e os indicadores necessários para rastreabilidade.

A definição da matriz efetivamente utilizada no treinamento dos modelos será realizada posteriormente, após a preparação e harmonização dos dados da UENP.

In [ ]:
# ============================================================
# VALIDAÇÃO FINAL DA BASE DE CERQUEIRA CÉSAR
# ============================================================

print("VALIDAÇÃO FINAL — CERQUEIRA CÉSAR")
print("-" * 60)

# ------------------------------------------------------------
# 1. DIMENSÕES
# ------------------------------------------------------------

print("Observações:", base_cc_ml.shape[0])
print("Colunas:", base_cc_ml.shape[1])

# ------------------------------------------------------------
# 2. DUPLICIDADES
# ------------------------------------------------------------

duplicados_animal_data = base_cc_ml.duplicated(
    subset=["brinco", "data_prova"]
).sum()

duplicados_exatos = base_cc_ml.duplicated().sum()

print("\nDUPLICIDADES")
print("-" * 30)

print(
    "Duplicidades animal + data:",
    duplicados_animal_data
)

print(
    "Linhas totalmente duplicadas:",
    duplicados_exatos
)

# ------------------------------------------------------------
# 3. VARIÁVEL-ALVO
# ------------------------------------------------------------

print("\nVARIÁVEL-ALVO")
print("-" * 30)

print(
    "kg_leite preenchido:",
    base_cc_ml["kg_leite"].notna().sum()
)

print(
    "kg_leite ausente:",
    base_cc_ml["kg_leite"].isna().sum()
)

# ------------------------------------------------------------
# 4. CONFLITOS
# ------------------------------------------------------------

print("\nCONFLITOS PRESERVADOS")
print("-" * 30)

print(
    "Conflitos de kg_leite:",
    base_cc_ml["conflito_kg_leite"].sum()
)

print(
    "Conflitos de estado reprodutivo:",
    base_cc_ml["conflito_estado_reprodutivo"].sum()
)

print(
    "Observações com algum conflito:",
    base_cc_ml["possui_conflito"].sum()
)

# ------------------------------------------------------------
# 5. VALIDAÇÕES BIOLÓGICAS / TEMPORAIS
# ------------------------------------------------------------

print("\nCONSISTÊNCIA DAS VARIÁVEIS")
print("-" * 30)

print(
    "DEL negativo:",
    (base_cc_ml["del"] < 0).sum()
)

print(
    "kg_anterior negativo:",
    (base_cc_ml["kg_anterior"] < 0).sum()
)

print(
    "Idades inferiores a 1 ano:",
    (base_cc_ml["idade_anos"] < 1).sum()
)

print(
    "Partos previstos no passado ou no dia do controle:",
    (
        base_cc_ml["dias_ate_parto_previsto"]
        .dropna()
        <= 0
    ).sum()
)

# ------------------------------------------------------------
# 6. TESTES FINAIS
# ------------------------------------------------------------

testes_finais_cc = {
    "733 observações preservadas":
        len(base_cc_ml) == 733,

    "Sem duplicidade animal-data":
        duplicados_animal_data == 0,

    "Sem linhas totalmente duplicadas":
        duplicados_exatos == 0,

    "3 alvos ausentes por conflito":
        base_cc_ml["kg_leite"].isna().sum() == 3,

    "3 conflitos de produção":
        base_cc_ml["conflito_kg_leite"].sum() == 3,

    "3 conflitos reprodutivos":
        base_cc_ml["conflito_estado_reprodutivo"].sum() == 3,

    "6 observações com algum conflito":
        base_cc_ml["possui_conflito"].sum() == 6,

    "Sem DEL negativo":
        (base_cc_ml["del"] < 0).sum() == 0,

    "Sem kg_anterior negativo":
        (base_cc_ml["kg_anterior"] < 0).sum() == 0,

    "Sem idade inferior a 1 ano":
        (base_cc_ml["idade_anos"] < 1).sum() == 0,

    "Partos previstos somente no futuro":
        (
            base_cc_ml["dias_ate_parto_previsto"]
            .dropna()
            <= 0
        ).sum() == 0
}

print("\nTESTES FINAIS")
print("-" * 30)

for teste, resultado in testes_finais_cc.items():
    print(
        f"{'OK' if resultado else 'ERRO'} - {teste}"
    )

base_cc_validada = all(testes_finais_cc.values())

print("\nBase de Cerqueira César validada:", base_cc_validada)

### 11.47 Armazenamento da base processada de Cerqueira César

Após a aprovação dos testes finais de integridade, a base processada de Cerqueira César será armazenada separadamente dos arquivos originais.

Serão salvos tanto o conjunto de dados preparado para futura harmonização com a UENP quanto a tabela que documenta as decisões adotadas para cada variável.

Os arquivos originais permanecem inalterados, garantindo a rastreabilidade de todo o processo de preparação dos dados.

In [ ]:
# ============================================================
# SALVAMENTO DA BASE PROCESSADA DE CERQUEIRA CÉSAR
# ============================================================

from pathlib import Path

# Localiza a pasta de dados processados
candidatos_pasta = [
    Path("../dados/processados"),
    Path("dados/processados")
]

pasta_processados = None

for pasta in candidatos_pasta:
    if pasta.parent.exists():
        pasta_processados = pasta
        break

if pasta_processados is None:
    raise FileNotFoundError(
        "Não foi possível localizar a pasta dados/processados."
    )

pasta_processados.mkdir(
    parents=True,
    exist_ok=True
)

# Caminhos dos arquivos
arquivo_base_cc = (
    pasta_processados
    / "cerqueira_cesar_base_processada.csv"
)

arquivo_decisoes_cc = (
    pasta_processados
    / "cerqueira_cesar_decisao_variaveis.csv"
)

# Salvamento
base_cc_ml.to_csv(
    arquivo_base_cc,
    index=False,
    encoding="utf-8-sig"
)

decisao_variaveis_cc.to_csv(
    arquivo_decisoes_cc,
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos salvos com sucesso:")
print("-", arquivo_base_cc)
print("-", arquivo_decisoes_cc)

# 12. Preparação e estruturação dos dados da UENP

Os dados da segunda propriedade analisada correspondem à Fazenda Escola da Universidade Estadual do Norte do Paraná (UENP) e estão armazenados no arquivo `Bovinocultura UENP.xlsx`.

Diferentemente dos controles leiteiros de Cerqueira César, as informações da UENP estão distribuídas em múltiplas planilhas, contendo dados cadastrais, produtivos, reprodutivos, sanitários, de pesagem e manejo.

Nesta etapa será inicialmente realizada uma inspeção estrutural do arquivo, com o objetivo de identificar as planilhas disponíveis, suas dimensões, colunas e organização geral.

Nenhuma transformação ou exclusão de dados será realizada antes da compreensão da estrutura das diferentes fontes.

In [ ]:
# ============================================================
# 12.1 INSPEÇÃO GERAL DO ARQUIVO DA UENP
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# Localiza o arquivo da UENP
candidatos_uenp = [
    Path("../dados/originais/uenp/Bovinocultura UENP.xlsx"),
    Path("dados/originais/uenp/Bovinocultura UENP.xlsx")
]

arquivo_uenp = None

for caminho in candidatos_uenp:
    if caminho.exists():
        arquivo_uenp = caminho
        break

if arquivo_uenp is None:
    raise FileNotFoundError(
        "Não foi possível localizar o arquivo Bovinocultura UENP.xlsx."
    )

print("Arquivo localizado:")
print(arquivo_uenp)

# ------------------------------------------------------------
# LISTAGEM DAS PLANILHAS
# ------------------------------------------------------------

excel_uenp = pd.ExcelFile(arquivo_uenp)

print("\nPLANILHAS DISPONÍVEIS")
print("-" * 60)

for i, aba in enumerate(excel_uenp.sheet_names, start=1):
    print(f"{i:02d}. {aba}")

print("\nTotal de planilhas:", len(excel_uenp.sheet_names))

### 12.2 Inspeção das principais fontes de dados

A partir das 18 planilhas existentes no arquivo da UENP, foram inicialmente selecionadas as fontes com maior potencial de contribuição para a construção da base analítica.

Nesta etapa serão inspecionadas as planilhas de cadastro geral, produção leiteira, pesagem, reprodução e doenças.

O objetivo é identificar suas dimensões, nomes das colunas, tipos gerais de organização e exemplos de registros antes da definição de qualquer procedimento de transformação ou integração.

In [ ]:
# ============================================================
# 12.2 INSPEÇÃO DAS PRINCIPAIS PLANILHAS DA UENP
# ============================================================

abas_prioritarias = [
    "Visão Geral",
    "Produção",
    "Pesagem 2026",
    "Reprodução",
    "Doenças"
]

dados_uenp = {}

for aba in abas_prioritarias:

    df = pd.read_excel(
        arquivo_uenp,
        sheet_name=aba
    )

    dados_uenp[aba] = df

    print("\n" + "=" * 70)
    print(f"PLANILHA: {aba}")
    print("=" * 70)

    print("Dimensões:", df.shape)

    print("\nColunas:")
    for i, coluna in enumerate(df.columns, start=1):
        print(f"{i:02d}. {coluna}")

    print("\nPrimeiros registros:")
    display(df.head())

### 12.3 Caracterização da planilha de produção leiteira

A planilha `Produção` apresenta os registros de produção leiteira em formato horizontal, com os animais organizados nas linhas e as datas de controle distribuídas nas colunas.

Foram identificados dois blocos principais no mesmo arquivo: um referente aos controles realizados em 2026 e outro aos controles de 2025.

Além de valores numéricos de produção, a planilha contém representações textuais como `SECA` e `-`. Antes da transformação dos dados para o formato animal-data, esses registros serão caracterizados para distinguir produção efetivamente mensurada de situações sem produção registrada.

Nesta etapa nenhuma observação será excluída ou convertida.

In [ ]:
# ============================================================
# 12.3 CARACTERIZAÇÃO DA PLANILHA DE PRODUÇÃO
# ============================================================

producao_uenp_raw = dados_uenp["Produção"].copy()

# ------------------------------------------------------------
# IDENTIFICAÇÃO DAS COLUNAS DE DATA
# ------------------------------------------------------------

def identificar_colunas_data(df, ano):
    colunas_encontradas = []

    for coluna in df.columns:
        data_convertida = pd.to_datetime(
            coluna,
            errors="coerce"
        )

        if (
            pd.notna(data_convertida)
            and data_convertida.year == ano
        ):
            colunas_encontradas.append(coluna)

    return colunas_encontradas


colunas_datas_2026 = identificar_colunas_data(
    producao_uenp_raw,
    2026
)

colunas_datas_2025 = identificar_colunas_data(
    producao_uenp_raw,
    2025
)

print("ESTRUTURA DA PRODUÇÃO UENP")
print("-" * 60)

print("\nDatas de controle de 2026:")
for coluna in colunas_datas_2026:
    print(
        "-",
        pd.to_datetime(coluna).date()
    )

print("\nDatas de controle de 2025:")
for coluna in colunas_datas_2025:
    print(
        "-",
        pd.to_datetime(coluna).date()
    )

# ------------------------------------------------------------
# ANIMAIS EM CADA BLOCO
# ------------------------------------------------------------

animais_2026 = (
    producao_uenp_raw["Animal"]
    .dropna()
    .astype(str)
    .str.strip()
)

animais_2025 = (
    producao_uenp_raw["Animal.1"]
    .dropna()
    .astype(str)
    .str.strip()
)

print("\nANIMAIS")
print("-" * 60)

print(
    "Animais preenchidos no bloco 2026:",
    len(animais_2026)
)

print(
    "Animais distintos em 2026:",
    animais_2026.nunique()
)

print(
    "Animais preenchidos no bloco 2025:",
    len(animais_2025)
)

print(
    "Animais distintos em 2025:",
    animais_2025.nunique()
)

print(
    "Animais presentes nos dois anos:",
    len(
        set(animais_2026)
        & set(animais_2025)
    )
)

# ------------------------------------------------------------
# TIPOS DE VALORES DE PRODUÇÃO
# ------------------------------------------------------------

def caracterizar_valores_producao(df, colunas):

    # Transforma todas as células do bloco em uma única série,
    # preservando inclusive os valores ausentes
    valores = pd.Series(
        df[colunas]
        .to_numpy()
        .ravel()
    )

    texto = (
        valores
        .astype("string")
        .str.strip()
        .str.upper()
    )

    numericos = pd.to_numeric(
        valores,
        errors="coerce"
    )

    return pd.Series({
        "total_celulas": len(valores),

        "numericos":
            numericos.notna().sum(),

        "SECA":
            texto.eq("SECA").sum(),

        "-":
            texto.eq("-").sum(),

        "ausentes_reais":
            valores.isna().sum()
    })


print("\nTIPOS DE REGISTROS")
print("-" * 60)

resumo_2026 = caracterizar_valores_producao(
    producao_uenp_raw,
    colunas_datas_2026
)

resumo_2025 = caracterizar_valores_producao(
    producao_uenp_raw,
    colunas_datas_2025
)

resumo_producao_uenp = pd.DataFrame({
    "2026": resumo_2026,
    "2025": resumo_2025
})

display(resumo_producao_uenp)

# ------------------------------------------------------------
# COBERTURA POR DATA
# ------------------------------------------------------------

print("\nREGISTROS POR DATA — 2026")
print("-" * 60)

for coluna in colunas_datas_2026:

    print(
        pd.to_datetime(coluna).date(),
        "→",
        producao_uenp_raw[coluna]
        .notna()
        .sum(),
        "valores preenchidos"
    )

print("\nREGISTROS POR DATA — 2025")
print("-" * 60)

for coluna in colunas_datas_2025:

    print(
        pd.to_datetime(coluna).date(),
        "→",
        producao_uenp_raw[coluna]
        .notna()
        .sum(),
        "valores preenchidos"
    )

### 12.4 Transformação dos registros de produção para o formato animal-data

A planilha de produção leiteira está organizada em formato horizontal, com uma coluna para cada data de controle. Para permitir integração com as demais fontes e posterior utilização em Machine Learning, os registros serão transformados para o formato longitudinal, no qual cada linha representa uma combinação entre animal e data de controle.

Os blocos de 2025 e 2026 serão tratados separadamente, pois utilizam colunas distintas para identificação dos animais, e posteriormente serão concatenados.

Os valores numéricos serão interpretados como produção de leite medida. Os marcadores `SECA` e `-` serão preservados como situações distintas e não serão convertidos para zero, pois não representam necessariamente produção igual a zero.

Nenhum registro será excluído em função desses marcadores nesta etapa.

In [ ]:
# ============================================================
# 12.4 TRANSFORMAÇÃO DA PRODUÇÃO PARA FORMATO ANIMAL-DATA
# ============================================================

def transformar_producao_uenp(
    df,
    coluna_animal,
    colunas_datas,
    ano
):
    # Mantém somente linhas que possuem animal identificado
    bloco = df[
        [coluna_animal] + colunas_datas
    ].copy()

    bloco = bloco[
        bloco[coluna_animal].notna()
    ].copy()

    # Formato largo -> formato longo
    longo = bloco.melt(
        id_vars=[coluna_animal],
        value_vars=colunas_datas,
        var_name="data_prova",
        value_name="valor_original"
    )

    # Padronização do nome do animal
    longo["animal"] = (
        longo[coluna_animal]
        .astype(str)
        .str.strip()
    )

    # Data do controle
    longo["data_prova"] = pd.to_datetime(
        longo["data_prova"],
        errors="coerce"
    )

    # Tentativa de converter produção para número
    longo["kg_leite"] = pd.to_numeric(
        longo["valor_original"],
        errors="coerce"
    )

    # Classificação do tipo de registro
    valor_texto = (
        longo["valor_original"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    longo["situacao_producao"] = np.select(
        [
            longo["kg_leite"].notna(),
            valor_texto.eq("SECA"),
            valor_texto.eq("-"),
            longo["valor_original"].isna()
        ],
        [
            "mensurada",
            "seca",
            "sem_registro",
            "ausente"
        ],
        default="outro"
    )

    longo["ano_origem"] = ano

    return longo[
        [
            "animal",
            "data_prova",
            "kg_leite",
            "valor_original",
            "situacao_producao",
            "ano_origem"
        ]
    ]


# ------------------------------------------------------------
# BLOCO 2026
# ------------------------------------------------------------

producao_uenp_2026 = transformar_producao_uenp(
    producao_uenp_raw,
    "Animal",
    colunas_datas_2026,
    2026
)

# ------------------------------------------------------------
# BLOCO 2025
# ------------------------------------------------------------

producao_uenp_2025 = transformar_producao_uenp(
    producao_uenp_raw,
    "Animal.1",
    colunas_datas_2025,
    2025
)

# ------------------------------------------------------------
# CONSOLIDAÇÃO DOS DOIS ANOS
# ------------------------------------------------------------

producao_uenp = pd.concat(
    [
        producao_uenp_2025,
        producao_uenp_2026
    ],
    ignore_index=True
)

producao_uenp = producao_uenp.sort_values(
    ["animal", "data_prova"]
).reset_index(drop=True)

# ------------------------------------------------------------
# VALIDAÇÕES
# ------------------------------------------------------------

print("PRODUÇÃO UENP — FORMATO LONGITUDINAL")
print("-" * 60)

print("Observações:", len(producao_uenp))
print(
    "Animais distintos:",
    producao_uenp["animal"].nunique()
)

print(
    "Datas distintas:",
    producao_uenp["data_prova"].nunique()
)

print("\nSituações de produção:")

display(
    producao_uenp["situacao_producao"]
    .value_counts(dropna=False)
)

print("\nRegistros por ano:")

display(
    producao_uenp["ano_origem"]
    .value_counts()
    .sort_index()
)

print("\nDuplicidades animal + data:")

duplicados_producao_uenp = producao_uenp.duplicated(
    subset=["animal", "data_prova"]
).sum()

print(duplicados_producao_uenp)

print("\nPrimeiros registros:")

display(
    producao_uenp.head(15)
)

### 12.5 Correspondência entre os animais da produção e o cadastro geral

Após a transformação dos controles leiteiros para o formato longitudinal, será avaliada a correspondência entre os animais presentes na planilha de produção e o cadastro disponível em `Visão Geral`.

Essa verificação é necessária porque o cadastro contém informações potencialmente relevantes, como número de identificação, raça e data de nascimento.

Antes da integração das tabelas, serão avaliados:

- número de animais presentes em cada fonte;
- possíveis duplicidades no cadastro;
- correspondências entre os nomes dos animais;
- animais presentes na produção sem registro correspondente no cadastro.

Nesta etapa nenhuma observação da produção será removida.

In [ ]:
# ============================================================
# 12.5 CORRESPONDÊNCIA ENTRE PRODUÇÃO E CADASTRO
# ============================================================

visao_geral_uenp = dados_uenp["Visão Geral"].copy()

# ------------------------------------------------------------
# FUNÇÃO DE NORMALIZAÇÃO BÁSICA DOS NOMES
# ------------------------------------------------------------

def normalizar_nome_animal(valor):
    if pd.isna(valor):
        return pd.NA

    return (
        str(valor)
        .strip()
        .casefold()
    )


# Nome normalizado no cadastro
visao_geral_uenp["animal_normalizado"] = (
    visao_geral_uenp["Animal"]
    .apply(normalizar_nome_animal)
)

# Nome normalizado na produção
producao_uenp["animal_normalizado"] = (
    producao_uenp["animal"]
    .apply(normalizar_nome_animal)
)

# ------------------------------------------------------------
# CARACTERIZAÇÃO DAS DUAS FONTES
# ------------------------------------------------------------

animais_cadastro = (
    visao_geral_uenp[
        visao_geral_uenp["Animal"].notna()
    ][
        ["Número", "Animal", "animal_normalizado"]
    ]
    .copy()
)

animais_producao = (
    producao_uenp[
        ["animal", "animal_normalizado"]
    ]
    .drop_duplicates()
    .copy()
)

print("CORRESPONDÊNCIA PRODUÇÃO x CADASTRO")
print("-" * 60)

print(
    "Animais distintos na produção:",
    animais_producao["animal_normalizado"].nunique()
)

print(
    "Animais preenchidos no cadastro:",
    len(animais_cadastro)
)

print(
    "Animais distintos no cadastro:",
    animais_cadastro["animal_normalizado"].nunique()
)

# ------------------------------------------------------------
# DUPLICIDADES NO CADASTRO
# ------------------------------------------------------------

duplicados_cadastro = (
    animais_cadastro[
        animais_cadastro.duplicated(
            subset=["animal_normalizado"],
            keep=False
        )
    ]
    .sort_values("animal_normalizado")
)

print("\nNomes duplicados no cadastro:")
print(
    duplicados_cadastro["animal_normalizado"].nunique()
)

if len(duplicados_cadastro) > 0:
    display(duplicados_cadastro)

# ------------------------------------------------------------
# CORRESPONDÊNCIAS
# ------------------------------------------------------------

nomes_cadastro = set(
    animais_cadastro["animal_normalizado"].dropna()
)

nomes_producao = set(
    animais_producao["animal_normalizado"].dropna()
)

encontrados = nomes_producao & nomes_cadastro
nao_encontrados = nomes_producao - nomes_cadastro

print("\nRESULTADO DA CORRESPONDÊNCIA")
print("-" * 60)

print(
    "Animais da produção encontrados no cadastro:",
    len(encontrados)
)

print(
    "Animais da produção NÃO encontrados no cadastro:",
    len(nao_encontrados)
)

print(
    "Cobertura cadastral dos animais da produção:",
    round(
        len(encontrados)
        / len(nomes_producao)
        * 100,
        2
    ),
    "%"
)

# ------------------------------------------------------------
# LISTAGEM DOS NÃO ENCONTRADOS
# ------------------------------------------------------------

print("\nAnimais da produção sem correspondência no cadastro:")

display(
    animais_producao[
        animais_producao["animal_normalizado"]
        .isin(nao_encontrados)
    ]
    .sort_values("animal")
)


### 12.6 Identificação de linhas não correspondentes a animais

A comparação entre os animais presentes na produção e o cadastro geral revelou três nomes sem correspondência: `Cora`, `MÉDIA` e `TOTAL`.

Os registros `MÉDIA` e `TOTAL` correspondem a linhas de resumo da própria planilha de produção e não representam animais individuais. Sua presença no formato longitudinal poderia introduzir valores agregados como se fossem observações reais de produção, provocando distorções na base destinada à análise e ao Machine Learning.

O registro `Cora`, por outro lado, será preservado inicialmente, pois apresenta registros individuais de produção e pode representar um animal histórico que não integra mais o cadastro atual.

Antes da remoção das linhas de resumo, será verificado em quais blocos e datas elas aparecem e quais valores foram associados a esses registros.

In [ ]:
# ============================================================
# 12.6 INVESTIGAÇÃO DE LINHAS DE RESUMO DA PRODUÇÃO
# ============================================================

nomes_resumo = ["média", "total"]

linhas_resumo = producao_uenp[
    producao_uenp["animal_normalizado"].isin(nomes_resumo)
].copy()

print("LINHAS DE RESUMO IDENTIFICADAS")
print("-" * 60)

print("Observações:", len(linhas_resumo))

print(
    "Nomes encontrados:",
    linhas_resumo["animal"].unique()
)

print("\nQuantidade por nome:")

display(
    linhas_resumo["animal"]
    .value_counts()
)

print("\nQuantidade por ano:")

display(
    linhas_resumo.groupby(
        ["animal", "ano_origem"]
    ).size()
)

print("\nRegistros completos:")

display(
    linhas_resumo[
        [
            "animal",
            "data_prova",
            "kg_leite",
            "valor_original",
            "situacao_producao",
            "ano_origem"
        ]
    ].sort_values(
        ["animal", "data_prova"]
    )
)

print("\nRegistro de Cora para conferência:")

display(
    producao_uenp.loc[
        producao_uenp["animal_normalizado"] == "cora",
        [
            "animal",
            "data_prova",
            "kg_leite",
            "valor_original",
            "situacao_producao",
            "ano_origem"
        ]
    ]
)

### 12.7 Remoção das linhas agregadas da planilha de produção

A investigação dos registros sem correspondência cadastral confirmou que `MÉDIA` e `TOTAL` não representam animais, mas linhas de consolidação presentes na planilha original.

Esses registros geraram 22 observações artificiais no formato longitudinal, sendo 11 referentes à média da produção e 11 ao total produzido em cada data de controle. Como esses valores representam informações agregadas do rebanho, sua permanência na base individual poderia introduzir distorções na análise e no treinamento dos modelos.

Dessa forma, as linhas `MÉDIA` e `TOTAL` serão removidas da base longitudinal.

O animal `Cora` será preservado, mesmo sem correspondência no cadastro atual, pois possui registros individuais de produção referentes a 2025 e pode representar um animal histórico que não integra mais a relação atual da propriedade.

In [ ]:
# ============================================================
# 12.7 REMOÇÃO DAS LINHAS AGREGADAS DA PRODUÇÃO
# ============================================================

producao_uenp_limpa = producao_uenp.loc[
    ~producao_uenp["animal_normalizado"].isin(
        ["média", "total"]
    )
].copy()

producao_uenp_limpa = (
    producao_uenp_limpa
    .sort_values(["animal", "data_prova"])
    .reset_index(drop=True)
)

print("PRODUÇÃO UENP APÓS REMOÇÃO DAS LINHAS AGREGADAS")
print("-" * 60)

print(
    "Observações:",
    len(producao_uenp_limpa)
)

print(
    "Animais distintos:",
    producao_uenp_limpa["animal"].nunique()
)

print(
    "Datas distintas:",
    producao_uenp_limpa["data_prova"].nunique()
)

print("\nRegistros por ano:")

display(
    producao_uenp_limpa["ano_origem"]
    .value_counts()
    .sort_index()
)

print("\nSituações de produção:")

display(
    producao_uenp_limpa["situacao_producao"]
    .value_counts(dropna=False)
)

print("\nDuplicidades animal + data:")

duplicados_uenp_limpa = (
    producao_uenp_limpa
    .duplicated(
        subset=["animal", "data_prova"]
    )
    .sum()
)

print(duplicados_uenp_limpa)

print("\nAnimais distintos:")

display(
    pd.Series(
        sorted(
            producao_uenp_limpa["animal"]
            .dropna()
            .unique()
        ),
        name="animal"
    )
)

### 12.8 Integração da produção com o cadastro geral

Após a remoção das linhas agregadas, a base longitudinal de produção contém 197 observações referentes a 20 animais reais.

A correspondência com a planilha `Visão Geral` mostrou que 19 dos 20 animais possuem cadastro atual, enquanto `Cora` apresenta registros históricos de produção, mas não está presente na relação cadastral atual.

Nesta etapa serão incorporadas apenas informações cadastrais de natureza estável, como número de identificação, raça e data de nascimento.

As variáveis `Parto`, `DEL` e `Nº de Partos` existentes em `Visão Geral` não serão associadas diretamente aos controles históricos, pois representam um retrato cadastral atual e não necessariamente a situação do animal na data de cada controle leiteiro.

A ausência de cadastro para `Cora` não implicará exclusão de seus registros.

In [ ]:
# ============================================================
# 12.8 INTEGRAÇÃO DA PRODUÇÃO COM O CADASTRO GERAL
# ============================================================

cadastro_uenp = visao_geral_uenp[
    [
        "Número",
        "Animal",
        "Raça",
        "Nascimento",
        "animal_normalizado"
    ]
].copy()

# ------------------------------------------------------------
# VALIDAÇÃO DO CADASTRO
# ------------------------------------------------------------

print("VALIDAÇÃO DO CADASTRO")
print("-" * 60)

print(
    "Animais cadastrados:",
    cadastro_uenp["animal_normalizado"].notna().sum()
)

print(
    "Nomes duplicados:",
    cadastro_uenp["animal_normalizado"]
    .duplicated()
    .sum()
)

print(
    "Números de identificação duplicados:",
    cadastro_uenp["Número"]
    .dropna()
    .duplicated()
    .sum()
)

# ------------------------------------------------------------
# MERGE
# ------------------------------------------------------------

producao_uenp_integrada = producao_uenp_limpa.merge(
    cadastro_uenp,
    how="left",
    on="animal_normalizado",
    validate="many_to_one"
)

# ------------------------------------------------------------
# VALIDAÇÃO DA INTEGRAÇÃO
# ------------------------------------------------------------

print("\nINTEGRAÇÃO PRODUÇÃO x CADASTRO")
print("-" * 60)

print(
    "Observações antes do merge:",
    len(producao_uenp_limpa)
)

print(
    "Observações após o merge:",
    len(producao_uenp_integrada)
)

print(
    "Observações com cadastro:",
    producao_uenp_integrada["Número"]
    .notna()
    .sum()
)

print(
    "Observações sem cadastro:",
    producao_uenp_integrada["Número"]
    .isna()
    .sum()
)

print(
    "Cobertura cadastral das observações:",
    round(
        producao_uenp_integrada["Número"]
        .notna()
        .mean()
        * 100,
        2
    ),
    "%"
)

print(
    "Animais distintos com cadastro:",
    producao_uenp_integrada.loc[
        producao_uenp_integrada["Número"].notna(),
        "animal"
    ].nunique()
)

print("\nAnimais sem cadastro:")

display(
    producao_uenp_integrada.loc[
        producao_uenp_integrada["Número"].isna(),
        ["animal", "data_prova", "kg_leite",
         "situacao_producao", "ano_origem"]
    ]
)

print("\nPrimeiros registros integrados:")

display(
    producao_uenp_integrada[
        [
            "Número",
            "animal",
            "data_prova",
            "kg_leite",
            "situacao_producao",
            "Raça",
            "Nascimento"
        ]
    ].head(15)
)

### 12.9 Avaliação das variáveis cadastrais e cálculo da idade

A integração entre os controles leiteiros e o cadastro geral permitiu associar informações cadastrais a 191 das 197 observações (96,95%).

Nesta etapa serão avaliadas as variáveis `Raça` e `Nascimento`. A data de nascimento será utilizada para calcular a idade do animal na data de cada controle leiteiro, evitando o uso direto da data absoluta como atributo preditor.

Também serão verificadas possíveis inconsistências, como datas de nascimento posteriores ao controle ou idades biologicamente incompatíveis com animais em produção.

Os seis registros históricos de `Cora`, que não possuem cadastro atual, permanecerão na base e apresentarão essas informações como ausentes.

In [ ]:
# ============================================================
# 12.9 AVALIAÇÃO DE RAÇA, NASCIMENTO E IDADE
# ============================================================

print("VARIÁVEIS CADASTRAIS — UENP")
print("-" * 60)

# ------------------------------------------------------------
# 1. RAÇA
# ------------------------------------------------------------

print("\n1. RAÇA")
print("-" * 30)

print(
    "Observações com raça:",
    producao_uenp_integrada["Raça"].notna().sum()
)

print(
    "Categorias distintas:",
    producao_uenp_integrada["Raça"].nunique(dropna=True)
)

print("\nDistribuição das raças:")

display(
    producao_uenp_integrada["Raça"]
    .value_counts(dropna=False)
)

# ------------------------------------------------------------
# 2. DATA DE NASCIMENTO
# ------------------------------------------------------------

print("\n2. DATA DE NASCIMENTO")
print("-" * 30)

nascimento_uenp = pd.to_datetime(
    producao_uenp_integrada["Nascimento"],
    errors="coerce"
)

print(
    "Nascimento preenchido:",
    producao_uenp_integrada["Nascimento"]
    .notna()
    .sum()
)

print(
    "Datas válidas:",
    nascimento_uenp.notna().sum()
)

print(
    "Valores preenchidos mas não interpretáveis:",
    (
        producao_uenp_integrada["Nascimento"].notna()
        & nascimento_uenp.isna()
    ).sum()
)

# ------------------------------------------------------------
# 3. CÁLCULO PRELIMINAR DA IDADE
# ------------------------------------------------------------

idade_uenp_teste = (
    (
        producao_uenp_integrada["data_prova"]
        - nascimento_uenp
    ).dt.days
    / 365.25
)

print("\n3. IDADE NA DATA DO CONTROLE")
print("-" * 30)

display(
    idade_uenp_teste.describe()
)

print(
    "Idades negativas:",
    (idade_uenp_teste < 0).sum()
)

print(
    "Idades inferiores a 1 ano:",
    (idade_uenp_teste < 1).sum()
)

print(
    "Idades inferiores a 1,5 ano:",
    (idade_uenp_teste < 1.5).sum()
)

# ------------------------------------------------------------
# 4. CASOS POTENCIALMENTE PROBLEMÁTICOS
# ------------------------------------------------------------

casos_idade_baixa_uenp = producao_uenp_integrada.loc[
    idade_uenp_teste < 1.5,
    [
        "Número",
        "animal",
        "data_prova",
        "kg_leite",
        "situacao_producao",
        "Raça",
        "Nascimento"
    ]
].copy()

casos_idade_baixa_uenp["idade_anos_calculada"] = (
    idade_uenp_teste.loc[
        casos_idade_baixa_uenp.index
    ]
)

print("\nCasos com idade inferior a 1,5 ano:")

display(
    casos_idade_baixa_uenp.sort_values(
        ["idade_anos_calculada", "animal"]
    )
)

### 12.10 Padronização da raça e criação da idade

A avaliação cadastral mostrou que todas as 191 observações associadas ao cadastro possuem datas de nascimento válidas e não foram identificadas idades negativas.

As categorias raciais apresentaram diferenças de formatação relacionadas a espaços em branco, fazendo com que registros de `Jersolanda` fossem contabilizados como categorias distintas. Essas representações serão padronizadas sem alterar o significado da raça.

A idade será calculada em anos a partir da diferença entre a data do controle leiteiro e a data de nascimento do animal.

As observações históricas de `Cora`, que não possuem correspondência no cadastro atual, permanecerão com raça e idade ausentes.

As duas observações com idade inferior a 1,5 ano correspondem a registros sem produção mensurada e, portanto, não serão consideradas inconsistências nesta etapa.

In [ ]:
# ============================================================
# 12.10 PADRONIZAÇÃO DA RAÇA E CRIAÇÃO DA IDADE
# ============================================================

base_uenp_preparada = producao_uenp_integrada.copy()

# ------------------------------------------------------------
# 1. PADRONIZAÇÃO DA RAÇA
# ------------------------------------------------------------

base_uenp_preparada["raca_padronizada"] = (
    base_uenp_preparada["Raça"]
    .astype("string")
    .str.strip()
)

# ------------------------------------------------------------
# 2. CONVERSÃO DA DATA DE NASCIMENTO
# ------------------------------------------------------------

base_uenp_preparada["data_nascimento_convertida"] = pd.to_datetime(
    base_uenp_preparada["Nascimento"],
    errors="coerce"
)

# ------------------------------------------------------------
# 3. CÁLCULO DA IDADE
# ------------------------------------------------------------

base_uenp_preparada["idade_anos"] = (
    (
        base_uenp_preparada["data_prova"]
        - base_uenp_preparada["data_nascimento_convertida"]
    ).dt.days
    / 365.25
)

# ------------------------------------------------------------
# 4. VERIFICAÇÕES
# ------------------------------------------------------------

print("VARIÁVEIS CADASTRAIS PADRONIZADAS — UENP")
print("-" * 60)

print("\nRaças após padronização:")

display(
    base_uenp_preparada["raca_padronizada"]
    .value_counts(dropna=False)
)

print("\nCobertura da idade:")

print(
    "Idades preenchidas:",
    base_uenp_preparada["idade_anos"].notna().sum()
)

print(
    "Idades ausentes:",
    base_uenp_preparada["idade_anos"].isna().sum()
)

print("\nResumo da idade:")

display(
    base_uenp_preparada["idade_anos"].describe()
)

# ------------------------------------------------------------
# 5. IDADE APENAS ENTRE CONTROLES COM PRODUÇÃO MENSURADA
# ------------------------------------------------------------

idades_com_producao = base_uenp_preparada.loc[
    base_uenp_preparada["situacao_producao"] == "mensurada",
    "idade_anos"
]

print("\nIdade entre observações com produção mensurada:")

display(
    idades_com_producao.describe()
)

print(
    "Produções mensuradas com idade < 1,5 ano:",
    (idades_com_producao < 1.5).sum()
)

### 12.11 Estruturação dos registros de peso corporal

A planilha `Pesagem 2026` contém registros de peso corporal organizados em formato horizontal, com os animais dispostos nas linhas e diferentes datas de pesagem nas colunas.

Para permitir a posterior associação temporal com os controles leiteiros, os dados serão transformados para o formato longitudinal, no qual cada linha representa uma combinação entre animal, data de pesagem e peso corporal.

Nesta etapa também serão avaliados o número de animais, a quantidade de pesagens disponíveis, possíveis valores ausentes, duplicidades e registros não numéricos.

Nenhuma pesagem será associada à produção antes da avaliação da proximidade temporal entre as duas fontes.

In [ ]:
# ============================================================
# 12.11 ESTRUTURAÇÃO DAS PESAGENS DE 2026
# ============================================================

pesagem_uenp_raw = dados_uenp["Pesagem 2026"].copy()

# ------------------------------------------------------------
# IDENTIFICAÇÃO DAS COLUNAS DE DATA
# ------------------------------------------------------------

colunas_datas_pesagem = []

for coluna in pesagem_uenp_raw.columns:

    data_convertida = pd.to_datetime(
        coluna,
        errors="coerce"
    )

    if (
        pd.notna(data_convertida)
        and data_convertida.year == 2026
    ):
        colunas_datas_pesagem.append(coluna)


print("PESAGEM UENP — 2026")
print("-" * 60)

print("\nDatas de pesagem encontradas:")

for coluna in colunas_datas_pesagem:
    print(
        "-",
        pd.to_datetime(coluna).date()
    )

# ------------------------------------------------------------
# TRANSFORMAÇÃO PARA FORMATO LONGITUDINAL
# ------------------------------------------------------------

pesagem_uenp = (
    pesagem_uenp_raw[
        ["Número", "Animal"] + colunas_datas_pesagem
    ]
    .copy()
)

# Mantém somente linhas com animal identificado
pesagem_uenp = pesagem_uenp[
    pesagem_uenp["Animal"].notna()
].copy()

pesagem_uenp = pesagem_uenp.melt(
    id_vars=["Número", "Animal"],
    value_vars=colunas_datas_pesagem,
    var_name="data_pesagem",
    value_name="peso_original"
)

# Padronização do nome
pesagem_uenp["animal_normalizado"] = (
    pesagem_uenp["Animal"]
    .apply(normalizar_nome_animal)
)

# Conversão da data
pesagem_uenp["data_pesagem"] = pd.to_datetime(
    pesagem_uenp["data_pesagem"],
    errors="coerce"
)

# Conversão do peso
pesagem_uenp["peso_kg"] = pd.to_numeric(
    pesagem_uenp["peso_original"],
    errors="coerce"
)

# ------------------------------------------------------------
# CARACTERIZAÇÃO
# ------------------------------------------------------------

print("\nESTRUTURA LONGITUDINAL")
print("-" * 60)

print(
    "Registros possíveis:",
    len(pesagem_uenp)
)

print(
    "Animais distintos:",
    pesagem_uenp["animal_normalizado"].nunique()
)

print(
    "Pesagens preenchidas:",
    pesagem_uenp["peso_kg"].notna().sum()
)

print(
    "Pesagens ausentes/não numéricas:",
    pesagem_uenp["peso_kg"].isna().sum()
)

print(
    "Datas distintas:",
    pesagem_uenp["data_pesagem"].nunique()
)

# ------------------------------------------------------------
# DUPLICIDADES
# ------------------------------------------------------------

duplicados_pesagem = pesagem_uenp.duplicated(
    subset=["animal_normalizado", "data_pesagem"]
).sum()

print(
    "Duplicidades animal + data:",
    duplicados_pesagem
)

# ------------------------------------------------------------
# DISTRIBUIÇÃO DOS PESOS
# ------------------------------------------------------------

print("\nResumo dos pesos:")

display(
    pesagem_uenp["peso_kg"].describe()
)

print("\nQuantidade de pesos válidos por data:")

display(
    pesagem_uenp.groupby(
        "data_pesagem"
    )["peso_kg"]
    .count()
)

# ------------------------------------------------------------
# POSSÍVEIS LINHAS NÃO INDIVIDUAIS
# ------------------------------------------------------------

print("\nAnimais/nomes presentes na planilha:")

display(
    pesagem_uenp[
        ["Número", "Animal", "animal_normalizado"]
    ]
    .drop_duplicates()
    .sort_values("Animal")
)

### 12.12 Correspondência entre os controles leiteiros e os registros de peso

A planilha de pesagem de 2026 contém 40 animais e 273 valores de peso corporal válidos distribuídos em oito datas.

Entretanto, nem todos os animais presentes nessa planilha participam dos controles leiteiros utilizados na base de produção. Antes de realizar qualquer associação temporal entre peso e produção, será verificada a correspondência entre os animais das duas fontes.

Como ambas as planilhas possuem um número de identificação individual, esse campo será priorizado para a integração, reduzindo a dependência de possíveis diferenças na grafia dos nomes.

Nesta etapa será avaliado apenas se os animais com controles de produção em 2026 possuem registros na planilha de pesagem. A associação entre cada controle e a pesagem temporalmente mais próxima será realizada posteriormente.

In [ ]:
# ============================================================
# 12.12 CORRESPONDÊNCIA ENTRE PRODUÇÃO E PESAGEM
# ============================================================

# Controles de produção de 2026
producao_uenp_2026_integrada = base_uenp_preparada.loc[
    base_uenp_preparada["ano_origem"] == 2026
].copy()

# Animais distintos da produção de 2026
animais_producao_2026 = (
    producao_uenp_2026_integrada[
        ["Número", "animal"]
    ]
    .drop_duplicates()
    .sort_values("Número")
)

# Animais distintos existentes na planilha de pesagem
animais_pesagem_2026 = (
    pesagem_uenp[
        ["Número", "Animal"]
    ]
    .drop_duplicates()
    .sort_values("Número")
)

print("CORRESPONDÊNCIA PRODUÇÃO x PESAGEM — 2026")
print("-" * 60)

print(
    "Animais distintos na produção de 2026:",
    animais_producao_2026["Número"].nunique()
)

print(
    "Animais distintos na pesagem de 2026:",
    animais_pesagem_2026["Número"].nunique()
)

# ------------------------------------------------------------
# IDENTIFICADORES PRESENTES NAS DUAS FONTES
# ------------------------------------------------------------

ids_producao = set(
    animais_producao_2026["Número"]
    .dropna()
)

ids_pesagem = set(
    animais_pesagem_2026["Número"]
    .dropna()
)

ids_em_comum = ids_producao & ids_pesagem
ids_sem_pesagem = ids_producao - ids_pesagem

print(
    "\nAnimais da produção encontrados na pesagem:",
    len(ids_em_comum)
)

print(
    "Animais da produção sem registro na planilha de pesagem:",
    len(ids_sem_pesagem)
)

if len(ids_producao) > 0:
    print(
        "Cobertura dos animais da produção:",
        round(
            len(ids_em_comum)
            / len(ids_producao)
            * 100,
            2
        ),
        "%"
    )

# ------------------------------------------------------------
# ANIMAIS DA PRODUÇÃO SEM PESAGEM
# ------------------------------------------------------------

print("\nAnimais da produção sem correspondência na pesagem:")

display(
    animais_producao_2026.loc[
        animais_producao_2026["Número"]
        .isin(ids_sem_pesagem)
    ]
)

# ------------------------------------------------------------
# QUANTIDADE DE PESAGENS VÁLIDAS POR ANIMAL DA PRODUÇÃO
# ------------------------------------------------------------

pesagens_validas_por_animal = (
    pesagem_uenp.loc[
        pesagem_uenp["Número"].isin(ids_producao)
    ]
    .groupby(["Número", "Animal"])["peso_kg"]
    .count()
    .rename("pesagens_validas")
    .reset_index()
    .sort_values(
        ["pesagens_validas", "Número"],
        ascending=[True, True]
    )
)

print("\nQuantidade de pesagens válidas por animal da produção:")

display(pesagens_validas_por_animal)

### 12.13 Associação temporal do peso corporal aos controles leiteiros

Todos os 19 animais presentes nos controles leiteiros de 2026 possuem registros completos nas oito datas da planilha de pesagem.

Para associar o peso corporal a cada observação de produção, será utilizada a pesagem mais recente disponível na mesma data ou anteriormente ao controle leiteiro.

Não serão utilizadas pesagens realizadas após a data do controle, mesmo quando temporalmente mais próximas, pois isso introduziria informação futura em uma variável potencialmente utilizada para previsão.

Além do valor do peso, será calculado o intervalo em dias entre a pesagem utilizada e o respectivo controle. Essa diferença permitirá avaliar a atualidade da informação antes de decidir pela utilização do peso corporal como atributo do modelo.

In [ ]:
# ============================================================
# 12.13 ASSOCIAÇÃO TEMPORAL DO PESO CORPORAL
# ============================================================

# Mantém somente pesagens válidas
pesagens_validas_uenp = pesagem_uenp.loc[
    pesagem_uenp["peso_kg"].notna()
].copy()

# Controles de produção de 2026
controles_2026_peso = (
    base_uenp_preparada.loc[
        base_uenp_preparada["ano_origem"] == 2026
    ]
    .copy()
)

# ------------------------------------------------------------
# CRUZAMENTO POR ANIMAL
# ------------------------------------------------------------

candidatos_peso = controles_2026_peso.merge(
    pesagens_validas_uenp[
        [
            "Número",
            "data_pesagem",
            "peso_kg"
        ]
    ],
    how="left",
    on="Número"
)

# Mantém somente pesagens realizadas na mesma data
# ou antes do controle leiteiro
candidatos_peso = candidatos_peso.loc[
    candidatos_peso["data_pesagem"]
    <= candidatos_peso["data_prova"]
].copy()

# Intervalo entre pesagem e controle
candidatos_peso["dias_desde_pesagem"] = (
    candidatos_peso["data_prova"]
    - candidatos_peso["data_pesagem"]
).dt.days

# ------------------------------------------------------------
# SELEÇÃO DA PESAGEM MAIS RECENTE
# ------------------------------------------------------------

peso_associado_uenp = (
    candidatos_peso
    .sort_values(
        [
            "Número",
            "data_prova",
            "dias_desde_pesagem"
        ]
    )
    .drop_duplicates(
        subset=["Número", "data_prova"],
        keep="first"
    )
    [
        [
            "Número",
            "data_prova",
            "data_pesagem",
            "peso_kg",
            "dias_desde_pesagem"
        ]
    ]
    .copy()
)

# ------------------------------------------------------------
# VALIDAÇÃO
# ------------------------------------------------------------

print("ASSOCIAÇÃO TEMPORAL DO PESO — UENP")
print("-" * 60)

print(
    "Controles de produção em 2026:",
    len(controles_2026_peso)
)

print(
    "Controles com peso anterior disponível:",
    len(peso_associado_uenp)
)

print(
    "Cobertura:",
    round(
        len(peso_associado_uenp)
        / len(controles_2026_peso)
        * 100,
        2
    ),
    "%"
)

print("\nIntervalo entre pesagem e controle:")

display(
    peso_associado_uenp["dias_desde_pesagem"]
    .describe()
)

print("\nDistribuição dos intervalos:")

display(
    peso_associado_uenp[
        "dias_desde_pesagem"
    ]
    .value_counts()
    .sort_index()
)

print("\nAssociações por data de controle:")

display(
    peso_associado_uenp.groupby(
        "data_prova"
    ).agg(
        controles=("Número", "size"),
        intervalo_min=("dias_desde_pesagem", "min"),
        intervalo_max=("dias_desde_pesagem", "max"),
        intervalo_medio=("dias_desde_pesagem", "mean")
    )
)

print("\nExemplos de associação:")

display(
    peso_associado_uenp.head(20)
)

### 12.14 Incorporação do peso corporal à base integrada

A associação temporal das pesagens aos controles leiteiros de 2026 apresentou cobertura de 100% das 95 observações.

Para cada controle foi utilizada exclusivamente a pesagem mais recente disponível na mesma data ou anteriormente à mensuração da produção. O intervalo entre a pesagem utilizada e o controle variou entre 0 e 21 dias, evitando a utilização de informações futuras.

Os valores de peso e o intervalo desde a pesagem serão incorporados à base integrada da UENP.

Como a planilha de pesagem disponível corresponde apenas ao ano de 2026, os controles referentes a 2025 permanecerão sem informação de peso. Dessa forma, o peso corporal será mantido como variável candidata, mas sua utilização definitiva dependerá da estratégia de harmonização e modelagem adotada posteriormente.

In [ ]:
# ============================================================
# 12.14 INCORPORAÇÃO DO PESO CORPORAL
# ============================================================

base_uenp_preparada = base_uenp_preparada.merge(
    peso_associado_uenp,
    how="left",
    on=["Número", "data_prova"],
    validate="one_to_one"
)

print("INCORPORAÇÃO DO PESO — UENP")
print("-" * 60)

print(
    "Observações após o merge:",
    len(base_uenp_preparada)
)

print(
    "Pesos associados:",
    base_uenp_preparada["peso_kg"].notna().sum()
)

print(
    "Pesos ausentes:",
    base_uenp_preparada["peso_kg"].isna().sum()
)

print(
    "Cobertura geral do peso:",
    round(
        base_uenp_preparada["peso_kg"]
        .notna()
        .mean()
        * 100,
        2
    ),
    "%"
)

# ------------------------------------------------------------
# COBERTURA POR ANO
# ------------------------------------------------------------

print("\nCobertura do peso por ano:")

cobertura_peso_ano = (
    base_uenp_preparada
    .groupby("ano_origem")
    .agg(
        observacoes=("animal", "size"),
        com_peso=("peso_kg", lambda x: x.notna().sum())
    )
)

cobertura_peso_ano["cobertura_%"] = (
    cobertura_peso_ano["com_peso"]
    / cobertura_peso_ano["observacoes"]
    * 100
).round(2)

display(cobertura_peso_ano)

# ------------------------------------------------------------
# COBERTURA ENTRE PRODUÇÕES MENSURADAS
# ------------------------------------------------------------

producao_mensurada_uenp = base_uenp_preparada.loc[
    base_uenp_preparada["situacao_producao"] == "mensurada"
]

print("\nPRODUÇÕES MENSURADAS")
print("-" * 60)

print(
    "Total:",
    len(producao_mensurada_uenp)
)

print(
    "Com peso:",
    producao_mensurada_uenp["peso_kg"]
    .notna()
    .sum()
)

print(
    "Sem peso:",
    producao_mensurada_uenp["peso_kg"]
    .isna()
    .sum()
)

print(
    "Cobertura do peso entre produções mensuradas:",
    round(
        producao_mensurada_uenp["peso_kg"]
        .notna()
        .mean()
        * 100,
        2
    ),
    "%"
)

print("\nResumo dos pesos associados:")

display(
    base_uenp_preparada["peso_kg"]
    .describe()
)

### 12.15 Validação temporal das informações de parto e DEL

A planilha `Visão Geral` apresenta, para cada animal, uma data de parto e um valor de DEL (dias em lactação).

Entretanto, essas informações representam um retrato cadastral da propriedade no momento de atualização da planilha e não podem ser atribuídas diretamente aos controles leiteiros históricos.

Antes de utilizar essas variáveis, será avaliada sua consistência interna. Para isso, será calculada a data de referência implícita correspondente à soma entre a data de parto e o DEL registrado.

Também será verificado em quantas observações de produção a data do controle ocorre após o parto atualmente registrado. Apenas nesses casos será possível considerar, posteriormente, a derivação temporal do DEL a partir dessa data de parto.

Nenhum valor de DEL histórico será criado nesta etapa.

In [ ]:
# ============================================================
# 12.15 VALIDAÇÃO TEMPORAL DE PARTO E DEL
# ============================================================

visao_geral_temporal = visao_geral_uenp[
    [
        "Número",
        "Animal",
        "Parto",
        "DEL",
        "Nº de Partos"
    ]
].copy()

# ------------------------------------------------------------
# CONVERSÕES
# ------------------------------------------------------------

visao_geral_temporal["parto_convertido"] = pd.to_datetime(
    visao_geral_temporal["Parto"],
    errors="coerce"
)

visao_geral_temporal["del_convertido"] = pd.to_numeric(
    visao_geral_temporal["DEL"],
    errors="coerce"
)

# Data de referência implícita:
# parto + DEL = data em que aquele DEL seria verdadeiro
visao_geral_temporal["data_referencia_del"] = (
    visao_geral_temporal["parto_convertido"]
    + pd.to_timedelta(
        visao_geral_temporal["del_convertido"],
        unit="D"
    )
)

print("VALIDAÇÃO DE PARTO E DEL — VISÃO GERAL")
print("-" * 60)

print(
    "Animais:",
    len(visao_geral_temporal)
)

print(
    "Partos válidos:",
    visao_geral_temporal["parto_convertido"]
    .notna()
    .sum()
)

print(
    "DEL válidos:",
    visao_geral_temporal["del_convertido"]
    .notna()
    .sum()
)

print("\nDistribuição do DEL:")

display(
    visao_geral_temporal["del_convertido"]
    .describe()
)

print("\nDatas de referência implícitas (Parto + DEL):")

display(
    visao_geral_temporal["data_referencia_del"]
    .value_counts()
    .sort_index()
)

print("\nIntervalo das datas de referência:")

print(
    "Mínima:",
    visao_geral_temporal["data_referencia_del"].min()
)

print(
    "Máxima:",
    visao_geral_temporal["data_referencia_del"].max()
)

# ------------------------------------------------------------
# RELAÇÃO COM OS CONTROLES DE PRODUÇÃO
# ------------------------------------------------------------

parto_para_producao = visao_geral_temporal[
    [
        "Número",
        "parto_convertido",
        "del_convertido",
        "Nº de Partos"
    ]
].copy()

analise_parto_controles = base_uenp_preparada.merge(
    parto_para_producao,
    how="left",
    on="Número",
    validate="many_to_one"
)

analise_parto_controles["controle_apos_parto_atual"] = (
    analise_parto_controles["data_prova"]
    >= analise_parto_controles["parto_convertido"]
)

analise_parto_controles["del_calculavel"] = (
    analise_parto_controles["data_prova"]
    - analise_parto_controles["parto_convertido"]
).dt.days

print("\nRELAÇÃO ENTRE PARTO ATUAL E CONTROLES")
print("-" * 60)

print(
    "Controles com parto cadastrado:",
    analise_parto_controles["parto_convertido"]
    .notna()
    .sum()
)

print(
    "Controles na mesma data ou após o parto cadastrado:",
    (
        analise_parto_controles["controle_apos_parto_atual"]
        == True
    ).sum()
)

print(
    "Controles anteriores ao parto cadastrado:",
    (
        (
            analise_parto_controles["parto_convertido"].notna()
        )
        & (
            analise_parto_controles["data_prova"]
            < analise_parto_controles["parto_convertido"]
        )
    ).sum()
)

print("\nDEL calculável nos controles posteriores ao parto:")

display(
    analise_parto_controles.loc[
        analise_parto_controles["controle_apos_parto_atual"] == True,
        "del_calculavel"
    ].describe()
)

print("\nExemplos:")

display(
    analise_parto_controles[
        [
            "Número",
            "animal",
            "data_prova",
            "kg_leite",
            "situacao_producao",
            "parto_convertido",
            "del_convertido",
            "del_calculavel",
            "Nº de Partos"
        ]
    ]
    .sort_values(
        ["Número", "data_prova"]
    )
    .head(40)
)

### 12.16 Avaliação do DEL derivado a partir da data de parto

A análise das informações da planilha `Visão Geral` demonstrou que os valores de DEL cadastrados correspondem a uma data de referência comum em 23 de setembro de 2026. Dessa forma, esses valores representam a situação atual dos animais e não podem ser atribuídos diretamente aos controles históricos.

Entretanto, a data de parto atualmente registrada pode ser utilizada para calcular os dias em lactação dos controles realizados após esse parto, uma vez que essas observações pertencem à lactação mais recente registrada para o animal.

Para os controles anteriores ao parto cadastrado, o DEL não será estimado, pois esses registros pertencem a lactações anteriores cujas datas de início não estão disponíveis nessa fonte.

Nesta etapa será avaliada a cobertura do DEL calculável, especialmente entre as observações que possuem produção de leite mensurada.

In [ ]:
# ============================================================
# 12.16 COBERTURA E VALIDADE DO DEL DERIVADO
# ============================================================

analise_del_uenp = analise_parto_controles.copy()

# ------------------------------------------------------------
# DEL VÁLIDO SOMENTE PARA CONTROLES NA MESMA DATA OU APÓS
# O PARTO ATUALMENTE CADASTRADO
# ------------------------------------------------------------

analise_del_uenp["del_derivado"] = analise_del_uenp[
    "del_calculavel"
].where(
    analise_del_uenp["controle_apos_parto_atual"] == True
)

print("AVALIAÇÃO DO DEL DERIVADO — UENP")
print("-" * 60)

print(
    "Observações totais:",
    len(analise_del_uenp)
)

print(
    "Observações com DEL derivável:",
    analise_del_uenp["del_derivado"]
    .notna()
    .sum()
)

print(
    "Cobertura geral:",
    round(
        analise_del_uenp["del_derivado"]
        .notna()
        .mean()
        * 100,
        2
    ),
    "%"
)

# ------------------------------------------------------------
# SITUAÇÃO PRODUTIVA
# ------------------------------------------------------------

print("\nCobertura por situação de produção:")

cobertura_del_situacao = (
    analise_del_uenp
    .groupby("situacao_producao")
    .agg(
        observacoes=("animal", "size"),
        com_del=("del_derivado", lambda x: x.notna().sum())
    )
)

cobertura_del_situacao["cobertura_%"] = (
    cobertura_del_situacao["com_del"]
    / cobertura_del_situacao["observacoes"]
    * 100
).round(2)

display(cobertura_del_situacao)

# ------------------------------------------------------------
# SOMENTE PRODUÇÕES MENSURADAS
# ------------------------------------------------------------

mensuradas_del_uenp = analise_del_uenp.loc[
    analise_del_uenp["situacao_producao"] == "mensurada"
].copy()

print("\nPRODUÇÕES MENSURADAS")
print("-" * 60)

print(
    "Produções mensuradas:",
    len(mensuradas_del_uenp)
)

print(
    "Com DEL derivado:",
    mensuradas_del_uenp["del_derivado"]
    .notna()
    .sum()
)

print(
    "Sem DEL derivado:",
    mensuradas_del_uenp["del_derivado"]
    .isna()
    .sum()
)

print(
    "Cobertura do DEL entre produções mensuradas:",
    round(
        mensuradas_del_uenp["del_derivado"]
        .notna()
        .mean()
        * 100,
        2
    ),
    "%"
)

# ------------------------------------------------------------
# DISTRIBUIÇÃO DO DEL NAS PRODUÇÕES MENSURADAS
# ------------------------------------------------------------

print("\nDistribuição do DEL entre produções mensuradas:")

display(
    mensuradas_del_uenp["del_derivado"]
    .describe()
)

print(
    "DEL igual a zero:",
    (mensuradas_del_uenp["del_derivado"] == 0).sum()
)

print(
    "DEL negativo:",
    (mensuradas_del_uenp["del_derivado"] < 0).sum()
)

print(
    "DEL acima de 305 dias:",
    (mensuradas_del_uenp["del_derivado"] > 305).sum()
)

print(
    "DEL acima de 365 dias:",
    (mensuradas_del_uenp["del_derivado"] > 365).sum()
)

# ------------------------------------------------------------
# REGISTROS COM DEL MAIS ALTO
# ------------------------------------------------------------

print("\nProduções mensuradas com DEL acima de 305 dias:")

display(
    mensuradas_del_uenp.loc[
        mensuradas_del_uenp["del_derivado"] > 305,
        [
            "Número",
            "animal",
            "data_prova",
            "kg_leite",
            "parto_convertido",
            "del_derivado"
        ]
    ]
    .sort_values("del_derivado")
)

### 12.17 Incorporação do DEL derivado à base da UENP

A data de parto atualmente registrada permitiu calcular os dias em lactação para 90 das 197 observações da base.

Entre as 134 observações com produção mensurada, 85 apresentaram DEL derivável, correspondendo a uma cobertura de 63,43%.

Os valores foram calculados apenas para controles realizados na mesma data ou após o parto atualmente cadastrado. Registros anteriores ao parto permaneceram sem DEL, pois pertencem a lactações anteriores cuja data de início não está disponível nessa fonte.

Foram identificadas observações com DEL superior a 305 dias, incluindo duas acima de 365 dias. Esses registros não foram considerados inconsistentes automaticamente, uma vez que lactações podem apresentar duração superior aos períodos convencionalmente utilizados como referência produtiva.

O DEL derivado será mantido como variável candidata para as etapas posteriores de modelagem.

In [ ]:
# ============================================================
# 12.17 INCORPORAÇÃO DO DEL DERIVADO
# ============================================================

del_para_base_uenp = analise_del_uenp[
    [
        "Número",
        "data_prova",
        "parto_convertido",
        "del_derivado"
    ]
].copy()

base_uenp_preparada = base_uenp_preparada.merge(
    del_para_base_uenp,
    how="left",
    on=["Número", "data_prova"],
    validate="one_to_one"
)

print("INCORPORAÇÃO DO DEL — UENP")
print("-" * 60)

print(
    "Observações após o merge:",
    len(base_uenp_preparada)
)

print(
    "DEL preenchido:",
    base_uenp_preparada["del_derivado"]
    .notna()
    .sum()
)

print(
    "DEL ausente:",
    base_uenp_preparada["del_derivado"]
    .isna()
    .sum()
)

print(
    "Cobertura geral:",
    round(
        base_uenp_preparada["del_derivado"]
        .notna()
        .mean()
        * 100,
        2
    ),
    "%"
)

# ------------------------------------------------------------
# COBERTURA POR ANO
# ------------------------------------------------------------

print("\nCobertura do DEL por ano:")

cobertura_del_ano = (
    base_uenp_preparada
    .groupby("ano_origem")
    .agg(
        observacoes=("animal", "size"),
        com_del=("del_derivado", lambda x: x.notna().sum())
    )
)

cobertura_del_ano["cobertura_%"] = (
    cobertura_del_ano["com_del"]
    / cobertura_del_ano["observacoes"]
    * 100
).round(2)

display(cobertura_del_ano)

# ------------------------------------------------------------
# COBERTURA APENAS ENTRE PRODUÇÕES MENSURADAS
# ------------------------------------------------------------

print("\nCobertura entre produções mensuradas por ano:")

mensuradas_del_ano = (
    base_uenp_preparada.loc[
        base_uenp_preparada["situacao_producao"] == "mensurada"
    ]
    .groupby("ano_origem")
    .agg(
        producoes_mensuradas=("animal", "size"),
        com_del=("del_derivado", lambda x: x.notna().sum())
    )
)

mensuradas_del_ano["cobertura_%"] = (
    mensuradas_del_ano["com_del"]
    / mensuradas_del_ano["producoes_mensuradas"]
    * 100
).round(2)

display(mensuradas_del_ano)

# ------------------------------------------------------------
# VALIDAÇÃO FINAL
# ------------------------------------------------------------

print("\nValidação do DEL incorporado:")

print(
    "DEL negativos:",
    (base_uenp_preparada["del_derivado"] < 0).sum()
)

print(
    "Duplicidades animal + data:",
    base_uenp_preparada.duplicated(
        subset=["animal", "data_prova"]
    ).sum()
)

### 12.18 Inspeção dos registros históricos de produção de 2024

Além dos controles leiteiros de 2025 e 2026 já estruturados, o arquivo da UENP contém uma planilha específica denominada `Produção 2024`.

Antes de incorporá-la à base longitudinal, será analisada sua estrutura, identificando os animais presentes, as datas disponíveis, os tipos de valores registrados e possíveis linhas de resumo ou agregação.

A inclusão desses registros dependerá da compatibilidade semântica entre os controles de 2024 e aqueles já utilizados para 2025 e 2026.

In [ ]:
# ============================================================
# 12.18 INSPEÇÃO DA PLANILHA PRODUÇÃO 2024
# ============================================================

from pathlib import Path

# Localiza o arquivo da UENP
caminhos_possiveis = [
    Path("../dados/originais/uenp/Bovinocultura UENP.xlsx"),
    Path("dados/originais/uenp/Bovinocultura UENP.xlsx")
]

caminho_uenp = next(
    (caminho for caminho in caminhos_possiveis if caminho.exists()),
    None
)

if caminho_uenp is None:
    raise FileNotFoundError(
        "Não foi possível localizar o arquivo Bovinocultura UENP.xlsx"
    )

# Carrega especificamente a aba Produção 2024
producao_2024_raw = pd.read_excel(
    caminho_uenp,
    sheet_name="Produção 2024"
)

print("PRODUÇÃO 2024 — ESTRUTURA ORIGINAL")
print("-" * 60)

print(
    "Dimensões:",
    producao_2024_raw.shape
)

print("\nColunas:")

for i, coluna in enumerate(producao_2024_raw.columns):
    print(i, repr(coluna))

print("\nPrimeiras linhas:")

display(
    producao_2024_raw.head(20)
)

print("\nÚltimas linhas:")

display(
    producao_2024_raw.tail(15)
)

print("\nTipos das colunas:")

display(
    producao_2024_raw.dtypes
)

print("\nValores preenchidos por coluna:")

display(
    producao_2024_raw.notna()
    .sum()
    .rename("preenchidos")
)

### 12.19 Estruturação dos controles leiteiros de 2024

A planilha `Produção 2024` contém registros individuais de produção distribuídos em sete datas de controle, entre maio e novembro de 2024.

Foi identificada uma linha denominada `Média`, correspondente a um resumo dos valores registrados em cada data. Por não representar um animal individual, essa linha será excluída da base destinada à modelagem.

Os registros serão transformados do formato horizontal para o formato longitudinal, mantendo separadamente valores numéricos de produção, marcações representadas por `-` e valores efetivamente ausentes.

A transformação seguirá a mesma estrutura utilizada para os controles de 2025 e 2026, permitindo posteriormente avaliar a integração dos três períodos.

In [ ]:
# ============================================================
# 12.19 ESTRUTURAÇÃO DA PRODUÇÃO DE 2024
# ============================================================

# Identificação das colunas correspondentes às datas
colunas_datas_2024 = []

for coluna in producao_2024_raw.columns:

    data_convertida = pd.to_datetime(
        coluna,
        errors="coerce"
    )

    if (
        pd.notna(data_convertida)
        and data_convertida.year == 2024
    ):
        colunas_datas_2024.append(coluna)


print("PRODUÇÃO UENP — 2024")
print("-" * 60)

print("\nDatas identificadas:")

for coluna in colunas_datas_2024:
    print("-", pd.to_datetime(coluna).date())


# ------------------------------------------------------------
# TRANSFORMAÇÃO PARA FORMATO LONGITUDINAL
# ------------------------------------------------------------

producao_uenp_2024 = (
    producao_2024_raw[
        ["Vaca"] + colunas_datas_2024
    ]
    .copy()
)

producao_uenp_2024 = producao_uenp_2024.melt(
    id_vars="Vaca",
    value_vars=colunas_datas_2024,
    var_name="data_prova",
    value_name="valor_original"
)

producao_uenp_2024 = producao_uenp_2024.rename(
    columns={"Vaca": "animal"}
)

# Padronização básica do nome
producao_uenp_2024["animal"] = (
    producao_uenp_2024["animal"]
    .astype("string")
    .str.strip()
)

producao_uenp_2024["animal_normalizado"] = (
    producao_uenp_2024["animal"]
    .apply(normalizar_nome_animal)
)

# Conversão da data
producao_uenp_2024["data_prova"] = pd.to_datetime(
    producao_uenp_2024["data_prova"],
    errors="coerce"
)

# Conversão da produção
producao_uenp_2024["kg_leite"] = pd.to_numeric(
    producao_uenp_2024["valor_original"],
    errors="coerce"
)


# ------------------------------------------------------------
# CLASSIFICAÇÃO DO TIPO DE REGISTRO
# ------------------------------------------------------------

def classificar_producao_2024(valor):

    if pd.isna(valor):
        return "ausente"

    if str(valor).strip() == "-":
        return "sem_registro"

    try:
        float(valor)
        return "mensurada"
    except (ValueError, TypeError):
        return "outro"


producao_uenp_2024["situacao_producao"] = (
    producao_uenp_2024["valor_original"]
    .apply(classificar_producao_2024)
)

producao_uenp_2024["ano_origem"] = 2024


# ------------------------------------------------------------
# REMOÇÃO DA LINHA DE RESUMO
# ------------------------------------------------------------

producao_uenp_2024_limpa = producao_uenp_2024.loc[
    producao_uenp_2024["animal_normalizado"] != "média"
].copy()

producao_uenp_2024_limpa = (
    producao_uenp_2024_limpa
    .sort_values(["animal", "data_prova"])
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# VALIDAÇÃO
# ------------------------------------------------------------

print("\nPRODUÇÃO 2024 APÓS ESTRUTURAÇÃO")
print("-" * 60)

print(
    "Observações:",
    len(producao_uenp_2024_limpa)
)

print(
    "Animais distintos:",
    producao_uenp_2024_limpa["animal"].nunique()
)

print(
    "Datas distintas:",
    producao_uenp_2024_limpa["data_prova"].nunique()
)

print("\nSituações de produção:")

display(
    producao_uenp_2024_limpa[
        "situacao_producao"
    ].value_counts(dropna=False)
)

print("\nDuplicidades animal + data:")

print(
    producao_uenp_2024_limpa.duplicated(
        subset=["animal", "data_prova"]
    ).sum()
)

print("\nAnimais encontrados:")

display(
    pd.Series(
        sorted(
            producao_uenp_2024_limpa[
                "animal"
            ].dropna().unique()
        ),
        name="animal"
    )
)

print("\nResumo das produções mensuradas:")

display(
    producao_uenp_2024_limpa[
        "kg_leite"
    ].describe()
)

### 12.20 Correspondência dos animais históricos de 2024

Os controles leiteiros de 2024 contêm 14 animais individuais e 72 registros de produção mensurada.

Como parte desses animais pode não estar mais presente no cadastro atual da propriedade, será avaliada inicialmente a correspondência entre os nomes registrados em 2024 e a planilha `Visão Geral`.

Também será verificado se os animais de 2024 aparecem nos controles leiteiros de 2025 e 2026. Essa análise permitirá distinguir animais que permanecem nas bases posteriores daqueles presentes apenas no histórico de 2024.

A ausência de correspondência com o cadastro atual não será utilizada como critério de exclusão dos controles históricos.

In [ ]:
# ============================================================
# 12.20 CORRESPONDÊNCIA DOS ANIMAIS DE 2024
# ============================================================

animais_2024 = (
    producao_uenp_2024_limpa[
        ["animal", "animal_normalizado"]
    ]
    .drop_duplicates()
    .sort_values("animal")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 1. CORRESPONDÊNCIA COM O CADASTRO ATUAL
# ------------------------------------------------------------

cadastro_nomes_uenp = (
    visao_geral_uenp[
        ["Número", "Animal", "Raça", "Nascimento", "animal_normalizado"]
    ]
    .drop_duplicates(subset=["animal_normalizado"])
    .copy()
)

comparacao_2024_cadastro = animais_2024.merge(
    cadastro_nomes_uenp,
    how="left",
    on="animal_normalizado",
    validate="one_to_one"
)

comparacao_2024_cadastro["encontrado_cadastro"] = (
    comparacao_2024_cadastro["Número"].notna()
)

print("ANIMAIS DE 2024 x CADASTRO ATUAL")
print("-" * 60)

print(
    "Animais de 2024:",
    len(comparacao_2024_cadastro)
)

print(
    "Encontrados no cadastro:",
    comparacao_2024_cadastro["encontrado_cadastro"].sum()
)

print(
    "Não encontrados:",
    (~comparacao_2024_cadastro["encontrado_cadastro"]).sum()
)

print("\nCorrespondência completa:")

display(
    comparacao_2024_cadastro[
        [
            "animal",
            "Número",
            "Animal",
            "Raça",
            "Nascimento",
            "encontrado_cadastro"
        ]
    ]
)

# ------------------------------------------------------------
# 2. PRESENÇA NOS CONTROLES DE 2025/2026
# ------------------------------------------------------------

animais_2025_2026 = set(
    producao_uenp_limpa[
        "animal_normalizado"
    ].dropna()
)

comparacao_2024_cadastro["aparece_2025_2026"] = (
    comparacao_2024_cadastro[
        "animal_normalizado"
    ].isin(animais_2025_2026)
)

print("\nPRESENÇA DOS ANIMAIS DE 2024 EM 2025/2026")
print("-" * 60)

display(
    comparacao_2024_cadastro[
        [
            "animal",
            "encontrado_cadastro",
            "aparece_2025_2026"
        ]
    ]
)

print(
    "\nAnimais de 2024 que também aparecem em 2025/2026:",
    comparacao_2024_cadastro[
        "aparece_2025_2026"
    ].sum()
)

print(
    "Animais exclusivos de 2024:",
    (
        ~comparacao_2024_cadastro[
            "aparece_2025_2026"
        ]
    ).sum()
)

# ------------------------------------------------------------
# 3. PRODUÇÕES MENSurADAS POR GRUPO
# ------------------------------------------------------------

producao_2024_classificada = producao_uenp_2024_limpa.merge(
    comparacao_2024_cadastro[
        [
            "animal_normalizado",
            "encontrado_cadastro",
            "aparece_2025_2026"
        ]
    ],
    how="left",
    on="animal_normalizado",
    validate="many_to_one"
)

print("\nPRODUÇÕES MENSURADAS DE 2024 POR DISPONIBILIDADE CADASTRAL")

display(
    producao_2024_classificada.loc[
        producao_2024_classificada["situacao_producao"] == "mensurada"
    ]
    .groupby(
        ["encontrado_cadastro", "aparece_2025_2026"]
    )
    .size()
    .rename("producoes_mensuradas")
)

### 12.21 Integração dos controles de 2024 com o cadastro atual

Dos 14 animais presentes nos controles leiteiros de 2024, sete possuem correspondência direta com o cadastro atual da propriedade.

Outros animais representam registros históricos de indivíduos que não estão mais presentes na relação cadastral atual. Esses registros não serão descartados, uma vez que constituem observações reais de produção.

Para os animais que possuem correspondência cadastral, serão incorporadas apenas informações consideradas estáveis ao longo do tempo, como número de identificação, raça e data de nascimento.

Informações atuais relacionadas à lactação, parto, DEL ou número de partos não serão atribuídas retrospectivamente aos controles de 2024.

A idade do animal será calculada na data de cada controle sempre que a data de nascimento estiver disponível.

In [ ]:
# ============================================================
# 12.21 INTEGRAÇÃO DA PRODUÇÃO DE 2024 COM O CADASTRO
# ============================================================

cadastro_estavel_uenp = visao_geral_uenp[
    [
        "Número",
        "Animal",
        "Raça",
        "Nascimento",
        "animal_normalizado"
    ]
].copy()

# ------------------------------------------------------------
# MERGE
# ------------------------------------------------------------

producao_uenp_2024_integrada = (
    producao_uenp_2024_limpa.merge(
        cadastro_estavel_uenp,
        how="left",
        on="animal_normalizado",
        validate="many_to_one"
    )
)

# ------------------------------------------------------------
# PADRONIZAÇÃO DA RAÇA
# ------------------------------------------------------------

producao_uenp_2024_integrada["raca_padronizada"] = (
    producao_uenp_2024_integrada["Raça"]
    .astype("string")
    .str.strip()
)

# ------------------------------------------------------------
# DATA DE NASCIMENTO E IDADE
# ------------------------------------------------------------

producao_uenp_2024_integrada[
    "data_nascimento_convertida"
] = pd.to_datetime(
    producao_uenp_2024_integrada["Nascimento"],
    errors="coerce"
)

producao_uenp_2024_integrada["idade_anos"] = (
    (
        producao_uenp_2024_integrada["data_prova"]
        - producao_uenp_2024_integrada[
            "data_nascimento_convertida"
        ]
    ).dt.days
    / 365.25
)

# ------------------------------------------------------------
# VALIDAÇÃO
# ------------------------------------------------------------

print("INTEGRAÇÃO DA PRODUÇÃO 2024 COM O CADASTRO")
print("-" * 60)

print(
    "Observações:",
    len(producao_uenp_2024_integrada)
)

print(
    "Observações com cadastro:",
    producao_uenp_2024_integrada["Número"]
    .notna()
    .sum()
)

print(
    "Observações sem cadastro:",
    producao_uenp_2024_integrada["Número"]
    .isna()
    .sum()
)

print(
    "Animais distintos com cadastro:",
    producao_uenp_2024_integrada.loc[
        producao_uenp_2024_integrada["Número"].notna(),
        "animal"
    ].nunique()
)

print("\nCobertura entre produções mensuradas:")

mensuradas_2024_integradas = (
    producao_uenp_2024_integrada.loc[
        producao_uenp_2024_integrada[
            "situacao_producao"
        ] == "mensurada"
    ]
)

print(
    "Produções mensuradas:",
    len(mensuradas_2024_integradas)
)

print(
    "Com cadastro:",
    mensuradas_2024_integradas["Número"]
    .notna()
    .sum()
)

print(
    "Com raça:",
    mensuradas_2024_integradas[
        "raca_padronizada"
    ].notna().sum()
)

print(
    "Com idade:",
    mensuradas_2024_integradas[
        "idade_anos"
    ].notna().sum()
)

print("\nResumo das idades disponíveis:")

display(
    mensuradas_2024_integradas[
        "idade_anos"
    ].describe()
)

print(
    "Idades negativas:",
    (
        mensuradas_2024_integradas[
            "idade_anos"
        ] < 0
    ).sum()
)

print(
    "Idades menores que 1,5 ano:",
    (
        mensuradas_2024_integradas[
            "idade_anos"
        ] < 1.5
    ).sum()
)

print("\nAnimais sem cadastro atual:")

display(
    producao_uenp_2024_integrada.loc[
        producao_uenp_2024_integrada["Número"].isna(),
        ["animal"]
    ]
    .drop_duplicates()
    .sort_values("animal")
)

### 12.22 Inspeção das pesagens históricas de 2024 e 2025

Além dos registros de peso corporal de 2026, o arquivo da UENP contém planilhas específicas de pesagem referentes aos anos de 2024 e 2025.

Antes de realizar qualquer associação com os controles leiteiros, será analisada a estrutura dessas duas fontes, verificando os animais registrados, as datas disponíveis, a quantidade de valores preenchidos e possíveis diferenças de organização entre os anos.

A incorporação desses registros poderá ampliar a disponibilidade do peso corporal nos controles históricos, mas somente será realizada após a avaliação da compatibilidade temporal e estrutural das informações.

In [ ]:
# ============================================================
# 12.22 INSPEÇÃO DAS PESAGENS HISTÓRICAS — 2024 E 2025
# ============================================================

pesagem_2024_raw = pd.read_excel(
    caminho_uenp,
    sheet_name="Pesagem Animais 2024"
)

pesagem_2025_raw = pd.read_excel(
    caminho_uenp,
    sheet_name="Pesagem Animais 2025"
)


def inspecionar_planilha_pesagem(df, nome):

    print("\n" + "=" * 70)
    print(nome)
    print("=" * 70)

    print(
        "\nDimensões:",
        df.shape
    )

    print("\nColunas:")

    for i, coluna in enumerate(df.columns):
        print(i, repr(coluna))

    print("\nPrimeiras linhas:")

    display(
        df.head(20)
    )

    print("\nÚltimas linhas:")

    display(
        df.tail(15)
    )

    print("\nTipos das colunas:")

    display(
        df.dtypes
    )

    print("\nValores preenchidos por coluna:")

    display(
        df.notna()
        .sum()
        .rename("preenchidos")
    )


# ------------------------------------------------------------
# INSPEÇÃO
# ------------------------------------------------------------

inspecionar_planilha_pesagem(
    pesagem_2024_raw,
    "PESAGEM ANIMAIS 2024"
)

inspecionar_planilha_pesagem(
    pesagem_2025_raw,
    "PESAGEM ANIMAIS 2025"
)

### 12.23 Estruturação e comparação das pesagens históricas

A inspeção das planilhas de pesagem mostrou que a aba `Pesagem Animais 2024` também contém registros referentes aos meses de janeiro, fevereiro e março de 2025.

Essas mesmas datas estão presentes na planilha `Pesagem Animais 2025`, criando uma sobreposição entre as duas fontes.

Antes da consolidação, ambas as planilhas serão transformadas para o formato longitudinal e os registros coincidentes serão comparados. Valores não numéricos, como indicações textuais de frequência de pesagem, serão preservados para inspeção, mas não interpretados como peso corporal.

A consolidação definitiva somente será realizada após a verificação de possíveis divergências entre as fontes.

In [ ]:
# ============================================================
# 12.23 ESTRUTURAÇÃO E COMPARAÇÃO DAS PESAGENS HISTÓRICAS
# ============================================================

def transformar_pesagem_historica(df, nome_fonte):

    # Identifica todas as colunas que representam datas
    colunas_datas = []

    for coluna in df.columns:

        data_convertida = pd.to_datetime(
            coluna,
            errors="coerce"
        )

        if pd.notna(data_convertida):
            colunas_datas.append(coluna)

    # Mantém identificadores e datas
    base = df[
        ["Número", "Animal"] + colunas_datas
    ].copy()

    # Formato longitudinal
    base = base.melt(
        id_vars=["Número", "Animal"],
        value_vars=colunas_datas,
        var_name="data_pesagem",
        value_name="valor_original"
    )

    # Padronizações
    base["data_pesagem"] = pd.to_datetime(
        base["data_pesagem"],
        errors="coerce"
    )

    base["animal_normalizado"] = (
        base["Animal"]
        .apply(normalizar_nome_animal)
    )

    base["peso_kg"] = pd.to_numeric(
        base["valor_original"],
        errors="coerce"
    )

    # Classificação do conteúdo original
    def classificar_valor(valor):

        if pd.isna(valor):
            return "ausente"

        if str(valor).strip() == "-":
            return "sem_registro"

        try:
            float(valor)
            return "mensurado"

        except (ValueError, TypeError):
            return "texto"

    base["situacao_pesagem"] = (
        base["valor_original"]
        .apply(classificar_valor)
    )

    base["fonte_pesagem"] = nome_fonte

    return base


# ------------------------------------------------------------
# TRANSFORMAÇÃO DAS DUAS FONTES
# ------------------------------------------------------------

pesagem_hist_2024 = transformar_pesagem_historica(
    pesagem_2024_raw,
    "Pesagem Animais 2024"
)

pesagem_hist_2025 = transformar_pesagem_historica(
    pesagem_2025_raw,
    "Pesagem Animais 2025"
)


# ------------------------------------------------------------
# CARACTERIZAÇÃO INDIVIDUAL
# ------------------------------------------------------------

for nome, df in [
    ("Pesagem Animais 2024", pesagem_hist_2024),
    ("Pesagem Animais 2025", pesagem_hist_2025)
]:

    print("\n" + "=" * 65)
    print(nome)
    print("=" * 65)

    print("Observações possíveis:", len(df))
    print("Animais distintos:", df["Número"].nunique())
    print("Datas distintas:", df["data_pesagem"].nunique())
    print("Pesos numéricos:", df["peso_kg"].notna().sum())

    print("\nSituações:")

    display(
        df["situacao_pesagem"]
        .value_counts(dropna=False)
    )

    print("\nIntervalo de datas:")

    print(
        df["data_pesagem"].min(),
        "até",
        df["data_pesagem"].max()
    )


# ------------------------------------------------------------
# VALORES TEXTUAIS NÃO NUMÉRICOS
# ------------------------------------------------------------

print("\nVALORES TEXTUAIS NÃO NUMÉRICOS")
print("-" * 60)

display(
    pesagem_hist_2024.loc[
        pesagem_hist_2024["situacao_pesagem"] == "texto",
        [
            "Número",
            "Animal",
            "data_pesagem",
            "valor_original"
        ]
    ]
)


# ------------------------------------------------------------
# SOBREPOSIÇÃO ENTRE AS DUAS PLANILHAS
# ------------------------------------------------------------

comparacao_pesagens = pesagem_hist_2024.merge(
    pesagem_hist_2025,
    how="inner",
    on=["Número", "data_pesagem"],
    suffixes=("_2024", "_2025")
)

# Apenas casos em que pelo menos uma fonte contém peso
comparacao_pesagens_validas = comparacao_pesagens.loc[
    comparacao_pesagens["peso_kg_2024"].notna()
    | comparacao_pesagens["peso_kg_2025"].notna()
].copy()

comparacao_pesagens_validas["pesos_iguais"] = (
    comparacao_pesagens_validas["peso_kg_2024"]
    == comparacao_pesagens_validas["peso_kg_2025"]
)

print("\nSOBREPOSIÇÃO ENTRE AS PLANILHAS")
print("-" * 60)

print(
    "Registros coincidentes animal + data:",
    len(comparacao_pesagens)
)

print(
    "Com peso em pelo menos uma fonte:",
    len(comparacao_pesagens_validas)
)

print(
    "Pesos iguais:",
    comparacao_pesagens_validas[
        "pesos_iguais"
    ].sum()
)

print(
    "Pesos diferentes:",
    (
        ~comparacao_pesagens_validas[
            "pesos_iguais"
        ]
    ).sum()
)

print("\nPossíveis divergências:")

display(
    comparacao_pesagens_validas.loc[
        ~comparacao_pesagens_validas["pesos_iguais"],
        [
            "Número",
            "Animal_2024",
            "Animal_2025",
            "data_pesagem",
            "peso_kg_2024",
            "peso_kg_2025"
        ]
    ]
)

### 12.24 Recuperação dos identificadores históricos dos animais

As planilhas históricas de pesagem apresentam, além do nome dos animais, um número individual de identificação.

Essa informação permite recuperar identificadores de animais que possuem registros produtivos históricos, mas que não estão presentes no cadastro atual da propriedade.

Antes de utilizar esses identificadores, será verificada a consistência da relação entre nome e número nas diferentes planilhas de pesagem. Apenas correspondências unívocas serão utilizadas.

Essa etapa possibilita melhorar a rastreabilidade dos registros históricos sem inferir correspondências por similaridade de nomes.

In [ ]:
# ============================================================
# 12.24 RECUPERAÇÃO DOS IDENTIFICADORES HISTÓRICOS
# ============================================================

# ------------------------------------------------------------
# 1. MAPAS NOME x NÚMERO NAS PESAGENS HISTÓRICAS
# ------------------------------------------------------------

identidade_pesagem_2024 = (
    pesagem_hist_2024[
        ["Número", "Animal", "animal_normalizado"]
    ]
    .drop_duplicates()
    .copy()
)

identidade_pesagem_2025 = (
    pesagem_hist_2025[
        ["Número", "Animal", "animal_normalizado"]
    ]
    .drop_duplicates()
    .copy()
)

identidade_historica = pd.concat(
    [
        identidade_pesagem_2024,
        identidade_pesagem_2025
    ],
    ignore_index=True
).drop_duplicates()

print("VALIDAÇÃO DOS IDENTIFICADORES HISTÓRICOS")
print("-" * 60)

print(
    "Relações nome/número distintas:",
    len(identidade_historica)
)

# ------------------------------------------------------------
# 2. UM NOME ASSOCIADO A MAIS DE UM NÚMERO?
# ------------------------------------------------------------

numeros_por_nome = (
    identidade_historica
    .groupby("animal_normalizado")["Número"]
    .nunique()
)

nomes_com_multiplos_ids = numeros_por_nome[
    numeros_por_nome > 1
]

print(
    "Nomes associados a mais de um Número:",
    len(nomes_com_multiplos_ids)
)

if len(nomes_com_multiplos_ids) > 0:
    display(nomes_com_multiplos_ids)

# ------------------------------------------------------------
# 3. UM NÚMERO ASSOCIADO A MAIS DE UM NOME?
# ------------------------------------------------------------

nomes_por_numero = (
    identidade_historica
    .groupby("Número")["animal_normalizado"]
    .nunique()
)

ids_com_multiplos_nomes = nomes_por_numero[
    nomes_por_numero > 1
]

print(
    "Números associados a mais de um nome:",
    len(ids_com_multiplos_nomes)
)

if len(ids_com_multiplos_nomes) > 0:
    display(ids_com_multiplos_nomes)

# ------------------------------------------------------------
# 4. MAPA FINAL APENAS SE A RELAÇÃO FOR UNÍVOCA
# ------------------------------------------------------------

mapa_identidade_historica = (
    identidade_historica[
        ["animal_normalizado", "Número"]
    ]
    .drop_duplicates()
)

print("\nMapa histórico de identificação:")

display(
    mapa_identidade_historica
    .sort_values("animal_normalizado")
)

# ------------------------------------------------------------
# 5. ANIMAIS DE 2024 QUE ANTES ESTAVAM SEM CADASTRO
# ------------------------------------------------------------

animais_2024_sem_cadastro = (
    producao_uenp_2024_integrada.loc[
        producao_uenp_2024_integrada["Número"].isna(),
        ["animal", "animal_normalizado"]
    ]
    .drop_duplicates()
)

recuperacao_2024 = animais_2024_sem_cadastro.merge(
    mapa_identidade_historica,
    how="left",
    on="animal_normalizado",
    validate="one_to_one"
)

print("\nRECUPERAÇÃO DOS ANIMAIS DE 2024 SEM CADASTRO")
print("-" * 60)

display(recuperacao_2024)

print(
    "Animais históricos sem cadastro:",
    len(recuperacao_2024)
)

print(
    "Identificadores recuperados:",
    recuperacao_2024["Número"].notna().sum()
)

print(
    "Ainda sem identificador:",
    recuperacao_2024["Número"].isna().sum()
)

# ------------------------------------------------------------
# 6. CORA
# ------------------------------------------------------------

print("\nIdentificação histórica da Cora:")

display(
    mapa_identidade_historica.loc[
        mapa_identidade_historica[
            "animal_normalizado"
        ] == "cora"
    ]
)

### 12.25 Incorporação dos identificadores históricos

As planilhas históricas de pesagem permitiram recuperar os números de identificação de animais que não estão mais presentes no cadastro atual.

A relação entre nome e número mostrou-se unívoca, sem ocorrência de um mesmo nome associado a múltiplos identificadores ou de um mesmo identificador associado a diferentes animais.

Antes da incorporação, os identificadores históricos serão comparados aos números já obtidos do cadastro atual. Os valores históricos serão utilizados apenas para preencher identificadores ausentes, sem substituir informações já existentes.

Essa estratégia permite ampliar a rastreabilidade dos controles históricos e possibilitar a associação com outras fontes de dados, como as pesagens realizadas em 2024 e 2025.

In [ ]:
# ============================================================
# 12.25 INCORPORAÇÃO DOS IDENTIFICADORES HISTÓRICOS
# ============================================================

mapa_ids_historicos = (
    mapa_identidade_historica[
        ["animal_normalizado", "Número"]
    ]
    .rename(
        columns={"Número": "numero_historico"}
    )
    .copy()
)


def incorporar_identificador_historico(df, nome_base):

    base = df.merge(
        mapa_ids_historicos,
        how="left",
        on="animal_normalizado",
        validate="many_to_one"
    )

    # --------------------------------------------------------
    # VERIFICAÇÃO DE CONFLITOS
    # --------------------------------------------------------

    conflitos = base.loc[
        base["Número"].notna()
        & base["numero_historico"].notna()
        & (
            base["Número"].astype(str)
            != base["numero_historico"].astype(str)
        )
    ].copy()

    print("\n" + "=" * 65)
    print(nome_base)
    print("=" * 65)

    print(
        "Conflitos entre cadastro atual e histórico:",
        len(conflitos)
    )

    if len(conflitos) > 0:

        display(
            conflitos[
                [
                    "animal",
                    "Número",
                    "numero_historico"
                ]
            ].drop_duplicates()
        )

        raise ValueError(
            "Foram encontrados conflitos de identificação. "
            "A incorporação foi interrompida."
        )

    # --------------------------------------------------------
    # FONTE DO IDENTIFICADOR
    # --------------------------------------------------------

    base["fonte_identificador"] = "ausente"

    base.loc[
        base["Número"].notna(),
        "fonte_identificador"
    ] = "cadastro_atual"

    base.loc[
        base["Número"].isna()
        & base["numero_historico"].notna(),
        "fonte_identificador"
    ] = "pesagem_historica"

    # --------------------------------------------------------
    # PREENCHIMENTO APENAS DOS AUSENTES
    # --------------------------------------------------------

    base["Número"] = (
        base["Número"]
        .fillna(base["numero_historico"])
    )

    base = base.drop(
        columns=["numero_historico"]
    )

    print(
        "Observações:",
        len(base)
    )

    print(
        "Com identificador:",
        base["Número"].notna().sum()
    )

    print(
        "Sem identificador:",
        base["Número"].isna().sum()
    )

    print(
        "Animais distintos:",
        base["animal"].nunique()
    )

    print(
        "Animais distintos identificados:",
        base.loc[
            base["Número"].notna(),
            "animal"
        ].nunique()
    )

    print("\nFonte dos identificadores:")

    display(
        base["fonte_identificador"]
        .value_counts(dropna=False)
    )

    return base


# ------------------------------------------------------------
# BASE DE 2024
# ------------------------------------------------------------

producao_uenp_2024_integrada = (
    incorporar_identificador_historico(
        producao_uenp_2024_integrada,
        "PRODUÇÃO UENP 2024"
    )
)


# ------------------------------------------------------------
# BASE DE 2025/2026
# ------------------------------------------------------------

base_uenp_preparada = (
    incorporar_identificador_historico(
        base_uenp_preparada,
        "PRODUÇÃO UENP 2025/2026"
    )
)


# ------------------------------------------------------------
# CONFERÊNCIA DA CORA
# ------------------------------------------------------------

print("\nREGISTROS DA CORA APÓS RECUPERAÇÃO DO IDENTIFICADOR")
print("-" * 65)

display(
    pd.concat(
        [
            producao_uenp_2024_integrada.loc[
                producao_uenp_2024_integrada[
                    "animal_normalizado"
                ] == "cora",
                [
                    "Número",
                    "animal",
                    "data_prova",
                    "kg_leite",
                    "fonte_identificador"
                ]
            ],
            base_uenp_preparada.loc[
                base_uenp_preparada[
                    "animal_normalizado"
                ] == "cora",
                [
                    "Número",
                    "animal",
                    "data_prova",
                    "kg_leite",
                    "fonte_identificador"
                ]
            ]
        ],
        ignore_index=True
    )
    .sort_values("data_prova")
)

### 12.26 Consolidação dos registros de peso corporal

As planilhas históricas de pesagem apresentaram sobreposição entre janeiro e março de 2025. A comparação dos registros coincidentes demonstrou que os valores disponíveis nas duas fontes são iguais, não sendo identificadas divergências de peso.

Para evitar duplicações sem perder registros exclusivos de alguma das fontes, as pesagens de 2024 e 2025 serão consolidadas pela combinação entre número do animal e data da pesagem.

Quando um mesmo registro estiver presente nas duas planilhas, será mantida uma única ocorrência. Em seguida, os registros históricos serão combinados com as pesagens de 2026.

Nesta etapa será criada uma base longitudinal única de peso corporal. A associação temporal desses pesos aos controles leiteiros será realizada posteriormente.

In [ ]:
# ============================================================
# 12.26 CONSOLIDAÇÃO DAS PESAGENS DE 2024, 2025 E 2026
# ============================================================

# ------------------------------------------------------------
# 1. CONSOLIDAÇÃO DAS FONTES HISTÓRICAS
# ------------------------------------------------------------

pesagens_historicas = pd.concat(
    [
        pesagem_hist_2024,
        pesagem_hist_2025
    ],
    ignore_index=True
)

# Prioridade para a planilha específica de 2025
pesagens_historicas["prioridade_fonte"] = (
    pesagens_historicas["fonte_pesagem"]
    .map({
        "Pesagem Animais 2024": 1,
        "Pesagem Animais 2025": 2
    })
)

# Prioriza registros que realmente possuem peso numérico
pesagens_historicas["tem_peso"] = (
    pesagens_historicas["peso_kg"].notna()
)

print("CONSOLIDAÇÃO DAS PESAGENS HISTÓRICAS")
print("-" * 60)

print(
    "Registros antes da consolidação:",
    len(pesagens_historicas)
)

print(
    "Duplicações Número + data:",
    pesagens_historicas.duplicated(
        subset=["Número", "data_pesagem"]
    ).sum()
)

# Quando existe o mesmo animal + data em duas fontes:
# 1. prioriza registro que possui peso
# 2. em empate, prioriza a planilha específica de 2025
pesagens_historicas = (
    pesagens_historicas
    .sort_values(
        [
            "Número",
            "data_pesagem",
            "tem_peso",
            "prioridade_fonte"
        ],
        ascending=[
            True,
            True,
            False,
            False
        ]
    )
    .drop_duplicates(
        subset=["Número", "data_pesagem"],
        keep="first"
    )
    .reset_index(drop=True)
)

print(
    "Registros após a consolidação:",
    len(pesagens_historicas)
)

print(
    "Duplicações restantes:",
    pesagens_historicas.duplicated(
        subset=["Número", "data_pesagem"]
    ).sum()
)


# ------------------------------------------------------------
# 2. HARMONIZAÇÃO DO NOME DA COLUNA ORIGINAL
# ------------------------------------------------------------

# Nas planilhas históricas a coluna se chama valor_original.
# Vamos padronizar para peso_original, igual à base de 2026.
pesagens_historicas = pesagens_historicas.rename(
    columns={
        "valor_original": "peso_original"
    }
)


# ------------------------------------------------------------
# 3. PADRONIZAÇÃO DA PESAGEM DE 2026
# ------------------------------------------------------------

pesagem_2026_padronizada = pesagem_uenp[
    [
        "Número",
        "Animal",
        "animal_normalizado",
        "data_pesagem",
        "peso_original",
        "peso_kg"
    ]
].copy()

pesagem_2026_padronizada["situacao_pesagem"] = "ausente"

pesagem_2026_padronizada.loc[
    pesagem_2026_padronizada["peso_kg"].notna(),
    "situacao_pesagem"
] = "mensurado"

pesagem_2026_padronizada["fonte_pesagem"] = (
    "Pesagem 2026"
)


# ------------------------------------------------------------
# 4. SELEÇÃO DAS COLUNAS COMUNS
# ------------------------------------------------------------

colunas_pesagem = [
    "Número",
    "Animal",
    "animal_normalizado",
    "data_pesagem",
    "peso_original",
    "peso_kg",
    "situacao_pesagem",
    "fonte_pesagem"
]

pesagens_historicas = (
    pesagens_historicas[
        colunas_pesagem
    ].copy()
)

pesagem_2026_padronizada = (
    pesagem_2026_padronizada[
        colunas_pesagem
    ].copy()
)


# ------------------------------------------------------------
# 5. BASE ÚNICA DE PESAGENS
# ------------------------------------------------------------

pesagem_uenp_consolidada = pd.concat(
    [
        pesagens_historicas,
        pesagem_2026_padronizada
    ],
    ignore_index=True
)

pesagem_uenp_consolidada = (
    pesagem_uenp_consolidada
    .sort_values(
        ["Número", "data_pesagem"]
    )
    .reset_index(drop=True)
)

pesagem_uenp_consolidada["ano_pesagem"] = (
    pesagem_uenp_consolidada[
        "data_pesagem"
    ].dt.year
)


# ------------------------------------------------------------
# 6. VALIDAÇÃO
# ------------------------------------------------------------

print("\nBASE CONSOLIDADA DE PESAGENS — UENP")
print("-" * 60)

print(
    "Registros totais:",
    len(pesagem_uenp_consolidada)
)

print(
    "Animais distintos:",
    pesagem_uenp_consolidada[
        "Número"
    ].nunique()
)

print(
    "Datas distintas:",
    pesagem_uenp_consolidada[
        "data_pesagem"
    ].nunique()
)

print(
    "Pesos numéricos:",
    pesagem_uenp_consolidada[
        "peso_kg"
    ].notna().sum()
)

print(
    "Duplicidades Número + data:",
    pesagem_uenp_consolidada.duplicated(
        subset=["Número", "data_pesagem"]
    ).sum()
)

print("\nRegistros por ano:")

display(
    pesagem_uenp_consolidada[
        "ano_pesagem"
    ]
    .value_counts()
    .sort_index()
)

print("\nPesos válidos por ano:")

display(
    pesagem_uenp_consolidada.loc[
        pesagem_uenp_consolidada["peso_kg"].notna(),
        "ano_pesagem"
    ]
    .value_counts()
    .sort_index()
)

print("\nResumo geral dos pesos:")

display(
    pesagem_uenp_consolidada[
        "peso_kg"
    ].describe()
)

### 12.27 Associação temporal das pesagens aos controles leiteiros

Após a consolidação das pesagens de 2024, 2025 e 2026, os registros de peso corporal serão associados temporalmente aos controles leiteiros.

Para cada controle será utilizada exclusivamente a pesagem válida mais recente realizada na mesma data ou anteriormente à mensuração da produção. Pesagens posteriores ao controle não serão utilizadas, evitando a introdução de informações futuras no conjunto de atributos.

Além do peso corporal, será calculado o intervalo, em dias, entre a pesagem selecionada e o respectivo controle leiteiro.

Nesta etapa será avaliada a cobertura e a distância temporal das associações antes de incorporar definitivamente o peso à base analítica.

In [ ]:
# ============================================================
# 12.27 ASSOCIAÇÃO TEMPORAL DO PESO — 2024 A 2026
# ============================================================

# ------------------------------------------------------------
# 1. BASE DE CONTROLES DOS TRÊS ANOS
# ------------------------------------------------------------

controles_2024_peso = producao_uenp_2024_integrada[
    [
        "Número",
        "animal",
        "data_prova",
        "kg_leite",
        "situacao_producao",
        "ano_origem"
    ]
].copy()

controles_2025_2026_peso = base_uenp_preparada[
    [
        "Número",
        "animal",
        "data_prova",
        "kg_leite",
        "situacao_producao",
        "ano_origem"
    ]
].copy()

controles_uenp_todos = pd.concat(
    [
        controles_2024_peso,
        controles_2025_2026_peso
    ],
    ignore_index=True
)

print("CONTROLES LEITEIROS — UENP")
print("-" * 60)

print(
    "Observações totais:",
    len(controles_uenp_todos)
)

print("\nObservações por ano:")

display(
    controles_uenp_todos[
        "ano_origem"
    ].value_counts().sort_index()
)


# ------------------------------------------------------------
# 2. MANTÉM SOMENTE PESAGENS NUMÉRICAS
# ------------------------------------------------------------

pesagens_validas_todas = (
    pesagem_uenp_consolidada.loc[
        pesagem_uenp_consolidada[
            "peso_kg"
        ].notna(),
        [
            "Número",
            "data_pesagem",
            "peso_kg",
            "fonte_pesagem"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 3. CRUZAMENTO CONTROLE x PESAGEM
# ------------------------------------------------------------

candidatos_peso_todos = controles_uenp_todos.merge(
    pesagens_validas_todas,
    how="left",
    on="Número"
)

# Somente pesagens realizadas na mesma data ou antes
candidatos_peso_todos = candidatos_peso_todos.loc[
    candidatos_peso_todos["data_pesagem"]
    <= candidatos_peso_todos["data_prova"]
].copy()

# Distância temporal
candidatos_peso_todos["dias_desde_pesagem"] = (
    candidatos_peso_todos["data_prova"]
    - candidatos_peso_todos["data_pesagem"]
).dt.days


# ------------------------------------------------------------
# 4. SELEÇÃO DA PESAGEM MAIS RECENTE
# ------------------------------------------------------------

peso_associado_uenp_todos = (
    candidatos_peso_todos
    .sort_values(
        [
            "Número",
            "data_prova",
            "dias_desde_pesagem"
        ]
    )
    .drop_duplicates(
        subset=["Número", "data_prova"],
        keep="first"
    )
    [
        [
            "Número",
            "data_prova",
            "data_pesagem",
            "peso_kg",
            "dias_desde_pesagem",
            "fonte_pesagem"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 5. COBERTURA GERAL
# ------------------------------------------------------------

print("\nASSOCIAÇÃO TEMPORAL DO PESO")
print("-" * 60)

print(
    "Controles totais:",
    len(controles_uenp_todos)
)

print(
    "Controles com peso anterior disponível:",
    len(peso_associado_uenp_todos)
)

print(
    "Cobertura geral:",
    round(
        len(peso_associado_uenp_todos)
        / len(controles_uenp_todos)
        * 100,
        2
    ),
    "%"
)


# ------------------------------------------------------------
# 6. COBERTURA POR ANO
# ------------------------------------------------------------

controles_com_peso = controles_uenp_todos.merge(
    peso_associado_uenp_todos[
        [
            "Número",
            "data_prova",
            "peso_kg",
            "dias_desde_pesagem"
        ]
    ],
    how="left",
    on=["Número", "data_prova"],
    validate="one_to_one"
)

print("\nCobertura por ano:")

cobertura_peso_ano_completa = (
    controles_com_peso
    .groupby("ano_origem")
    .agg(
        observacoes=("animal", "size"),
        com_peso=("peso_kg", lambda x: x.notna().sum())
    )
)

cobertura_peso_ano_completa["cobertura_%"] = (
    cobertura_peso_ano_completa["com_peso"]
    / cobertura_peso_ano_completa["observacoes"]
    * 100
).round(2)

display(cobertura_peso_ano_completa)


# ------------------------------------------------------------
# 7. COBERTURA ENTRE PRODUÇÕES MENSURADAS
# ------------------------------------------------------------

print("\nCobertura entre produções mensuradas:")

cobertura_peso_mensuradas = (
    controles_com_peso.loc[
        controles_com_peso[
            "situacao_producao"
        ] == "mensurada"
    ]
    .groupby("ano_origem")
    .agg(
        producoes_mensuradas=("animal", "size"),
        com_peso=("peso_kg", lambda x: x.notna().sum())
    )
)

cobertura_peso_mensuradas["cobertura_%"] = (
    cobertura_peso_mensuradas["com_peso"]
    / cobertura_peso_mensuradas[
        "producoes_mensuradas"
    ]
    * 100
).round(2)

display(cobertura_peso_mensuradas)


# ------------------------------------------------------------
# 8. DISTÂNCIA TEMPORAL
# ------------------------------------------------------------

print("\nIntervalo entre pesagem e controle:")

display(
    peso_associado_uenp_todos[
        "dias_desde_pesagem"
    ].describe()
)

print("\nIntervalos por ano:")

display(
    controles_com_peso.loc[
        controles_com_peso["peso_kg"].notna()
    ]
    .groupby("ano_origem")[
        "dias_desde_pesagem"
    ]
    .agg(
        ["count", "min", "median", "mean", "max"]
    )
)

print("\nDistribuição dos intervalos:")

display(
    peso_associado_uenp_todos[
        "dias_desde_pesagem"
    ]
    .value_counts()
    .sort_index()
)

### 12.28 Incorporação definitiva do peso corporal

A consolidação das diferentes fontes de pesagem permitiu associar peso corporal a todas as 295 observações de produção disponíveis entre 2024 e 2026.

A associação utilizou exclusivamente a pesagem mais recente realizada na mesma data ou anteriormente ao controle leiteiro. O intervalo entre pesagem e controle variou de 0 a 30 dias, com mediana geral de 2 dias.

Todas as 206 observações com produção de leite mensurada possuem peso corporal associado.

Dessa forma, o peso passa a apresentar cobertura completa na base da UENP e será mantido como variável candidata para as etapas posteriores de modelagem. O intervalo entre a pesagem e o controle também será preservado para rastreabilidade da associação temporal.

In [ ]:
# ============================================================
# 12.28 INCORPORAÇÃO DEFINITIVA DO PESO CORPORAL
# ============================================================

peso_definitivo_uenp = peso_associado_uenp_todos[
    [
        "Número",
        "data_prova",
        "data_pesagem",
        "peso_kg",
        "dias_desde_pesagem",
        "fonte_pesagem"
    ]
].copy()


# ------------------------------------------------------------
# 1. BASE DE 2024
# ------------------------------------------------------------

producao_uenp_2024_integrada = (
    producao_uenp_2024_integrada.merge(
        peso_definitivo_uenp,
        how="left",
        on=["Número", "data_prova"],
        validate="one_to_one"
    )
)


# ------------------------------------------------------------
# 2. BASE DE 2025/2026
# ------------------------------------------------------------
# Remove a associação provisória feita anteriormente somente
# com as pesagens de 2026, pois agora temos a versão definitiva.

colunas_peso_antigas = [
    "data_pesagem",
    "peso_kg",
    "dias_desde_pesagem"
]

base_uenp_preparada = base_uenp_preparada.drop(
    columns=[
        coluna
        for coluna in colunas_peso_antigas
        if coluna in base_uenp_preparada.columns
    ]
)

base_uenp_preparada = (
    base_uenp_preparada.merge(
        peso_definitivo_uenp,
        how="left",
        on=["Número", "data_prova"],
        validate="one_to_one"
    )
)


# ------------------------------------------------------------
# 3. VALIDAÇÃO
# ------------------------------------------------------------

print("PESO DEFINITIVO — UENP")
print("-" * 60)

print("\nBASE 2024")

print(
    "Observações:",
    len(producao_uenp_2024_integrada)
)

print(
    "Com peso:",
    producao_uenp_2024_integrada[
        "peso_kg"
    ].notna().sum()
)

print(
    "Sem peso:",
    producao_uenp_2024_integrada[
        "peso_kg"
    ].isna().sum()
)


print("\nBASE 2025/2026")

print(
    "Observações:",
    len(base_uenp_preparada)
)

print(
    "Com peso:",
    base_uenp_preparada[
        "peso_kg"
    ].notna().sum()
)

print(
    "Sem peso:",
    base_uenp_preparada[
        "peso_kg"
    ].isna().sum()
)


# ------------------------------------------------------------
# 4. PRODUÇÕES MENSURADAS DOS TRÊS ANOS
# ------------------------------------------------------------

validacao_peso_final = pd.concat(
    [
        producao_uenp_2024_integrada[
            [
                "Número",
                "animal",
                "data_prova",
                "kg_leite",
                "situacao_producao",
                "ano_origem",
                "peso_kg",
                "dias_desde_pesagem"
            ]
        ],
        base_uenp_preparada[
            [
                "Número",
                "animal",
                "data_prova",
                "kg_leite",
                "situacao_producao",
                "ano_origem",
                "peso_kg",
                "dias_desde_pesagem"
            ]
        ]
    ],
    ignore_index=True
)

mensuradas_peso_final = validacao_peso_final.loc[
    validacao_peso_final[
        "situacao_producao"
    ] == "mensurada"
]

print("\nPRODUÇÕES MENSURADAS")
print("-" * 60)

print(
    "Total:",
    len(mensuradas_peso_final)
)

print(
    "Com peso:",
    mensuradas_peso_final[
        "peso_kg"
    ].notna().sum()
)

print(
    "Sem peso:",
    mensuradas_peso_final[
        "peso_kg"
    ].isna().sum()
)

print(
    "Duplicidades Número + data:",
    validacao_peso_final.duplicated(
        subset=["Número", "data_prova"]
    ).sum()
)

print(
    "Maior intervalo desde a pesagem:",
    validacao_peso_final[
        "dias_desde_pesagem"
    ].max(),
    "dias"
)

### 12.29 Caracterização das informações reprodutivas

A planilha `Reprodução` contém informações relacionadas a serviços reprodutivos, inseminação artificial, diagnóstico de gestação, secagem, previsão de parto e situação reprodutiva dos animais.

Como esses registros podem representar eventos pontuais ou o estado reprodutivo mais recente disponível, não serão associados diretamente aos controles leiteiros históricos antes da avaliação de sua estrutura temporal.

Nesta etapa serão analisadas a cobertura das variáveis, os tipos de dados registrados, as datas disponíveis e a distribuição das situações reprodutivas, buscando identificar quais informações podem ser utilizadas de forma temporalmente consistente.

In [ ]:
# ============================================================
# 12.29 CARACTERIZAÇÃO DOS DADOS REPRODUTIVOS
# ============================================================

reproducao_uenp = dados_uenp["Reprodução"].copy()

print("REPRODUÇÃO — UENP")
print("-" * 60)

print(
    "Dimensões:",
    reproducao_uenp.shape
)

print("\nColunas:")

for i, coluna in enumerate(reproducao_uenp.columns):
    print(i, repr(coluna))


# ------------------------------------------------------------
# 1. COBERTURA DAS VARIÁVEIS
# ------------------------------------------------------------

print("\nCOBERTURA DAS VARIÁVEIS")
print("-" * 60)

cobertura_reproducao = pd.DataFrame({
    "preenchidos": reproducao_uenp.notna().sum(),
    "ausentes": reproducao_uenp.isna().sum()
})

cobertura_reproducao["cobertura_%"] = (
    cobertura_reproducao["preenchidos"]
    / len(reproducao_uenp)
    * 100
).round(2)

display(cobertura_reproducao)


# ------------------------------------------------------------
# 2. PRIMEIROS REGISTROS
# ------------------------------------------------------------

print("\nRegistros da planilha:")

display(reproducao_uenp)


# ------------------------------------------------------------
# 3. CONVERSÃO PRELIMINAR DAS COLUNAS TEMPORAIS
# ------------------------------------------------------------

colunas_temporais_repro = [
    "Serviço",
    "IA",
    "Início retorno",
    "Fim retorno",
    "DG",
    "Secagem",
    "Previsão Parto"
]

for coluna in colunas_temporais_repro:

    if coluna in reproducao_uenp.columns:

        reproducao_uenp[
            f"{coluna}_convertida"
        ] = pd.to_datetime(
            reproducao_uenp[coluna],
            errors="coerce"
        )


# ------------------------------------------------------------
# 4. INTERVALO DAS DATAS
# ------------------------------------------------------------

print("\nINTERVALOS TEMPORAIS")
print("-" * 60)

for coluna in colunas_temporais_repro:

    coluna_convertida = f"{coluna}_convertida"

    if coluna_convertida in reproducao_uenp.columns:

        serie = reproducao_uenp[
            coluna_convertida
        ]

        print(f"\n{coluna}:")

        print(
            "Datas válidas:",
            serie.notna().sum()
        )

        print(
            "Mínima:",
            serie.min()
        )

        print(
            "Máxima:",
            serie.max()
        )


# ------------------------------------------------------------
# 5. SITUAÇÃO REPRODUTIVA
# ------------------------------------------------------------

if "Situação" in reproducao_uenp.columns:

    print("\nSITUAÇÃO REPRODUTIVA")
    print("-" * 60)

    display(
        reproducao_uenp["Situação"]
        .astype("string")
        .str.strip()
        .value_counts(dropna=False)
    )


# ------------------------------------------------------------
# 6. DIAS EM GESTAÇÃO
# ------------------------------------------------------------

if "Dias em gestação" in reproducao_uenp.columns:

    dias_gestacao = pd.to_numeric(
        reproducao_uenp["Dias em gestação"],
        errors="coerce"
    )

    print("\nDIAS EM GESTAÇÃO")
    print("-" * 60)

    display(
        dias_gestacao.describe()
    )

    print(
        "Valores preenchidos mas não numéricos:",
        (
            reproducao_uenp["Dias em gestação"].notna()
            & dias_gestacao.isna()
        ).sum()
    )


# ------------------------------------------------------------
# 7. ANIMAIS PRESENTES
# ------------------------------------------------------------

print("\nANIMAIS PRESENTES NA PLANILHA")

display(
    reproducao_uenp[
        ["Animal"]
    ]
    .dropna()
    .drop_duplicates()
    .sort_values("Animal")
)

### 12.30 Validação semântica das informações reprodutivas

A inspeção inicial demonstrou que a planilha `Reprodução` combina diferentes tipos de informação.

A variável `Serviço` corresponde a uma quantidade numérica e não a uma data. O campo `DG` representa o resultado do diagnóstico de gestação, enquanto outras colunas, como `IA`, `Sexagem`, `Secagem` e `Previsão Parto`, apresentam informações temporais.

Para compreender se essas datas correspondem a eventos históricos independentes ou a informações derivadas de um acompanhamento reprodutivo atual, serão analisadas as relações temporais entre os campos.

Também será verificada a data de referência implícita na combinação entre a data de inseminação e os dias em gestação registrados.

In [ ]:
# ============================================================
# 12.30 VALIDAÇÃO SEMÂNTICA DAS INFORMAÇÕES REPRODUTIVAS
# ============================================================

reproducao_uenp_corrigida = dados_uenp["Reprodução"].copy()

# Remove espaços extras dos nomes das colunas
reproducao_uenp_corrigida.columns = (
    reproducao_uenp_corrigida.columns
    .astype(str)
    .str.strip()
)

print("VALIDAÇÃO SEMÂNTICA — REPRODUÇÃO UENP")
print("-" * 60)

print("\nColunas após padronização:")
print(reproducao_uenp_corrigida.columns.tolist())


# ------------------------------------------------------------
# 1. SERVIÇO É NUMÉRICO
# ------------------------------------------------------------

reproducao_uenp_corrigida["servico_num"] = pd.to_numeric(
    reproducao_uenp_corrigida["Serviço"],
    errors="coerce"
)

print("\nSERVIÇO")
print("-" * 30)

display(
    reproducao_uenp_corrigida["servico_num"]
    .describe()
)

print("\nDistribuição:")

display(
    reproducao_uenp_corrigida["servico_num"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 2. DATAS REPRODUTIVAS
# ------------------------------------------------------------

colunas_data_repro = [
    "IA",
    "Início retorno",
    "Fim retorno",
    "Sexagem",
    "Secagem",
    "Previsão Parto"
]

for coluna in colunas_data_repro:

    reproducao_uenp_corrigida[coluna] = pd.to_datetime(
        reproducao_uenp_corrigida[coluna],
        errors="coerce"
    )


# ------------------------------------------------------------
# 3. DIAS EM GESTAÇÃO
# ------------------------------------------------------------

reproducao_uenp_corrigida["dias_gestacao_num"] = pd.to_numeric(
    reproducao_uenp_corrigida["Dias em gestação"],
    errors="coerce"
)

# Data de referência implícita
reproducao_uenp_corrigida["data_referencia_gestacao"] = (
    reproducao_uenp_corrigida["IA"]
    + pd.to_timedelta(
        reproducao_uenp_corrigida["dias_gestacao_num"],
        unit="D"
    )
)

print("\nDATA DE REFERÊNCIA IMPLÍCITA — IA + DIAS EM GESTAÇÃO")
print("-" * 60)

display(
    reproducao_uenp_corrigida[
        "data_referencia_gestacao"
    ]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 4. RELAÇÕES ENTRE AS DATAS
# ------------------------------------------------------------

reproducao_com_ia = reproducao_uenp_corrigida.loc[
    reproducao_uenp_corrigida["IA"].notna()
].copy()

reproducao_com_ia["dias_ia_inicio_retorno"] = (
    reproducao_com_ia["Início retorno"]
    - reproducao_com_ia["IA"]
).dt.days

reproducao_com_ia["dias_ia_fim_retorno"] = (
    reproducao_com_ia["Fim retorno"]
    - reproducao_com_ia["IA"]
).dt.days

reproducao_com_ia["dias_ia_sexagem"] = (
    reproducao_com_ia["Sexagem"]
    - reproducao_com_ia["IA"]
).dt.days

reproducao_com_ia["dias_ia_parto_previsto"] = (
    reproducao_com_ia["Previsão Parto"]
    - reproducao_com_ia["IA"]
).dt.days

reproducao_com_ia["dias_secagem_ate_parto"] = (
    reproducao_com_ia["Previsão Parto"]
    - reproducao_com_ia["Secagem"]
).dt.days


print("\nRELAÇÕES TEMPORAIS ENTRE OS CAMPOS")
print("-" * 60)

display(
    reproducao_com_ia[
        [
            "Animal",
            "IA",
            "DG",
            "Sexagem",
            "dias_gestacao_num",
            "data_referencia_gestacao",
            "dias_ia_inicio_retorno",
            "dias_ia_fim_retorno",
            "dias_ia_sexagem",
            "dias_ia_parto_previsto",
            "dias_secagem_ate_parto",
            "Situação"
        ]
    ]
)


# ------------------------------------------------------------
# 5. SITUAÇÃO ATUAL
# ------------------------------------------------------------

print("\nSITUAÇÃO REPRODUTIVA ATUAL")
print("-" * 60)

display(
    reproducao_uenp_corrigida["Situação"]
    .astype("string")
    .str.strip()
    .value_counts(dropna=False)
)

### 12.31 Decisão sobre a utilização das informações reprodutivas

A análise da planilha `Reprodução` demonstrou que as informações disponíveis representam predominantemente o acompanhamento reprodutivo atual dos animais.

Para os nove animais com inseminação registrada, a soma entre a data de IA e os dias em gestação resultou, em todos os casos, na data de 23 de setembro de 2026, evidenciando uma data de referência comum para atualização dessas informações.

Também foram identificadas relações temporais padronizadas entre a inseminação e outras variáveis da planilha, como início e fim do período de retorno, sexagem, previsão de parto e secagem. Parte desses campos representa, portanto, datas calculadas ou programadas a partir da inseminação.

A variável de situação reprodutiva também corresponde ao estado atual dos animais e não apresenta histórico suficiente para representar sua condição em cada controle leiteiro de 2024, 2025 e 2026.

Dessa forma, as informações da planilha `Reprodução` serão preservadas como fonte auxiliar, mas não serão utilizadas diretamente como atributos preditores na base analítica principal, evitando a atribuição retrospectiva de informações e a introdução de vazamento temporal.

### 12.32 Caracterização dos registros sanitários

A planilha `Doenças` contém registros individuais de ocorrências sanitárias, incluindo identificação do animal, data, diagnóstico e tratamento.

Como essas informações possuem referência temporal explícita, será avaliada a possibilidade de relacionar eventos sanitários aos controles leiteiros sem utilizar dados posteriores à data da produção.

Inicialmente serão analisados o número de eventos registrados, os animais envolvidos, o período abrangido, os diagnósticos encontrados e a correspondência dos animais com a base de produção.

Nenhuma variável sanitária será incorporada ao modelo antes da avaliação de sua cobertura temporal.

In [ ]:
# ============================================================
# 12.32 CARACTERIZAÇÃO DOS REGISTROS SANITÁRIOS
# ============================================================

doencas_uenp = dados_uenp["Doenças"].copy()

print("DOENÇAS — UENP")
print("-" * 60)

print(
    "Dimensões:",
    doencas_uenp.shape
)

print("\nColunas:")

for i, coluna in enumerate(doencas_uenp.columns):
    print(i, repr(coluna))


# ------------------------------------------------------------
# 1. PADRONIZAÇÃO
# ------------------------------------------------------------

doencas_uenp.columns = (
    doencas_uenp.columns
    .astype(str)
    .str.strip()
)

doencas_uenp["animal_normalizado"] = (
    doencas_uenp["ANIMAL"]
    .apply(normalizar_nome_animal)
)

doencas_uenp["data_diagnostico"] = pd.to_datetime(
    doencas_uenp["DATA"],
    errors="coerce"
)


# ------------------------------------------------------------
# 2. COBERTURA
# ------------------------------------------------------------

print("\nCOBERTURA")
print("-" * 60)

display(
    pd.DataFrame({
        "preenchidos": doencas_uenp.notna().sum(),
        "ausentes": doencas_uenp.isna().sum()
    })
)


# ------------------------------------------------------------
# 3. PERÍODO DOS EVENTOS
# ------------------------------------------------------------

print("\nPERÍODO DOS REGISTROS")
print("-" * 60)

print(
    "Datas válidas:",
    doencas_uenp["data_diagnostico"].notna().sum()
)

print(
    "Data mínima:",
    doencas_uenp["data_diagnostico"].min()
)

print(
    "Data máxima:",
    doencas_uenp["data_diagnostico"].max()
)


# ------------------------------------------------------------
# 4. ANIMAIS E DIAGNÓSTICOS
# ------------------------------------------------------------

print("\nAnimais distintos com registro sanitário:")

print(
    doencas_uenp["animal_normalizado"].nunique()
)

print("\nDiagnósticos:")

display(
    doencas_uenp["DIAGNÓSTICO"]
    .astype("string")
    .str.strip()
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 5. REGISTROS COMPLETOS
# ------------------------------------------------------------

print("\nRegistros sanitários:")

display(
    doencas_uenp[
        [
            "ANIMAL",
            "data_diagnostico",
            "DIAGNÓSTICO",
            "TRATAMENTO"
        ]
    ]
    .sort_values(
        ["data_diagnostico", "ANIMAL"]
    )
)


# ------------------------------------------------------------
# 6. CORRESPONDÊNCIA COM OS ANIMAIS DA PRODUÇÃO
# ------------------------------------------------------------

animais_producao_uenp = set(
    pd.concat(
        [
            producao_uenp_2024_integrada[
                "animal_normalizado"
            ],
            base_uenp_preparada[
                "animal_normalizado"
            ]
        ],
        ignore_index=True
    ).dropna()
)

doencas_uenp["animal_na_base_producao"] = (
    doencas_uenp["animal_normalizado"]
    .isin(animais_producao_uenp)
)

print("\nCORRESPONDÊNCIA COM A PRODUÇÃO")
print("-" * 60)

print(
    "Eventos sanitários totais:",
    len(doencas_uenp)
)

print(
    "Eventos de animais presentes na produção:",
    doencas_uenp[
        "animal_na_base_producao"
    ].sum()
)

print(
    "Eventos sem correspondência com a produção:",
    (
        ~doencas_uenp[
            "animal_na_base_producao"
        ]
    ).sum()
)

print("\nEventos sem correspondência:")

display(
    doencas_uenp.loc[
        ~doencas_uenp["animal_na_base_producao"],
        [
            "ANIMAL",
            "data_diagnostico",
            "DIAGNÓSTICO"
        ]
    ]
)

### 12.33 Relação temporal entre os registros sanitários e os controles leiteiros

A planilha sanitária contém 11 eventos referentes a 10 animais, porém apenas três desses eventos correspondem a animais presentes na base de produção utilizada no estudo.

Também foi identificada uma inconsistência de origem na data de um registro referente ao animal Ayla, armazenada como `0205-09-23`. Como esse animal não apresenta controles leiteiros na base analisada, o registro será preservado como inconsistência da fonte sem correção inferida.

Para avaliar a utilidade das informações sanitárias como possíveis atributos, será analisada a distância temporal entre os diagnósticos dos animais presentes na base e seus respectivos controles leiteiros.

Será considerada separadamente a ocorrência de diagnósticos anteriores aos controles, evitando que eventos sanitários futuros sejam utilizados como informação preditora.

In [ ]:
# ============================================================
# 12.33 RELAÇÃO TEMPORAL ENTRE DOENÇAS E CONTROLES LEITEIROS
# ============================================================

# ------------------------------------------------------------
# 1. SOMENTE EVENTOS DE ANIMAIS PRESENTES NA PRODUÇÃO
# ------------------------------------------------------------

doencas_relevantes_uenp = doencas_uenp.loc[
    doencas_uenp["animal_na_base_producao"]
].copy()

print("EVENTOS SANITÁRIOS RELEVANTES — UENP")
print("-" * 60)

print(
    "Eventos:",
    len(doencas_relevantes_uenp)
)

print(
    "Animais distintos:",
    doencas_relevantes_uenp[
        "animal_normalizado"
    ].nunique()
)

display(
    doencas_relevantes_uenp[
        [
            "ANIMAL",
            "data_diagnostico",
            "DIAGNÓSTICO",
            "TRATAMENTO"
        ]
    ].sort_values("data_diagnostico")
)


# ------------------------------------------------------------
# 2. CRUZAMENTO COM OS CONTROLES DOS MESMOS ANIMAIS
# ------------------------------------------------------------

controles_sanitarios = controles_uenp_todos.copy()

controles_sanitarios["animal_normalizado"] = (
    controles_sanitarios["animal"]
    .apply(normalizar_nome_animal)
)

relacao_doenca_controle = doencas_relevantes_uenp[
    [
        "animal_normalizado",
        "ANIMAL",
        "data_diagnostico",
        "DIAGNÓSTICO"
    ]
].merge(
    controles_sanitarios[
        [
            "animal_normalizado",
            "data_prova",
            "kg_leite",
            "situacao_producao",
            "ano_origem"
        ]
    ],
    how="left",
    on="animal_normalizado"
)

# Positivo = controle ocorreu depois do diagnóstico
# Negativo = controle ocorreu antes do diagnóstico
relacao_doenca_controle["dias_controle_apos_diagnostico"] = (
    relacao_doenca_controle["data_prova"]
    - relacao_doenca_controle["data_diagnostico"]
).dt.days


# ------------------------------------------------------------
# 3. CONTROLES PRÓXIMOS AO EVENTO
# ------------------------------------------------------------

print("\nCONTROLES ATÉ 30 DIAS ANTES OU DEPOIS DO DIAGNÓSTICO")
print("-" * 60)

controles_proximos_doenca = relacao_doenca_controle.loc[
    relacao_doenca_controle[
        "dias_controle_apos_diagnostico"
    ].between(-30, 30)
].copy()

display(
    controles_proximos_doenca[
        [
            "ANIMAL",
            "DIAGNÓSTICO",
            "data_diagnostico",
            "data_prova",
            "dias_controle_apos_diagnostico",
            "kg_leite",
            "situacao_producao"
        ]
    ].sort_values(
        ["ANIMAL", "data_prova"]
    )
)


# ------------------------------------------------------------
# 4. EVENTO CONHECIDO ANTES DO CONTROLE
# ------------------------------------------------------------

controles_pos_diagnostico_30d = relacao_doenca_controle.loc[
    relacao_doenca_controle[
        "dias_controle_apos_diagnostico"
    ].between(0, 30)
].copy()

print("\nCONTROLES REALIZADOS ATÉ 30 DIAS APÓS UM DIAGNÓSTICO")
print("-" * 60)

print(
    "Quantidade:",
    len(controles_pos_diagnostico_30d)
)

print(
    "Com produção mensurada:",
    (
        controles_pos_diagnostico_30d[
            "situacao_producao"
        ] == "mensurada"
    ).sum()
)

display(
    controles_pos_diagnostico_30d[
        [
            "ANIMAL",
            "DIAGNÓSTICO",
            "data_diagnostico",
            "data_prova",
            "dias_controle_apos_diagnostico",
            "kg_leite",
            "situacao_producao"
        ]
    ].sort_values(
        ["ANIMAL", "data_prova"]
    )
)


# ------------------------------------------------------------
# 5. CONTROLE MAIS PRÓXIMO DE CADA DIAGNÓSTICO
# ------------------------------------------------------------

relacao_doenca_controle["distancia_absoluta"] = (
    relacao_doenca_controle[
        "dias_controle_apos_diagnostico"
    ].abs()
)

controle_mais_proximo = (
    relacao_doenca_controle
    .sort_values(
        [
            "animal_normalizado",
            "data_diagnostico",
            "distancia_absoluta"
        ]
    )
    .drop_duplicates(
        subset=[
            "animal_normalizado",
            "data_diagnostico",
            "DIAGNÓSTICO"
        ],
        keep="first"
    )
)

print("\nCONTROLE MAIS PRÓXIMO DE CADA EVENTO")
print("-" * 60)

display(
    controle_mais_proximo[
        [
            "ANIMAL",
            "DIAGNÓSTICO",
            "data_diagnostico",
            "data_prova",
            "dias_controle_apos_diagnostico",
            "kg_leite",
            "situacao_producao"
        ]
    ]
)

### 12.34 Decisão sobre a utilização dos registros sanitários

A análise dos registros sanitários demonstrou baixa correspondência temporal com os controles leiteiros utilizados no estudo.

Dos 11 eventos registrados, apenas três pertencem a animais presentes na base de produção. Entre esses casos, não foi identificado nenhum controle leiteiro realizado nos 30 dias posteriores ao diagnóstico.

O único controle temporalmente próximo a um evento sanitário ocorreu para o animal Toddy, três dias antes do diagnóstico de piometra. Como o diagnóstico ainda não estava disponível na data da mensuração da produção, sua utilização como atributo representaria informação futura.

Os controles mais próximos dos eventos registrados para Mery e Nala ocorreram aproximadamente 300 dias antes dos respectivos diagnósticos.

Dessa forma, os registros sanitários serão preservados como informação auxiliar, mas não serão utilizados como atributos preditores na base analítica principal, devido à baixa cobertura temporal e ao risco de utilização retrospectiva de informações ainda não conhecidas no momento dos controles.

### 12.35 Construção da base longitudinal unificada da UENP

Após a preparação individual dos registros de 2024, 2025 e 2026, os controles leiteiros serão reunidos em uma única base longitudinal.

A base manterá as informações de rastreabilidade, produção, características individuais, peso corporal e DEL quando este puder ser calculado de maneira temporalmente válida.

Também serão derivadas informações referentes ao controle imediatamente anterior de cada animal. A produção anterior constitui uma informação potencialmente relevante para a estimativa da produção atual, desde que corresponda exclusivamente a uma mensuração realizada antes da observação que se pretende estimar.

Para preservar essa relação temporal, os registros serão ordenados individualmente por animal e data antes da criação das variáveis referentes ao controle anterior.

In [ ]:
# ============================================================
# 12.35 BASE LONGITUDINAL UNIFICADA — UENP
# ============================================================

colunas_base_uenp = [
    "Número",
    "animal",
    "animal_normalizado",
    "data_prova",
    "ano_origem",
    "kg_leite",
    "situacao_producao",
    "raca_padronizada",
    "data_nascimento_convertida",
    "idade_anos",
    "peso_kg",
    "data_pesagem",
    "dias_desde_pesagem",
    "fonte_pesagem",
    "del_derivado",
    "parto_convertido",
    "fonte_identificador"
]


# ------------------------------------------------------------
# 1. PADRONIZAÇÃO DAS DUAS BASES
# ------------------------------------------------------------

base_uenp_2024_final = (
    producao_uenp_2024_integrada
    .reindex(columns=colunas_base_uenp)
    .copy()
)

base_uenp_2025_2026_final = (
    base_uenp_preparada
    .reindex(columns=colunas_base_uenp)
    .copy()
)


# ------------------------------------------------------------
# 2. UNIÃO DOS TRÊS ANOS
# ------------------------------------------------------------

base_uenp_unificada = pd.concat(
    [
        base_uenp_2024_final,
        base_uenp_2025_2026_final
    ],
    ignore_index=True
)

base_uenp_unificada["propriedade"] = "UENP"

base_uenp_unificada = (
    base_uenp_unificada
    .sort_values(
        ["Número", "data_prova"]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 3. VARIÁVEL TEMPORAL: MÊS DO CONTROLE
# ------------------------------------------------------------

base_uenp_unificada["mes_controle"] = (
    base_uenp_unificada[
        "data_prova"
    ].dt.month
)


# ------------------------------------------------------------
# 4. CONTROLE ANTERIOR DO MESMO ANIMAL
# ------------------------------------------------------------

base_uenp_unificada["data_controle_anterior"] = (
    base_uenp_unificada
    .groupby("Número")["data_prova"]
    .shift(1)
)

base_uenp_unificada["kg_leite_anterior"] = (
    base_uenp_unificada
    .groupby("Número")["kg_leite"]
    .shift(1)
)

base_uenp_unificada["situacao_anterior"] = (
    base_uenp_unificada
    .groupby("Número")["situacao_producao"]
    .shift(1)
)

base_uenp_unificada["dias_desde_controle_anterior"] = (
    base_uenp_unificada["data_prova"]
    - base_uenp_unificada["data_controle_anterior"]
).dt.days


# ------------------------------------------------------------
# 5. VALIDAÇÃO GERAL
# ------------------------------------------------------------

print("BASE LONGITUDINAL UNIFICADA — UENP")
print("-" * 60)

print(
    "Observações:",
    len(base_uenp_unificada)
)

print(
    "Animais distintos:",
    base_uenp_unificada["Número"].nunique()
)

print(
    "Produções mensuradas:",
    (
        base_uenp_unificada[
            "situacao_producao"
        ] == "mensurada"
    ).sum()
)

print(
    "Duplicidades Número + data:",
    base_uenp_unificada.duplicated(
        subset=["Número", "data_prova"]
    ).sum()
)


# ------------------------------------------------------------
# 6. COBERTURA DAS PRINCIPAIS VARIÁVEIS
# ------------------------------------------------------------

print("\nCOBERTURA DAS PRINCIPAIS VARIÁVEIS")
print("-" * 60)

variaveis_verificar = [
    "kg_leite",
    "peso_kg",
    "raca_padronizada",
    "idade_anos",
    "del_derivado",
    "kg_leite_anterior",
    "dias_desde_controle_anterior",
    "mes_controle"
]

for coluna in variaveis_verificar:

    preenchidos = (
        base_uenp_unificada[
            coluna
        ].notna().sum()
    )

    percentual = (
        preenchidos
        / len(base_uenp_unificada)
        * 100
    )

    print(
        f"{coluna}: "
        f"{preenchidos}/{len(base_uenp_unificada)} "
        f"({percentual:.2f}%)"
    )


# ------------------------------------------------------------
# 7. SOMENTE PRODUÇÕES MENSURADAS
# ------------------------------------------------------------

base_uenp_mensurada = base_uenp_unificada.loc[
    base_uenp_unificada[
        "situacao_producao"
    ] == "mensurada"
].copy()

print("\nENTRE AS PRODUÇÕES MENSURADAS")
print("-" * 60)

print(
    "Observações:",
    len(base_uenp_mensurada)
)

for coluna in [
    "peso_kg",
    "raca_padronizada",
    "idade_anos",
    "del_derivado",
    "kg_leite_anterior",
    "dias_desde_controle_anterior"
]:

    preenchidos = (
        base_uenp_mensurada[
            coluna
        ].notna().sum()
    )

    percentual = (
        preenchidos
        / len(base_uenp_mensurada)
        * 100
    )

    print(
        f"{coluna}: "
        f"{preenchidos}/{len(base_uenp_mensurada)} "
        f"({percentual:.2f}%)"
    )


# ------------------------------------------------------------
# 8. SITUAÇÃO DO CONTROLE ANTERIOR
# ------------------------------------------------------------

print("\nSituação do controle anterior entre produções mensuradas:")

display(
    base_uenp_mensurada[
        "situacao_anterior"
    ].value_counts(dropna=False)
)


# ------------------------------------------------------------
# 9. INTERVALO ENTRE CONTROLES
# ------------------------------------------------------------

print("\nIntervalo entre controles consecutivos:")

display(
    base_uenp_mensurada[
        "dias_desde_controle_anterior"
    ].describe()
)

### 12.36 Inventário e decisão das variáveis da UENP

Após a preparação e integração das diferentes fontes, foi construída uma base longitudinal contendo 295 controles referentes a 26 animais, dos quais 206 apresentam produção de leite mensurada.

As variáveis disponíveis foram avaliadas considerando cobertura, significado, disponibilidade temporal e risco de vazamento de informação.

O peso corporal apresenta cobertura completa e foi associado exclusivamente a pesagens realizadas na mesma data ou anteriormente ao controle leiteiro.

Raça e idade apresentam cobertura parcial devido à ausência de cadastro atual para alguns animais históricos, mas permanecem metodologicamente válidas.

O DEL foi derivado apenas quando a data de parto disponível correspondia à lactação do respectivo controle, resultando em cobertura parcial.

A produção anterior foi calculada exclusivamente a partir do controle cronologicamente anterior do mesmo animal, preservando a ordem temporal.

Informações reprodutivas atuais e registros sanitários foram mantidos como fontes auxiliares, mas não serão utilizados diretamente como atributos preditores devido à ausência de histórico temporal suficiente.

A definição definitiva das variáveis utilizadas nos modelos será realizada posteriormente, durante a harmonização entre as bases da UENP e da propriedade de Cerqueira César.

In [ ]:
# ============================================================
# 12.36 INVENTÁRIO E DECISÃO DAS VARIÁVEIS — UENP
# ============================================================

decisao_variaveis_uenp = pd.DataFrame([
    {
        "variavel": "kg_leite",
        "papel": "alvo",
        "decisao": "utilizar",
        "justificativa": (
            "Produção de leite mensurada. Variável resposta "
            "do problema de regressão."
        )
    },
    {
        "variavel": "peso_kg",
        "papel": "preditora candidata",
        "decisao": "avaliar no ML",
        "justificativa": (
            "Cobertura completa. Peso obtido na mesma data ou "
            "antes do controle, sem uso de informação futura."
        )
    },
    {
        "variavel": "raca_padronizada",
        "papel": "preditora candidata",
        "decisao": "avaliar no ML",
        "justificativa": (
            "Característica individual estável. Ausente apenas "
            "para parte dos animais históricos."
        )
    },
    {
        "variavel": "idade_anos",
        "papel": "preditora candidata",
        "decisao": "avaliar no ML",
        "justificativa": (
            "Calculada na data de cada controle a partir da "
            "data de nascimento."
        )
    },
    {
        "variavel": "del_derivado",
        "papel": "preditora candidata",
        "decisao": "avaliar no ML",
        "justificativa": (
            "Calculado somente quando a data de parto disponível "
            "é temporalmente compatível com o controle. "
            "Apresenta cobertura parcial."
        )
    },
    {
        "variavel": "kg_leite_anterior",
        "papel": "preditora candidata",
        "decisao": "avaliar no ML",
        "justificativa": (
            "Produção mensurada no controle imediatamente anterior "
            "do mesmo animal, utilizando apenas informação passada."
        )
    },
    {
        "variavel": "dias_desde_controle_anterior",
        "papel": "preditora candidata",
        "decisao": "avaliar no ML",
        "justificativa": (
            "Representa o intervalo temporal desde o controle "
            "anterior e está disponível no momento da previsão."
        )
    },
    {
        "variavel": "mes_controle",
        "papel": "preditora candidata",
        "decisao": "avaliar no ML",
        "justificativa": (
            "Representa componente temporal/sazonal sem utilizar "
            "informação futura."
        )
    },
    {
        "variavel": "dias_desde_pesagem",
        "papel": "controle de qualidade",
        "decisao": "não priorizar como preditora",
        "justificativa": (
            "Utilizada principalmente para documentar a defasagem "
            "temporal entre pesagem e controle."
        )
    },
    {
        "variavel": "data_prova",
        "papel": "rastreabilidade/temporal",
        "decisao": "não utilizar diretamente",
        "justificativa": (
            "Mantida para ordenação temporal, validação e "
            "separação dos dados."
        )
    },
    {
        "variavel": "data_pesagem",
        "papel": "rastreabilidade",
        "decisao": "não utilizar diretamente",
        "justificativa": (
            "Utilizada para validar a associação temporal do peso."
        )
    },
    {
        "variavel": "parto_convertido",
        "papel": "auxiliar",
        "decisao": "não utilizar diretamente",
        "justificativa": (
            "Data de parto atual utilizada somente para derivar "
            "DEL quando temporalmente compatível."
        )
    },
    {
        "variavel": "situacao_producao",
        "papel": "controle de qualidade",
        "decisao": "não utilizar diretamente",
        "justificativa": (
            "Distingue produção mensurada, seca, sem registro e "
            "ausência. O modelo de regressão utilizará apenas "
            "observações com alvo mensurado."
        )
    },
    {
        "variavel": "situacao_anterior",
        "papel": "auxiliar",
        "decisao": "preservar",
        "justificativa": (
            "Mantida para interpretar a ausência da produção "
            "anterior e caracterizar o histórico do controle."
        )
    },
    {
        "variavel": "Número",
        "papel": "identificador",
        "decisao": "não utilizar como preditora",
        "justificativa": (
            "Identificador individual utilizado para integração "
            "e controle da dependência entre observações."
        )
    },
    {
        "variavel": "animal",
        "papel": "identificador",
        "decisao": "não utilizar como preditora",
        "justificativa": (
            "Nome do animal mantido apenas para rastreabilidade."
        )
    },
    {
        "variavel": "reproducao",
        "papel": "fonte auxiliar",
        "decisao": "não utilizar",
        "justificativa": (
            "Representa predominantemente situação atual de 2026 "
            "e não possui histórico suficiente por controle."
        )
    },
    {
        "variavel": "sanidade",
        "papel": "fonte auxiliar",
        "decisao": "não utilizar",
        "justificativa": (
            "Baixa cobertura temporal e ausência de controles "
            "posteriores próximos aos diagnósticos registrados."
        )
    }
])

print("DECISÃO DAS VARIÁVEIS — UENP")
print("-" * 60)

display(decisao_variaveis_uenp)

In [ ]:
# ============================================================
# BASE CANDIDATA PARA MODELAGEM — UENP
# ============================================================

base_uenp_ml = base_uenp_unificada[
    [
        "propriedade",
        "Número",
        "animal",
        "data_prova",
        "ano_origem",

        # alvo
        "kg_leite",

        # candidatas
        "peso_kg",
        "raca_padronizada",
        "idade_anos",
        "del_derivado",
        "kg_leite_anterior",
        "dias_desde_controle_anterior",
        "mes_controle",

        # rastreabilidade / qualidade
        "situacao_producao",
        "situacao_anterior",
        "data_pesagem",
        "dias_desde_pesagem",
        "fonte_identificador"
    ]
].copy()


print("BASE CANDIDATA PARA MACHINE LEARNING — UENP")
print("-" * 60)

print(
    "Observações:",
    len(base_uenp_ml)
)

print(
    "Colunas:",
    base_uenp_ml.shape[1]
)

print(
    "Animais:",
    base_uenp_ml["Número"].nunique()
)

print(
    "Produções mensuradas:",
    base_uenp_ml["kg_leite"].notna().sum()
)

print(
    "Duplicidades Número + data:",
    base_uenp_ml.duplicated(
        subset=["Número", "data_prova"]
    ).sum()
)

print("\nCobertura entre as 206 produções mensuradas:")

mensuradas_ml_uenp = base_uenp_ml.loc[
    base_uenp_ml["kg_leite"].notna()
]

for coluna in [
    "peso_kg",
    "raca_padronizada",
    "idade_anos",
    "del_derivado",
    "kg_leite_anterior",
    "dias_desde_controle_anterior",
    "mes_controle"
]:

    n = mensuradas_ml_uenp[coluna].notna().sum()

    print(
        f"{coluna}: "
        f"{n}/{len(mensuradas_ml_uenp)} "
        f"({n / len(mensuradas_ml_uenp) * 100:.2f}%)"
    )

### 12.37 Validação final e exportação da base processada

Ao final da preparação, a base da UENP contém 295 controles leiteiros referentes a 26 animais, abrangendo os anos de 2024, 2025 e 2026.

Dessas observações, 206 possuem produção de leite mensurada e constituem os registros potencialmente utilizáveis como variável-alvo nos experimentos de regressão.

Antes da exportação, serão realizados testes finais de integridade para verificar duplicidades, disponibilidade dos identificadores, coerência da variável-alvo, validade temporal das pesagens, valores negativos indevidos e consistência das variáveis derivadas.

A base processada e a tabela contendo as decisões metodológicas referentes às variáveis serão exportadas para o diretório de dados processados. A seleção definitiva dos atributos utilizados em cada experimento será realizada posteriormente, durante a harmonização com os dados da propriedade de Cerqueira César.

In [ ]:
# ============================================================
# 12.37 VALIDAÇÃO FINAL E EXPORTAÇÃO — UENP
# ============================================================

print("VALIDAÇÃO FINAL — UENP")
print("-" * 60)

# ------------------------------------------------------------
# 1. TESTES DE ESTRUTURA
# ------------------------------------------------------------

teste_observacoes = len(base_uenp_ml) == 295

teste_colunas = base_uenp_ml.shape[1] == 18

teste_animais = (
    base_uenp_ml["Número"].nunique() == 26
)

teste_identificadores = (
    base_uenp_ml["Número"].notna().all()
)

teste_duplicidades = (
    base_uenp_ml.duplicated(
        subset=["Número", "data_prova"]
    ).sum() == 0
)


# ------------------------------------------------------------
# 2. VARIÁVEL-ALVO
# ------------------------------------------------------------

teste_alvo = (
    base_uenp_ml["kg_leite"].notna().sum() == 206
)

teste_mensurada_sem_valor = (
    (
        (base_uenp_ml["situacao_producao"] == "mensurada")
        & base_uenp_ml["kg_leite"].isna()
    ).sum() == 0
)

teste_valor_em_nao_mensurada = (
    (
        (base_uenp_ml["situacao_producao"] != "mensurada")
        & base_uenp_ml["kg_leite"].notna()
    ).sum() == 0
)

teste_producao_negativa = (
    (base_uenp_ml["kg_leite"] < 0).sum() == 0
)


# ------------------------------------------------------------
# 3. PESO CORPORAL
# ------------------------------------------------------------

teste_peso_completo = (
    base_uenp_ml["peso_kg"].notna().all()
)

teste_peso_negativo = (
    (base_uenp_ml["peso_kg"] <= 0).sum() == 0
)

teste_pesagem_futura = (
    (
        base_uenp_ml["data_pesagem"]
        > base_uenp_ml["data_prova"]
    ).sum() == 0
)

teste_intervalo_peso = (
    (base_uenp_ml["dias_desde_pesagem"] < 0).sum() == 0
)


# ------------------------------------------------------------
# 4. VARIÁVEIS TEMPORAIS E DERIVADAS
# ------------------------------------------------------------

teste_idade_negativa = (
    (base_uenp_ml["idade_anos"] < 0).sum() == 0
)

teste_del_negativo = (
    (base_uenp_ml["del_derivado"] < 0).sum() == 0
)

teste_intervalo_controle_negativo = (
    (
        base_uenp_ml[
            "dias_desde_controle_anterior"
        ] <= 0
    ).sum() == 0
)

teste_mes = (
    base_uenp_ml["mes_controle"]
    .dropna()
    .between(1, 12)
    .all()
)


# ------------------------------------------------------------
# 5. PROPRIEDADE E ANOS
# ------------------------------------------------------------

teste_propriedade = (
    base_uenp_ml["propriedade"]
    .eq("UENP")
    .all()
)

teste_anos = (
    set(
        base_uenp_ml["ano_origem"]
        .dropna()
        .unique()
    )
    == {2024, 2025, 2026}
)


# ------------------------------------------------------------
# 6. RESULTADOS DOS TESTES
# ------------------------------------------------------------

testes_uenp = {
    "295 observações": teste_observacoes,
    "18 colunas": teste_colunas,
    "26 animais": teste_animais,
    "identificadores completos": teste_identificadores,
    "sem duplicidades animal + data": teste_duplicidades,
    "206 produções mensuradas": teste_alvo,
    "mensurada sempre possui kg_leite": teste_mensurada_sem_valor,
    "não mensurada não possui kg_leite": teste_valor_em_nao_mensurada,
    "sem produção negativa": teste_producao_negativa,
    "peso com cobertura completa": teste_peso_completo,
    "sem peso inválido": teste_peso_negativo,
    "nenhuma pesagem futura": teste_pesagem_futura,
    "intervalo da pesagem não negativo": teste_intervalo_peso,
    "sem idade negativa": teste_idade_negativa,
    "sem DEL negativo": teste_del_negativo,
    "intervalo entre controles positivo": teste_intervalo_controle_negativo,
    "mês entre 1 e 12": teste_mes,
    "propriedade identificada como UENP": teste_propriedade,
    "anos 2024, 2025 e 2026": teste_anos
}

for nome, resultado in testes_uenp.items():
    print(
        f"{nome}:",
        resultado
    )


base_uenp_validada = all(
    testes_uenp.values()
)

print("\n" + "=" * 60)

print(
    "Base da UENP validada:",
    base_uenp_validada
)


# ------------------------------------------------------------
# 7. SOMENTE EXPORTA SE TODOS OS TESTES FOREM APROVADOS
# ------------------------------------------------------------

if not base_uenp_validada:
    raise ValueError(
        "A base da UENP não passou em todos os testes. "
        "Os arquivos não serão exportados."
    )


# ------------------------------------------------------------
# 8. DIRETÓRIO DE SAÍDA
# ------------------------------------------------------------

from pathlib import Path

caminhos_processados = [
    Path("../dados/processados"),
    Path("dados/processados")
]

pasta_processados = next(
    (
        caminho
        for caminho in caminhos_processados
        if caminho.exists()
    ),
    caminhos_processados[0]
)

pasta_processados.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 9. EXPORTAÇÃO
# ------------------------------------------------------------

arquivo_base_uenp = (
    pasta_processados
    / "uenp_base_processada.csv"
)

arquivo_decisoes_uenp = (
    pasta_processados
    / "uenp_decisao_variaveis.csv"
)

base_uenp_ml.to_csv(
    arquivo_base_uenp,
    index=False,
    encoding="utf-8-sig"
)

decisao_variaveis_uenp.to_csv(
    arquivo_decisoes_uenp,
    index=False,
    encoding="utf-8-sig"
)


print("\nARQUIVOS EXPORTADOS")
print("-" * 60)

print(
    "Base processada:",
    arquivo_base_uenp
)

print(
    "Decisões das variáveis:",
    arquivo_decisoes_uenp
)

print("\nPreparação da UENP concluída.")